In [1]:
# Set DATASET_1 to your private dataset file path (.csv, .json, or .jsonl).
# Local file example (within same folder): DATASET_1 = "private_test.csv" (also supports .json and .jsonl).
# Hugging Face example: DATASET_1 = "jise/ruletaker".
# For one dataset, set DATASET_2 through DATASET_5 to None (IMPORTANT), then run all cells to generate submission.csv.
# Fill slots consecutively (loading stops at the first None), then run all cells to generate submission.csv and accuracy prints only when reference labels (LABEL_COLUMN) are avaible or keep to 'auto'.

DATASET_1 = "jise/ruletaker"
DATASET_2 = "CounterBench/CounterBench"
DATASET_3 = "yale-nlp/FOLIO"
DATASET_4 = None
DATASET_5 = None
DATASETS = [DATASET_1, DATASET_2, DATASET_3, DATASET_4, DATASET_5]
OUTPUT_CSV = 'submission.csv'
LABEL_COLUMN = 'auto'
COLUMN_MAP = None
HF_SPLIT = None
HF_CONFIG = None
HF_REVISION = None
INSTALL_HF_PACKAGES = False
HF_LOGIN = False
TIMEOUT_SECONDS = 2.0
PROGRESS_EVERY = 25


In [2]:
import csv
import json
import itertools
import math
import random
import re
import time
from collections import Counter
from dataclasses import dataclass
from pathlib import Path
from time import perf_counter
from dataclasses import dataclass
import itertools
import math
import re
import time
import itertools
import math
import re
import time
from fractions import Fraction
import re

@dataclass(frozen=True)
class FOL_Func:
    name: str
    args: tuple = ()

@dataclass(frozen=True)
class FOL_Atom:
    name: str
    args: tuple = ()
    negative: bool = False

    def opposite(self):
        return FOL_Atom(self.name, self.args, not self.negative)

class FOL_Limit(Exception):
    pass

class FOL_ParseError(ValueError):
    pass

def FOL_check(deadline):
    if time.perf_counter() >= deadline:
        raise FOL_Limit('time limit')

def FOL_isvar(term):
    return isinstance(term, str) and term.startswith('?')

def FOL_termvars(term):
    if FOL_isvar(term):
        return {term}
    if isinstance(term, FOL_Func):
        return set().union(*(FOL_termvars(a) for a in term.args))
    return set()

def FOL_atomvars(atom):
    return set().union(*(FOL_termvars(a) for a in atom.args))

def FOL_free(formula, bound=frozenset()):
    if isinstance(formula, FOL_Atom):
        return FOL_atomvars(formula) - bound
    if formula[0] in ('forall', 'exists'):
        return FOL_free(formula[2], bound | {formula[1]})
    return set().union(*(FOL_free(a, bound) for a in formula[1:]))
import json
import math
import re
import itertools
import json
import math
import re
import time
from fractions import Fraction
import re
import re
from time import perf_counter
import importlib
import json
import subprocess
import sys
from collections import Counter
from pathlib import Path


In [3]:
class FOL_Parser:
    token = re.compile('\\s*(<->|<=>|->|—>|=>|!=|==|[∀∃¬∧∨⊕→⇒↔⇔⟷≠~!&|=(),.:]|\\??\\w+(?:[-\']\\w+)*|\'(?:[^\'\\\\]|\\\\.)*\'|"(?:[^"\\\\]|\\\\.)*")')
    simple_name = re.compile("\\??\\w+(?:[-']\\w+)*")
    quoted_name = re.compile('(?:\'(?:[^\'\\\\]|\\\\.)*\'|"(?:[^"\\\\]|\\\\.)*")')
    aliases = {'forall': 'forall', '∀': 'forall', 'exists': 'exists', '∃': 'exists', 'not': 'not', '¬': 'not', '~': 'not', '!': 'not', 'and': 'and', '∧': 'and', '&': 'and', 'or': 'or', '∨': 'or', '|': 'or', 'xor': 'xor', '⊕': 'xor', '->': 'implies', '—>': 'implies', '=>': 'implies', '→': 'implies', '⇒': 'implies', 'implies': 'implies', '<->': 'iff', '<=>': 'iff', '↔': 'iff', '⇔': 'iff', '⟷': 'iff', 'iff': 'iff', '!=': '!=', '≠': '!=', '==': '='}

    def __init__(self, text, deadline):
        self.tokens, self.raw_tokens, self.spans = ([], [], [])
        self.index, self.bound, self.deadline = (0, set(), deadline)
        self.text = text.strip() if isinstance(text, str) else ''
        if not self.text:
            raise self.error('Expected a nonempty formula string', 'syntax', 0)
        position = 0
        while position < len(self.text):
            FOL_check(deadline)
            match = self.token.match(self.text, position)
            if not match:
                while position < len(self.text) and self.text[position].isspace():
                    position += 1
                raise self.error(f'Unsupported symbol near {self.text[position:position + 20]!r}', 'lexical', position)
            raw = match.group(1)
            self.tokens.append(self.aliases.get(raw.lower(), raw))
            self.raw_tokens.append(raw)
            self.spans.append((match.start(1), match.end(1)))
            position = match.end()

    def error(self, message, kind='syntax', position=None):
        if position is None:
            position = self.spans[self.index][0] if self.index < len(self.spans) else len(self.text)
        error = FOL_ParseError(f'{kind.capitalize()} error at position {position}: {message}')
        error.kind, error.position = (kind, position)
        return error

    def peek(self):
        return self.tokens[self.index] if self.index < len(self.tokens) else None

    def take(self, expected=None):
        FOL_check(self.deadline)
        token = self.peek()
        if token is None or (expected is not None and token != expected):
            raise self.error(f"Expected {expected or 'a token'}, found {token!r}")
        self.index += 1
        return token

    def identifier(self, dotted=False):
        start = self.index
        value = self.take()
        raw = self.raw_tokens[start]
        quoted = self.quoted_name.fullmatch(raw) is not None
        if value in set(self.aliases.values()) or not (quoted or self.simple_name.fullmatch(raw)):
            raise self.error(f'Expected an identifier, found {raw!r}', position=self.spans[start][0])
        value = raw
        while dotted and (not quoted) and (self.peek() == '.') and (self.index + 1 < len(self.tokens)):
            following = self.raw_tokens[self.index + 1]
            adjacent = self.spans[self.index - 1][1] == self.spans[self.index][0] and self.spans[self.index][1] == self.spans[self.index + 1][0]
            if not adjacent or not self.simple_name.fullmatch(following) or following.startswith('?'):
                break
            self.take('.')
            self.take()
            value += '.' + following
        return value

    def term(self):
        name = self.identifier(dotted=True)
        if self.peek() != '(':
            return '?' + name.lstrip('?') if name in self.bound or FOL_isvar(name) else name
        self.take('(')
        args = []
        if self.peek() != ')':
            args.append(self.term())
            while self.peek() == ',':
                self.take(',')
                args.append(self.term())
        self.take(')')
        return FOL_Func(name, tuple(args)) if args else name

    def formula(self, level=0):
        operators = (('iff',), ('implies',), ('or', 'xor'), ('and',))
        if level == len(operators):
            return self.unary()
        left = self.formula(level + 1)
        while self.peek() in operators[level]:
            operator = self.take()
            right = self.formula(level if operator == 'implies' else level + 1)
            left = (operator, left, right)
        return left

    def unary(self):
        token = self.peek()
        if token == 'not':
            self.take()
            return ('not', self.unary())
        if token in ('forall', 'exists'):
            operator = self.take()
            variables = [self.identifier()]
            while self.peek() == ',':
                self.take(',')
                variables.append(self.identifier())
            if any((v.startswith(("'", '"')) or v.lstrip('?').isdigit() for v in variables)):
                raise self.error('Quantified variables must be names')
            wide = self.peek() in ('.', ':')
            if wide:
                self.take()
            previous = self.bound.copy()
            self.bound.update((v.lstrip('?') for v in variables))
            try:
                body = self.formula() if wide else self.unary()
            finally:
                self.bound = previous
            for variable in reversed(variables):
                body = (operator, '?' + variable.lstrip('?'), body)
            return body
        if token == '(':
            self.take()
            body = self.formula()
            self.take(')')
            return body
        head = self.term()
        if self.peek() in ('=', '!='):
            operator = self.take()
            return FOL_Atom('=', (head, self.term()), operator == '!=')
        return FOL_Atom(head.name, head.args) if isinstance(head, FOL_Func) else FOL_Atom(head)

    def parse(self):
        result = self.formula()
        if self.peek() == '.' and self.index == len(self.tokens) - 1:
            self.take()
        if self.peek() is not None:
            raise self.error(f'Unexpected trailing token {self.peek()!r}')
        binders = set()

        def collect(formula):
            if isinstance(formula, FOL_Atom):
                return
            if formula[0] in ('forall', 'exists'):
                binders.add(formula[1].lstrip('?'))
                collect(formula[2])
            else:
                for child in formula[1:]:
                    collect(child)

        def check(formula):
            if isinstance(formula, FOL_Atom):

                def term(value):
                    if isinstance(value, FOL_Func):
                        for argument in value.args:
                            term(argument)
                    elif value in binders:
                        raise self.error(f'Variable {value!r} occurs outside its quantifier scope', 'scope')
                for argument in formula.args:
                    term(argument)
            else:
                for child in formula[2:] if formula[0] in ('forall', 'exists') else formula[1:]:
                    check(child)
        collect(result)
        check(result)
        return result


In [4]:
def FOL_subterm(term, substitution):
    if FOL_isvar(term) and term in substitution:
        return FOL_subterm(substitution[term], substitution)
    if isinstance(term, FOL_Func):
        return FOL_Func(term.name, tuple((FOL_subterm(a, substitution) for a in term.args)))
    return term

def FOL_subatom(atom, substitution):
    return FOL_Atom(atom.name, tuple((FOL_subterm(a, substitution) for a in atom.args)), atom.negative)

def FOL_unify(first, second, complementary=False):
    if (first.name, len(first.args)) != (second.name, len(second.args)) or (first.negative != second.negative) != complementary:
        return None
    substitution, pending = ({}, list(zip(first.args, second.args)))
    while pending:
        a, b = (FOL_subterm(t, substitution) for t in pending.pop())
        if a == b:
            continue
        if FOL_isvar(a) or FOL_isvar(b):
            variable, term = (a, b) if FOL_isvar(a) else (b, a)
            if variable in FOL_termvars(term):
                return None
            substitution[variable] = term
        elif isinstance(a, FOL_Func) and isinstance(b, FOL_Func) and ((a.name, len(a.args)) == (b.name, len(b.args))):
            pending.extend(zip(a.args, b.args))
        else:
            return None
    return substitution

def FOL_clauses(formulas, deadline, limit=12000, diagnostics=None):
    formulas = list(formulas)
    used, counter = (set(), itertools.count())

    def collect(f):
        FOL_check(deadline)
        if isinstance(f, FOL_Atom):
            used.add(f.name)

            def term(t):
                used.add(t.name if isinstance(t, FOL_Func) else t)
                if isinstance(t, FOL_Func):
                    for a in t.args:
                        term(a)
            for a in f.args:
                term(a)
        elif f[0] in ('forall', 'exists'):
            used.add(f[1])
            collect(f[2])
        else:
            for a in f[1:]:
                collect(a)
    for formula in formulas:
        collect(formula)

    def fresh(prefix):
        while True:
            name = prefix + str(next(counter))
            if name not in used:
                used.add(name)
                return name

    def standardize(f, environment):
        FOL_check(deadline)
        if isinstance(f, FOL_Atom):

            def replace(t):
                if FOL_isvar(t):
                    return environment.get(t, t)
                if isinstance(t, FOL_Func):
                    return FOL_Func(t.name, tuple((replace(a) for a in t.args)))
                return t
            return FOL_Atom(f.name, tuple((replace(a) for a in f.args)), f.negative)
        if f[0] in ('forall', 'exists'):
            variable = fresh('?FOL_v')
            return (f[0], variable, standardize(f[2], {**environment, f[1]: variable}))
        return (f[0], *(standardize(a, environment) for a in f[1:]))
    standardized = [standardize(f, {}) for f in formulas]
    roots = []

    def replace_term(t, environment):
        if FOL_isvar(t):
            return environment.get(t, t)
        if isinstance(t, FOL_Func):
            return FOL_Func(t.name, tuple((replace_term(a, environment) for a in t.args)))
        return t

    def replace_formula(f, environment):
        FOL_check(deadline)
        if isinstance(f, FOL_Atom):
            return FOL_Atom(f.name, tuple((replace_term(t, environment) for t in f.args)), f.negative)
        if f[0] in ('forall', 'exists'):
            return (f[0], f[1], replace_formula(f[2], environment))
        return (f[0], *(replace_formula(a, environment) for a in f[1:]))

    def prepare_root(f, positive=True, environment=None):
        FOL_check(deadline)
        environment = {} if environment is None else environment
        if not isinstance(f, FOL_Atom):
            op = f[0]
            if op == 'not':
                return prepare_root(f[1], not positive, environment)
            if op in ('forall', 'exists'):
                if positive == (op == 'forall'):
                    value = f[1]
                else:
                    dependencies = set().union(*(FOL_termvars(replace_term(v, environment)) for v in FOL_free(f)))
                    value = FOL_Func(fresh('FOL_sk'), tuple(sorted(dependencies)))
                return prepare_root(f[2], positive, {**environment, f[1]: value})
            if positive and op == 'and' or (not positive and op == 'or'):
                prepare_root(f[1], positive, environment)
                prepare_root(f[2], positive, environment)
                return
        body = replace_formula(f, environment)
        roots.append(body if positive else ('not', body))
    for formula in standardized:
        prepare_root(formula)
    clauses, proxies, done, free_cache = (set(), {}, set(), {})
    instantiated_bodies = []

    def variables(f):
        key = id(f)
        if key not in free_cache:
            if isinstance(f, FOL_Atom):
                result = FOL_atomvars(f)
            elif f[0] in ('forall', 'exists'):
                result = variables(f[2]) - {f[1]}
            else:
                result = set().union(*(variables(a) for a in f[1:]))
            free_cache[key] = result
        return free_cache[key]

    def add(*literals):
        FOL_check(deadline)
        clause = frozenset(literals)
        if any((a.opposite() in clause for a in clause)):
            return
        if len(clauses) >= limit and clause not in clauses:
            raise FOL_Limit('definitional CNF clause limit')
        clauses.add(clause)

    def literal(f, positive=True):
        FOL_check(deadline)
        if isinstance(f, FOL_Atom):
            return f if positive else f.opposite()
        op = f[0]
        if op == 'not':
            return literal(f[1], not positive)
        key = id(f)
        if key not in proxies:
            proxies[key] = FOL_Atom(fresh('FOL_def'), tuple(sorted(variables(f))))
        proxy = proxies[key]
        signed = proxy if positive else proxy.opposite()
        if (key, positive) in done:
            return signed
        done.add((key, positive))
        guard = signed.opposite()
        if op == 'and':
            a, b = (literal(f[1], positive), literal(f[2], positive))
            if positive:
                add(guard, a)
                add(guard, b)
            else:
                add(guard, a, b)
        elif op == 'or':
            a, b = (literal(f[1], positive), literal(f[2], positive))
            if positive:
                add(guard, a, b)
            else:
                add(guard, a)
                add(guard, b)
        elif op == 'implies':
            a, b = (literal(f[1], not positive), literal(f[2], positive))
            if positive:
                add(guard, a, b)
            else:
                add(guard, a)
                add(guard, b)
        elif op in ('iff', 'xor'):
            a, na = (literal(f[1], True), literal(f[1], False))
            b, nb = (literal(f[2], True), literal(f[2], False))
            if positive == (op == 'iff'):
                add(guard, na, b)
                add(guard, a, nb)
            else:
                add(guard, a, b)
                add(guard, na, nb)
        elif op in ('forall', 'exists'):
            body, environment = (f, {})
            universals = list(proxy.args)
            while not isinstance(body, FOL_Atom) and body[0] in ('forall', 'exists'):
                FOL_check(deadline)
                quantifier, variable, next_body = body
                if positive == (quantifier == 'forall'):
                    value = variable
                    universals.append(value)
                else:
                    value = FOL_Func(fresh('FOL_sk'), tuple(universals))
                environment[variable] = value
                body = next_body
            body = replace_formula(body, environment)
            instantiated_bodies.append(body)
            add(guard, literal(body, positive))
        else:
            raise FOL_ParseError(f'Unsupported formula operator {op!r}')
        return signed
    for root in roots:
        add(literal(root))
    if diagnostics is not None:
        diagnostics.update(cnf='polarity-aware definitional', cnf_clauses=len(clauses), auxiliary_predicates=len(proxies))
    return list(clauses)

def FOL_equality(clauses, deadline):
    if not any((a.name == '=' for c in clauses for a in c)):
        return []
    predicates, functions = (set(), set())

    def visit(t):
        if isinstance(t, FOL_Func):
            functions.add((t.name, len(t.args)))
            for a in t.args:
                visit(a)
    for c in clauses:
        FOL_check(deadline)
        for a in c:
            predicates.add((a.name, len(a.args)))
            for t in a.args:
                visit(t)
    eq = lambda x, y: FOL_Atom('=', (x, y))
    x, y, z = ('?eqx', '?eqy', '?eqz')
    result = [{eq(x, x)}, {eq(x, y).opposite(), eq(y, x)}, {eq(x, y).opposite(), eq(y, z).opposite(), eq(x, z)}]
    for is_function, symbols in ((False, predicates - {('=', 2)}), (True, functions)):
        for name, arity in symbols:
            for position in range(arity):
                FOL_check(deadline)
                args = tuple(('?eqa' + str(i) for i in range(arity)))
                changed = list(args)
                changed[position] = y
                body = {eq(args[position], y).opposite()}
                if is_function:
                    body.add(eq(FOL_Func(name, args), FOL_Func(name, tuple(changed))))
                else:
                    body.update((FOL_Atom(name, args, True), FOL_Atom(name, tuple(changed))))
                result.append(body)
    return [frozenset(c) for c in result]


In [5]:
def FOL_ground_sat(clauses, deadline, limit=50000, diagnostics=None):
    constants, prepared, estimate = (set(), [], 0)
    for clause in clauses:
        FOL_check(deadline)
        for atom in clause:
            for term in atom.args:
                if isinstance(term, FOL_Func) and term.args:
                    if diagnostics is not None:
                        diagnostics['grounding_skipped'] = 'positive-arity function terms'
                    return None
                if not FOL_isvar(term):
                    constants.add(term)
    domain = sorted(constants or {'FOL_domain'}, key=repr)
    for clause in clauses:
        FOL_check(deadline)
        variables = sorted(set().union(*(FOL_atomvars(a) for a in clause)))
        estimate += len(domain) ** len(variables)
        if estimate > limit:
            if diagnostics is not None:
                diagnostics['grounding_skipped'] = 'grounding estimate exceeds limit'
            return None
        prepared.append((clause, variables))
    ground, names, inverse = (set(), {}, {})
    for clause, variables in prepared:
        for values in itertools.product(domain, repeat=len(variables)):
            FOL_check(deadline)
            literals, tautology = (set(), False)
            substitution = dict(zip(variables, values))
            for atom in clause:
                a = FOL_subatom(atom, substitution)
                if a.name == '=':
                    if len(a.args) != 2:
                        raise FOL_ParseError('Equality requires two arguments')
                    if a.args[0] == a.args[1]:
                        if not a.negative:
                            tautology = True
                            break
                        continue
                    a = FOL_Atom('=', tuple(sorted(a.args, key=repr)), a.negative)
                key = FOL_Atom(a.name, a.args)
                if key not in names:
                    number = len(names) + 1
                    names[key], inverse[number] = (number, key)
                number = names[key]
                literals.add(-number if a.negative else number)
            if not tautology and (not any((-a in literals for a in literals))):
                ground.add(frozenset(literals))

    def candidate_model():
        stack = [(list(ground), {})]
        while stack:
            current, assignment = stack.pop()
            while True:
                FOL_check(deadline)
                if any((not clause for clause in current)):
                    break
                if not current:
                    return assignment
                units = {next(iter(c)) for c in current if len(c) == 1}
                if any((-u in units for u in units)):
                    break
                if not units:
                    all_literals = set().union(*current)
                    units = {u for u in all_literals if -u not in all_literals}
                if units:
                    assignment.update({abs(u): u > 0 for u in units})
                    current = [frozenset((x for x in c if -x not in units)) for c in current if not c & units]
                    continue
                counts = {}
                for clause in current:
                    for literal in clause:
                        number = abs(literal)
                        counts[number] = counts.get(number, 0) + 1 / 2 ** min(len(clause), 20)
                variable = max(counts, key=lambda v: (counts[v], -v))
                stack.append((current + [frozenset([-variable])], dict(assignment)))
                current = current + [frozenset([variable])]
        return None

    def equality_conflict(assignment):
        parents = {term: term for term in domain}

        def find(term):
            while parents[term] != term:
                parents[term] = parents[parents[term]]
                term = parents[term]
            return term
        true_equalities = []
        for number, value in assignment.items():
            FOL_check(deadline)
            atom = inverse[number]
            if atom.name == '=' and value:
                left, right = map(find, atom.args)
                parents[left] = right
                true_equalities.append(number)
        assumptions, predicate_values = ({-n for n in true_equalities}, {})
        for number, value in assignment.items():
            FOL_check(deadline)
            atom = inverse[number]
            if atom.name == '=':
                if not value and find(atom.args[0]) == find(atom.args[1]):
                    return frozenset(assumptions | {number})
            else:
                key = (atom.name, tuple((find(term) for term in atom.args)))
                previous = predicate_values.get(key)
                if previous is not None and previous[0] != value:
                    old_value, old_number = previous
                    return frozenset(assumptions | {-number if value else number, -old_number if old_value else old_number})
                predicate_values[key] = (value, number)
        return None
    checks, lemmas = (0, 0)
    try:
        while True:
            FOL_check(deadline)
            model = candidate_model()
            if model is None:
                return False
            checks += 1
            conflict = equality_conflict(model)
            if conflict is None:
                return True
            if conflict in ground:
                raise AssertionError('Equality lemma did not exclude the candidate model')
            ground.add(conflict)
            lemmas += 1
            if len(ground) > limit:
                raise FOL_Limit('ground clause limit')
    finally:
        if diagnostics is not None:
            diagnostics.update(backend='finite DPLL with lazy equality', ground_clauses=len(ground), ground_atoms=len(names), theory_checks=checks, equality_lemmas=lemmas)

class FOL_FiniteUnsupported(ValueError):
    pass

class FOL_FiniteEncoder:

    def __init__(self, size, deadline, max_vars=12000):
        self.size, self.deadline, self.max_vars = (size, deadline, max_vars)
        self.next_id, self.clauses = (1, [])
        self.constants, self.predicates, self.gates = ({}, {}, {})

    def variable(self):
        FOL_check(self.deadline)
        if self.next_id > self.max_vars:
            raise FOL_Limit('finite-model variable limit')
        number = self.next_id
        self.next_id += 1
        return number

    @staticmethod
    def negate(value):
        return not value if isinstance(value, bool) else -value

    def add(self, *values):
        FOL_check(self.deadline)
        if any((v is True for v in values)):
            return
        clause = frozenset((v for v in values if v is not False))
        if not any((-v in clause for v in clause)):
            self.clauses.append(clause)
        if len(self.clauses) > 40000:
            raise FOL_Limit('finite-model clause limit')

    def conjunction(self, left, right):
        if left is False or right is False:
            return False
        if left is True:
            return right
        if right is True:
            return left
        if left == right:
            return left
        if left == -right:
            return False
        key = tuple(sorted((left, right)))
        if key not in self.gates:
            proxy = self.variable()
            self.gates[key] = proxy
            self.add(-proxy, left)
            self.add(-proxy, right)
            self.add(proxy, -left, -right)
        return self.gates[key]

    def disjunction(self, left, right):
        return self.negate(self.conjunction(self.negate(left), self.negate(right)))

    def fold(self, values, conjunction=True):
        result = True if conjunction else False
        operation = self.conjunction if conjunction else self.disjunction
        for value in values:
            result = operation(result, value)
        return result

    def term(self, term, environment):
        if isinstance(term, FOL_Func):
            raise FOL_FiniteUnsupported('explicit function terms in original input')
        if FOL_isvar(term):
            if term not in environment:
                raise FOL_FiniteUnsupported('unbound finite-model variable')
            return tuple((i == environment[term] for i in range(self.size)))
        if term not in self.constants:
            selectors = tuple((self.variable() for _ in range(self.size)))
            self.constants[term] = selectors
            self.add(*selectors)
            for index, first in enumerate(selectors):
                for second in selectors[index + 1:]:
                    self.add(-first, -second)
        return self.constants[term]

    def formula(self, formula, environment=None):
        FOL_check(self.deadline)
        environment = {} if environment is None else environment
        if isinstance(formula, FOL_Atom):
            terms = [self.term(term, environment) for term in formula.args]
            if formula.name == '=':
                if len(terms) != 2:
                    raise FOL_FiniteUnsupported('invalid equality arity')
                result = self.fold((self.conjunction(terms[0][i], terms[1][i]) for i in range(self.size)), False)
            else:
                options = []
                for values in itertools.product(range(self.size), repeat=len(terms)):
                    FOL_check(self.deadline)
                    selection = self.fold((term[v] for term, v in zip(terms, values)))
                    if selection is False:
                        continue
                    key = (formula.name, values)
                    if key not in self.predicates:
                        self.predicates[key] = self.variable()
                    options.append(self.conjunction(selection, self.predicates[key]))
                result = self.fold(options, False)
            return self.negate(result) if formula.negative else result
        operator = formula[0]
        if operator == 'not':
            return self.negate(self.formula(formula[1], environment))
        if operator in ('forall', 'exists'):
            return self.fold((self.formula(formula[2], {**environment, formula[1]: value}) for value in range(self.size)), operator == 'forall')
        left, right = (self.formula(formula[1], environment), self.formula(formula[2], environment))
        if operator == 'and':
            return self.conjunction(left, right)
        if operator == 'or':
            return self.disjunction(left, right)
        if operator == 'implies':
            return self.disjunction(self.negate(left), right)
        same = self.conjunction(self.disjunction(self.negate(left), right), self.disjunction(self.negate(right), left))
        if operator == 'iff':
            return same
        if operator == 'xor':
            return self.negate(same)
        raise FOL_FiniteUnsupported(f'unsupported finite-model operator {operator!r}')

def FOL_integer_sat(clauses, deadline, max_steps=100000):
    stack, steps = ([(list(clauses), {})], 0)
    while stack:
        current, assignment = stack.pop()
        while True:
            FOL_check(deadline)
            steps += 1
            if steps > max_steps:
                raise FOL_Limit('finite-model SAT step limit')
            if any((not clause for clause in current)):
                break
            if not current:
                return assignment
            units = {next(iter(c)) for c in current if len(c) == 1}
            if any((-u in units for u in units)):
                break
            if not units:
                literals = set().union(*current)
                units = {literal for literal in literals if -literal not in literals}
            if units:
                assignment.update({abs(u): u > 0 for u in units})
                current = [frozenset((x for x in c if -x not in units)) for c in current if not c & units]
                continue
            counts = {}
            for clause in current:
                for literal in clause:
                    variable = abs(literal)
                    counts[variable] = counts.get(variable, 0) + 1 / 2 ** min(len(clause), 20)
            variable = max(counts, key=lambda v: (counts[v], -v))
            stack.append((current + [frozenset([-variable])], dict(assignment)))
            current = current + [frozenset([variable])]
    return None

def FOL_verify_model(formulas, witness, deadline=None):
    try:
        size, constants = (witness['domain_size'], witness['constants'])
        if type(size) is not int or size < 1 or (not isinstance(constants, dict)):
            return False
        if any((type(v) is not int or not 0 <= v < size for v in constants.values())):
            return False
        entries = witness['true_predicates']
        if any((not isinstance(name, str) or any((type(v) is not int or not 0 <= v < size for v in args)) for name, args in entries)):
            return False
        truths = {(name, tuple(args)) for name, args in entries}

        def term(t, environment):
            if isinstance(t, FOL_Func):
                raise FOL_FiniteUnsupported('function term in model verification')
            return environment[t] if FOL_isvar(t) else constants[t]

        def evaluate(f, environment):
            if deadline is not None:
                FOL_check(deadline)
            if isinstance(f, FOL_Atom):
                values = tuple((term(t, environment) for t in f.args))
                if f.name == '=' and len(values) != 2:
                    raise FOL_FiniteUnsupported('invalid equality arity')
                result = values[0] == values[1] if f.name == '=' else (f.name, values) in truths
                return not result if f.negative else result
            operator = f[0]
            if operator == 'not':
                return not evaluate(f[1], environment)
            if operator in ('forall', 'exists'):
                values = (evaluate(f[2], {**environment, f[1]: value}) for value in range(size))
                return all(values) if operator == 'forall' else any(values)
            if operator == 'and':
                return evaluate(f[1], environment) and evaluate(f[2], environment)
            if operator == 'or':
                return evaluate(f[1], environment) or evaluate(f[2], environment)
            if operator == 'implies':
                return not evaluate(f[1], environment) or evaluate(f[2], environment)
            if operator == 'iff':
                return evaluate(f[1], environment) == evaluate(f[2], environment)
            if operator == 'xor':
                return evaluate(f[1], environment) != evaluate(f[2], environment)
            raise FOL_FiniteUnsupported('unknown model-verification operator')
        return all((evaluate(f, {}) for f in formulas))
    except (KeyError, TypeError, ValueError, RecursionError):
        return False

def FOL_finite_model(formulas, deadline, max_domain=3):
    formulas, closed = (list(formulas), [])
    for formula in formulas:
        for variable in sorted(FOL_free(formula)):
            formula = ('forall', variable, formula)
        closed.append(formula)
    started, attempted = (time.perf_counter(), [])
    try:
        for size in range(1, max_domain + 1):
            encoder = FOL_FiniteEncoder(size, deadline)
            for formula in closed:
                encoder.add(encoder.formula(formula))
            assignment = FOL_integer_sat(encoder.clauses, deadline)
            attempted.append(size)
            if assignment is None:
                continue
            constants = {name: next((i for i, selector in enumerate(selectors) if assignment.get(selector, False))) for name, selectors in encoder.constants.items()}
            witness = {'domain_size': size, 'constants': constants, 'true_predicates': [(name, list(args)) for (name, args), variable in encoder.predicates.items() if assignment.get(variable, False)]}
            if not FOL_verify_model(closed, witness, deadline):
                return {'found': False, 'reason': 'candidate failed independent verification', 'domains_attempted': attempted}
            return {'found': True, 'reason': 'independently verified original-AST model', 'witness': witness, 'domains_attempted': attempted, 'seconds': time.perf_counter() - started}
        reason = 'no witness within domain bound; FOL satisfiability remains unresolved'
    except (FOL_Limit, FOL_FiniteUnsupported, RecursionError) as exc:
        reason = str(exc)
    return {'found': False, 'reason': reason, 'domains_attempted': attempted, 'seconds': time.perf_counter() - started}

def FOL_canonical(clause):
    mapping = {}

    def rename(t):
        if FOL_isvar(t):
            return mapping.setdefault(t, '?c' + str(len(mapping)))
        if isinstance(t, FOL_Func):
            return FOL_Func(t.name, tuple((rename(a) for a in t.args)))
        return t
    return frozenset((FOL_Atom(a.name, tuple((rename(t) for t in a.args)), a.negative) for a in sorted(clause, key=repr)))

def FOL_unit_refute(clauses, deadline, limit=1500, diagnostics=None):
    from collections import deque
    seen, pending, units, others = (set(), deque(), set(), set())
    steps, reason = (0, 'inconclusive')

    def add(clause):
        FOL_check(deadline)
        clause = FOL_canonical(clause)
        if clause in seen or any((a.opposite() in clause for a in clause)):
            return False
        if len(seen) >= limit:
            raise FOL_Limit('unit prepass clause limit')
        seen.add(clause)
        pending.append(clause)
        return True
    try:
        for clause in clauses:
            add(clause)
        while pending:
            FOL_check(deadline)
            given = pending.popleft()
            if not given:
                reason = 'refutation'
                return True
            partners = units | others if len(given) == 1 else units
            for other in list(partners):
                variables = sorted(set().union(*(FOL_atomvars(a) for a in other)))
                fresh = frozenset((FOL_subatom(a, {v: '?unit' + str(i) for i, v in enumerate(variables)}) for a in other))
                for a in given:
                    for b in fresh:
                        steps += 1
                        FOL_check(deadline)
                        sub = FOL_unify(a, b, complementary=True)
                        if sub is not None:
                            child = frozenset((FOL_subatom(x, sub) for x in given - {a} | fresh - {b}))
                            if add(child) and (not child):
                                reason = 'refutation'
                                return True
            (units if len(given) == 1 else others).add(given)
        return False
    except FOL_Limit as exc:
        reason = str(exc)
        return False
    finally:
        if diagnostics is not None:
            diagnostics['unit_prepass'] = {'outcome': reason, 'clauses': len(seen), 'steps': steps}

def FOL_resolution(clauses, deadline, limit=4000):
    seen, pending, active = (set(), [], [])

    def add(clause):
        clause = FOL_canonical(clause)
        if any((a.opposite() in clause for a in clause)) or clause in seen:
            return
        if any((other <= clause for other in seen)):
            return
        if len(seen) >= limit:
            raise FOL_Limit('resolution clause limit')
        seen.add(clause)
        pending.append(clause)
    for clause in clauses:
        add(clause)
    selections = 0
    while pending:
        FOL_check(deadline)
        selections += 1
        index = 0 if selections % 5 == 0 else min(range(len(pending)), key=lambda i: len(pending[i]))
        given = pending.pop(index)
        if not given:
            return False
        literals = list(given)
        for i, a in enumerate(literals):
            for b in literals[i + 1:]:
                FOL_check(deadline)
                sub = FOL_unify(a, b)
                if sub is not None:
                    add(frozenset((FOL_subatom(x, sub) for x in given - {b})))
        for other in active + [given]:
            variables = sorted(set().union(*(FOL_atomvars(a) for a in other)))
            fresh = frozenset((FOL_subatom(a, {v: '?r' + str(i) for i, v in enumerate(variables)}) for a in other))
            for a in given:
                for b in fresh:
                    FOL_check(deadline)
                    sub = FOL_unify(a, b, complementary=True)
                    if sub is not None:
                        add(frozenset((FOL_subatom(x, sub) for x in given - {a} | fresh - {b})))
        active.append(given)
    return True

def solve_fol(premises, query, timeout=2.0):
    start = time.perf_counter()
    result = {'flag': False, 'status': 'unsupported', 'resolved': False, 'trace': [], 'seconds': 0.0, 'reason': '', 'category': 'unsupported_input', 'details': {'probes': []}}
    try:
        if not isinstance(premises, (list, tuple)) or not all((isinstance(p, str) for p in premises)):
            raise FOL_ParseError('premises must be a list of formula strings')
        timeout_seconds = float(timeout)
        if not math.isfinite(timeout_seconds) or timeout_seconds < 0:
            raise ValueError('timeout must be a finite, nonnegative number')
        deadline = start + timeout_seconds
        formulas, blanks = ([], [])
        for index, text in enumerate(premises, 1):
            if not text.strip():
                blanks.append(index)
                continue
            try:
                formulas.append(FOL_Parser(text.strip(), deadline).parse())
            except FOL_ParseError as exc:
                result['details']['parser_error'] = {'location': 'premise', 'index': index, 'formula': text, 'kind': getattr(exc, 'kind', 'syntax'), 'position': getattr(exc, 'position', None)}
                raise FOL_ParseError(f'Premise {index}: {exc}') from exc
        result['details']['ignored_blank_premises'] = blanks
        try:
            question = FOL_Parser(query, deadline).parse()
        except FOL_ParseError as exc:
            result['details']['parser_error'] = {'location': 'query', 'formula': query, 'kind': getattr(exc, 'kind', 'syntax'), 'position': getattr(exc, 'position', None)}
            raise FOL_ParseError(f'Query: {exc}') from exc
        if FOL_free(question):
            result['details']['parser_error'] = {'location': 'query', 'formula': query, 'kind': 'scope', 'position': None}
            raise FOL_ParseError('Explicitly quantify free variables in the query')
        answers = []
        for index, probe in enumerate((('not', question), question)):
            local_deadline = min(deadline, time.perf_counter() + max(0.0, deadline - time.perf_counter()) / 2) if index == 0 else deadline
            label = 'KB + not(query)' if index == 0 else 'KB + query'
            details = {'polarity': label}
            probe_start = time.perf_counter()
            try:
                clauses = FOL_clauses(formulas + [probe], local_deadline, diagnostics=details)
                prepass_deadline = min(local_deadline, time.perf_counter() + min(0.04, max(0.0, local_deadline - time.perf_counter()) * 0.1))
                proved = FOL_unit_refute(clauses, prepass_deadline, diagnostics=details)
                backend = 'proof-only unit resolution'
                if proved:
                    satisfiable = False
                else:
                    satisfiable = FOL_ground_sat(clauses, local_deadline, diagnostics=details)
                    backend = 'finite DPLL with lazy equality'
                    if satisfiable is None:
                        remaining = max(0.0, local_deadline - time.perf_counter())
                        model_deadline = min(local_deadline, time.perf_counter() + min(0.2, remaining * 0.35))
                        model_search = FOL_finite_model(formulas + [probe], model_deadline)
                        details['finite_model_search'] = model_search
                        if model_search['found']:
                            satisfiable = True
                            backend = 'independently verified finite interpretation'
                        else:
                            clauses += FOL_equality(clauses, local_deadline)
                            satisfiable = FOL_resolution(clauses, local_deadline)
                            backend = 'first-order resolution with equality axioms'
                answers.append(not satisfiable)
                details.update(outcome='sat' if satisfiable else 'unsat', backend=backend)
                result['trace'].append(f"{label}: {('contradiction proved' if not satisfiable else 'satisfiability established')} by {backend} ({len(clauses)} clauses).")
            except FOL_Limit as exc:
                answers.append(None)
                details.update(outcome='unresolved', limit=str(exc))
                result['trace'].append(f'{label} unresolved: {exc}.')
            details['seconds'] = time.perf_counter() - probe_start
            result['details']['probes'].append(details)
        yes, no = answers
        result['status'] = 'both' if yes is True and no is True else 'true' if yes is True else 'false' if no is True else 'unknown' if yes is False and no is False else 'timeout'
        result['flag'] = yes is True
        result['resolved'] = yes is not None
        if yes is True:
            result['category'] = 'proved_entailment'
            result['reason'] = 'Query proved by classical refutation.'
        elif yes is False:
            result['category'] = 'proved_non_entailment'
            result['reason'] = 'A model of KB + not(query) establishes that the query is not entailed.'
        else:
            limits = ' '.join((p.get('limit', '') for p in result['details']['probes']))
            result['category'] = 'inference_time_limit' if 'time limit' in limits else 'inference_clause_limit'
            result['reason'] = 'Query entailment unresolved: False is a binary fallback, not an established answer.'
    except FOL_Limit as exc:
        result.update(status='timeout', reason=str(exc), category='parsing_time_limit')
    except (FOL_ParseError, ValueError, TypeError, RecursionError) as exc:
        result['reason'] = str(exc)
        if isinstance(exc, RecursionError):
            result['category'] = 'unsupported_formula_depth'
        elif isinstance(exc, FOL_ParseError):
            kind = result['details'].get('parser_error', {}).get('kind', getattr(exc.__cause__, 'kind', getattr(exc, 'kind', 'syntax')))
            result['category'] = 'parse_' + kind
    result['seconds'] = time.perf_counter() - start
    return result


In [6]:
class RT_Unsupported(ValueError):
    pass

class RT_Timeout(RuntimeError):
    pass

def RT_check(deadline):
    if time.perf_counter() >= deadline:
        raise RT_Timeout('Reasoning time limit reached.')

def RT_answer(start, flag=False, status='unsupported', resolved=False, trace=None, reason=''):
    return {'flag': bool(flag), 'status': status, 'resolved': bool(resolved), 'trace': trace or [], 'reason': reason, 'seconds': time.perf_counter() - start}
RT_RULE_PATTERNS = ('if\\s+(.+?),?\\s+then\\s+(.+)', 'assuming\\s+(.+?),\\s*we can say\\s+(.+)', 'based on\\s+(.+?),\\s*it is evident that\\s+(.+)', 'given that\\s+(.+?),\\s*it follows that\\s+(.+)', 'when\\s+(.+?),\\s*we can conclude\\s+(.+)')

def RT_atomic(text):
    if not isinstance(text, str):
        raise RT_Unsupported('Each ground sentence must be text.')
    atom = re.sub('\\s+', ' ', text.strip().rstrip('.?!').strip()).casefold()
    if not atom or re.search('[.;:?!]', atom):
        raise RT_Unsupported('Expected one nonempty ground sentence.')
    if re.search("\\b(and|or|not|never|no|neither|nor|all|every|everyone|someone|something|somebody|anyone|anything|they|them|he|she|it)\\b|n't", atom):
        raise RT_Unsupported('Negation, conjunction, pronouns or quantification need a formal representation in this simple adapter.')
    if any((re.fullmatch(pattern, atom, re.I) for pattern in RT_RULE_PATTERNS)):
        raise RT_Unsupported('A rule cannot be used as an atomic fact.')
    return atom

def solve_ruletaker(context, query, semantics='ground_horn', timeout=2.0):
    start = time.perf_counter()
    try:
        deadline = start + max(0.0, float(timeout))
        RT_check(deadline)
        if semantics != 'ground_horn':
            raise RT_Unsupported('This adapter supports ground_horn semantics only.')
        if not isinstance(context, str) or not isinstance(query, str):
            raise RT_Unsupported('Context and query must be strings.')
        sections = re.fullmatch('\\s*Facts\\s*:\\s*(.*?)\\s*Rules\\s*:\\s*(.*?)\\s*', context, re.I | re.S)
        if not sections:
            raise RT_Unsupported('Expected the jise format: Facts: ... Rules: ...')
        facts = [s.strip() for s in sections[1].split('.') if s.strip()]
        sentences = [s.strip() for s in sections[2].split('.') if s.strip()]
        if len(facts) + len(sentences) > 10000:
            raise RT_Unsupported('Ground theory exceeds the 10,000-sentence limit.')
        known, provenance = (set(), {})
        for fact in facts:
            RT_check(deadline)
            atom = RT_atomic(fact)
            known.add(atom)
            provenance[atom] = None
        rules = []
        for sentence in sentences:
            RT_check(deadline)
            match = next((m for pattern in RT_RULE_PATTERNS if (m := re.fullmatch(pattern, sentence, re.I))), None)
            if match is None:
                raise RT_Unsupported('Unsupported rule template: ' + sentence)
            rules.append((RT_atomic(match[1]), RT_atomic(match[2])))
        target = RT_atomic(query)
        changed = True
        while changed:
            changed = False
            for premise, conclusion in rules:
                RT_check(deadline)
                if premise in known and conclusion not in known:
                    known.add(conclusion)
                    provenance[conclusion] = premise
                    changed = True
        trace = []
        if target in known:
            chain, current = ([], target)
            while current is not None:
                RT_check(deadline)
                chain.append(current)
                current = provenance[current]
            for atom in reversed(chain[-200:]):
                parent = provenance[atom]
                trace.append(atom + (' [given fact]' if parent is None else ' [derived from: ' + parent + ']'))
            if len(chain) > 200:
                trace.append('Trace display limited to 200 steps.')
        entailed = target in known
        return RT_answer(start, entailed, 'true' if entailed else 'false', True, trace, 'Derived by ground Horn forward chaining.' if entailed else 'Not derived after complete definite-Horn closure; not entailed.')
    except RT_Timeout as exc:
        return RT_answer(start, status='timeout', reason=str(exc))
    except (RT_Unsupported, TypeError, ValueError) as exc:
        return RT_answer(start, reason=str(exc))


In [7]:
class CB_Unsupported(ValueError):
    pass

def CB_expression(text):
    text = re.sub('\\btogether\\b', '', text, flags=re.I).strip().lower()
    tokens = re.findall('[a-z_][a-z_0-9]*|[()]', text)
    if re.sub('\\s+', '', text) != ''.join(tokens):
        raise CB_Unsupported('Unsupported Boolean expression: ' + text)
    position = 0

    def atom():
        nonlocal position
        if position >= len(tokens):
            raise CB_Unsupported('Incomplete Boolean expression.')
        token = tokens[position]
        position += 1
        if token == 'not':
            return ('not', atom())
        if token == '(':
            value = disjunction()
            if position >= len(tokens) or tokens[position] != ')':
                raise CB_Unsupported('Unclosed Boolean parenthesis.')
            position += 1
            return value
        if token in {'and', 'or', ')', 'cause', 'causes'}:
            raise CB_Unsupported('Unexpected Boolean token: ' + token)
        return ('var', token)

    def conjunction():
        nonlocal position
        parts = [atom()]
        while position < len(tokens) and tokens[position] == 'and':
            position += 1
            parts.append(atom())
        return parts[0] if len(parts) == 1 else ('and', tuple(parts))

    def disjunction():
        nonlocal position
        parts = [conjunction()]
        while position < len(tokens) and tokens[position] == 'or':
            position += 1
            parts.append(conjunction())
        return parts[0] if len(parts) == 1 else ('or', tuple(parts))
    result = disjunction()
    if position != len(tokens):
        raise CB_Unsupported('Unconsumed Boolean expression.')
    return result

def CB_variables(expr):
    if expr[0] == 'var':
        return {expr[1]}
    if expr[0] == 'not':
        return CB_variables(expr[1])
    return set().union(*(CB_variables(child) for child in expr[1]))

def CB_evaluate(expr, values):
    op, body = expr
    if op == 'var':
        return values[body]
    if op == 'not':
        return not CB_evaluate(body, values)
    results = (CB_evaluate(child, values) for child in body)
    return all(results) if op == 'and' else any(results)

class CB_Diagnostic(ValueError):

    def __init__(self, code, message, **details):
        super().__init__(message)
        self.code, self.details = (code, details)

def CB_v3_literals(text, role):
    text = re.sub('\\b(?:occurs?|occurred)\\b', '', text.lower()).strip(' .?')
    assignments = {}
    for part in re.split('\\s+and\\s+|\\s*,\\s*', text):
        part = re.sub('^and\\s+', '', part.strip())
        match = re.fullmatch('(not\\s+)?([a-z_][a-z_0-9]*)(\\s+not)?', part)
        if not match or (match[1] and match[3]):
            raise CB_Diagnostic('unsupported_causal_grammar', f'Expected a Boolean literal in {role}: {part!r}')
        name, value = (match[2], not bool(match[1] or match[3]))
        if name in assignments and assignments[name] != value:
            raise CB_Diagnostic('conflicting_' + role, f'Both values of {name!r} were supplied in {role}.', variable=name)
        assignments[name] = value
    return assignments

def CB_v3_merge(target, additions, role):
    conflicts = sorted((name for name, value in additions.items() if name in target and target[name] != value))
    if conflicts:
        raise CB_Diagnostic('conflicting_' + role, 'Conflicting assignments in ' + role + ': ' + ', '.join(conflicts), conflicts=conflicts, first=dict(target), second=dict(additions))
    target.update(additions)

def CB_v3_has_cycle(equations):
    pending = {name: CB_variables(expr) & equations.keys() for name, expr in equations.items()}
    while pending:
        ready = {name for name, dependencies in pending.items() if not dependencies}
        if not ready:
            return True
        pending = {name: dependencies - ready for name, dependencies in pending.items() if name not in ready}
    return False

def CB_parse(given_info, question, deadline, observation_mode='factual'):
    if observation_mode not in {'factual', 'counterfactual'}:
        raise CB_Diagnostic('invalid_observation_mode', 'observation_mode must be factual or counterfactual.')
    if not isinstance(given_info, str) or not isinstance(question, str):
        raise CB_Diagnostic('invalid_input', 'Causal context and question must be strings.')
    if len(given_info) + len(question) > 200000:
        raise CB_Diagnostic('input_limit', 'Causal input exceeds 200,000 characters.')
    equations, observations, priors = ({}, {}, {})
    source = re.sub('^\\s*We know that\\s+', '', given_info, flags=re.I)
    for sentence in re.split('\\.(?!\\d)', source):
        RT_check(deadline)
        sentence = sentence.strip()
        if not sentence:
            continue
        observed = re.fullmatch('We observed\\s+(.+)', sentence, re.I)
        prior = re.fullmatch('([a-z_][a-z_0-9]*)\\s*~\\s*Bern\\(\\s*([0-9.]+)\\s*\\)', sentence, re.I)
        if observed:
            additions = CB_v3_literals(observed[1], 'observations')
            CB_v3_merge(observations, additions, 'observations')
            continue
        if prior:
            name, probability = (prior[1].lower(), Fraction(prior[2]))
            if not 0 <= probability <= 1:
                raise CB_Diagnostic('invalid_root_prior', 'Bernoulli parameter must be in [0,1].')
            if name in priors and priors[name] != probability:
                raise CB_Diagnostic('conflicting_root_prior', 'Different priors supplied for ' + name)
            priors[name] = probability
            continue
        clauses = []
        for fragment in sentence.split(','):
            if re.search('\\bcauses?\\b', fragment, re.I):
                clauses.append(fragment.strip())
            elif clauses:
                clauses[-1] += ', ' + fragment.strip()
            else:
                raise CB_Diagnostic('unsupported_causal_grammar', 'Missing causal equation: ' + fragment)
        for clause in clauses:
            RT_check(deadline)
            clause = re.sub('^and\\s+', '', clause, flags=re.I)
            match = re.fullmatch('(.+?)\\s+causes?\\s+(.+)', clause, re.I)
            if not match:
                raise CB_Diagnostic('unsupported_causal_grammar', 'Unsupported clause: ' + clause)
            expression = CB_expression(match[1])
            for name, positive in CB_v3_literals(match[2], 'equation_outputs').items():
                equation = expression if positive else ('not', expression)
                if name in equations and equations[name] != equation:
                    raise CB_Diagnostic('multiple_structural_equations', 'Different equations supplied for ' + name, variable=name)
                equations[name] = equation
    if not equations:
        raise CB_Diagnostic('missing_structural_equations', 'No causal equations supplied.')
    regular = re.fullmatch('\\s*Would (.+?) occur if (.+?)\\?\\s*', question, re.I)
    nested = re.fullmatch('\\s*Assume (.+?), and based on this assumption, further suppose (.+?)\\.\\s+Would (.+?) occur\\?\\s*', question, re.I)
    original = {}
    if regular:
        condition = re.split('\\s+instead of\\s+', regular[2], flags=re.I)
        if len(condition) > 2:
            raise CB_Diagnostic('unsupported_causal_grammar', 'Multiple instead-of clauses.')
        intervention = CB_v3_literals(condition[0], 'interventions')
        target = CB_v3_literals(regular[1], 'target')
        if len(condition) == 2:
            original = CB_v3_literals(condition[1], 'factual_evidence')
        question_form = 'constant_intervention'
    elif nested:
        intervention = CB_v3_literals(nested[1], 'interventions')
        intervention.update(CB_v3_literals(nested[2], 'interventions'))
        target = CB_v3_literals(nested[3], 'target')
        question_form = 'sequential_constant_interventions'
    else:
        raise CB_Diagnostic('unsupported_question_grammar', 'Expected a constant intervention or sequential-constant question.')
    if len(target) != 1:
        raise CB_Diagnostic('unsupported_target', 'Exactly one target literal is required.')
    variables = set(equations).union(*(CB_variables(expr) for expr in equations.values()), priors)
    for role, assignments in (('intervention', intervention), ('target', target), ('observation', observations), ('factual_evidence', original)):
        missing = sorted(set(assignments) - variables)
        if missing:
            raise CB_Diagnostic('undefined_' + role + '_variable', 'Undeclared ' + role + ' variable(s): ' + ', '.join(missing), variables=missing)
    factual, counterfactual = (dict(original), {})
    if observation_mode == 'factual':
        CB_v3_merge(factual, observations, 'factual_evidence')
    else:
        counterfactual = dict(observations)
    return {'equations': equations, 'variables': variables, 'factual': factual, 'counterfactual': counterfactual, 'intervention': intervention, 'target': target, 'priors': priors, 'observations': observations, 'original': original, 'question_form': question_form}

def CB_worlds(equations, roots, intervention, deadline):
    values = dict(roots)
    values.update(intervention)
    pending = {name: expr for name, expr in equations.items() if name not in intervention}
    dependencies = {name: CB_variables(expr) for name, expr in pending.items()}
    while pending:
        RT_check(deadline)
        ready = [name for name in pending if dependencies[name] <= values.keys()]
        if not ready:
            break
        for name in ready:
            RT_check(deadline)
            values[name] = CB_evaluate(pending.pop(name), values)
    names = sorted(pending)
    for bits in itertools.product((False, True), repeat=len(names)):
        RT_check(deadline)
        world = {**values, **dict(zip(names, bits))}
        if all((world[name] == CB_evaluate(expr, world) for name, expr in pending.items())):
            yield world

def CB_resolve_marginals(problem, deadline):
    equations, variables = (problem['equations'], problem['variables'])
    declared = dict(problem['priors'])
    endogenous = {name: probability for name, probability in declared.items() if name in equations}
    roots = sorted(variables - equations.keys())
    info = {'interpretation': 'unconditional marginal; deterministic equations unchanged', 'declared_marginals': {name: str(probability) for name, probability in declared.items()}, 'root_marginals': {name: str(probability) for name, probability in declared.items() if name not in equations}, 'endogenous_marginals': sorted(endogenous)}
    if not endogenous:
        return (problem, info)
    if len(roots) > 1:
        raise CB_Diagnostic('unsupported_endogenous_prior', 'Endogenous marginals with multiple roots need a joint distribution; no independence assumption was introduced.', roots=roots)
    if CB_v3_has_cycle(equations):
        raise CB_Diagnostic('unsupported_endogenous_prior', 'Endogenous marginals in a cyclic model need an equilibrium distribution.')
    worlds = []
    for bits in itertools.product((False, True), repeat=len(roots)):
        RT_check(deadline)
        world = next(CB_worlds(equations, dict(zip(roots, bits)), {}, deadline), None)
        if world is None:
            raise CB_Diagnostic('invalid_marginal_model', 'The acyclic marginal model has no interpretation.')
        worlds.append(world)
    theta = declared.get(roots[0]) if roots else None
    constraints = []
    for name, probability in endogenous.items():
        RT_check(deadline)
        low, high = (worlds[0][name], worlds[-1][name])
        if low == high:
            expected = Fraction(int(low))
            if probability != expected:
                raise CB_Diagnostic('conflicting_marginal_priors', f'{name} is constant {int(low)}, incompatible with its declared marginal.', variable=name, declared=str(probability), required=str(expected))
            constraints.append(f'{name} is constant {int(low)}; marginal validated.')
            continue
        candidate = probability if not low and high else 1 - probability
        relation = 'identity' if not low and high else 'negation'
        if theta is not None and theta != candidate:
            raise CB_Diagnostic('conflicting_marginal_priors', 'The supplied marginals require different probabilities for the same root.', root=roots[0], first=str(theta), second=str(candidate), variable=name)
        theta = candidate
        constraints.append(f'{name} is the {relation} of {roots[0]}; P({roots[0]}=True)={theta}.')
    updated = dict(problem)
    root_priors = {name: probability for name, probability in declared.items() if name not in equations}
    if roots and theta is not None:
        root_priors[roots[0]] = theta
    updated['priors'] = root_priors
    info.update(root_marginals={name: str(probability) for name, probability in root_priors.items()}, root_probability_identified=not roots or theta is not None, constraints=constraints, unconditioned_models_examined=len(worlds), inference_order='Marginals inferred before factual or counterfactual conditioning.')
    if roots and theta is None:
        info['unidentified_root_policy'] = 'Keep both root values; conclude only if all compatible worlds agree.'
    return (updated, info)

def solve_causal(given_info, question, timeout=2.0, observation_mode='factual'):
    start = time.perf_counter()
    diagnostics = {'observation_mode': observation_mode, 'equation_interpretation': 'deterministic_assignment', 'prior_interpretation': 'unconditional marginals; restricted exact single-root support', 'root_support': 'Cartesian Bernoulli support; no probability estimate', 'factual_root_contexts': 0, 'factual_worlds_examined': 0, 'counterfactual_worlds_examined': 0, 'conditioned_worlds': 0}
    trace = []

    def answer(code, reason, flag=False, status='unsupported', resolved=False, **details):
        result = RT_answer(start, flag, status, resolved, trace, reason)
        result.update(reason_code=code, diagnostics={**diagnostics, **details}, proof_kind='unanimous_counterfactual_models' if resolved else 'none')
        return result
    try:
        timeout = float(timeout)
        if not math.isfinite(timeout) or timeout < 0:
            raise CB_Diagnostic('invalid_timeout', 'timeout must be finite and nonnegative.')
        deadline = start + timeout
        RT_check(deadline)
        problem = CB_parse(given_info, question, deadline, observation_mode)
        equations, variables = (problem['equations'], problem['variables'])
        if len(variables) > 16:
            raise CB_Diagnostic('variable_limit', 'More than 16 causal variables.')
        problem, marginal_info = CB_resolve_marginals(problem, deadline)
        diagnostics['marginal_prior_analysis'] = marginal_info
        cyclic = CB_v3_has_cycle(equations)
        diagnostics.update(has_cycle=cyclic, factual_evidence=problem['factual'], counterfactual_evidence=problem['counterfactual'], interventions=problem['intervention'], question_form=problem['question_form'], nested_scope='sequential constants only')
        roots = sorted(variables - equations.keys())
        target_name, target_value = next(iter(problem['target'].items()))
        outcomes, factual_models = (set(), False)
        for bits in itertools.product((False, True), repeat=len(roots)):
            RT_check(deadline)
            root_values = dict(zip(roots, bits))
            if any((problem['priors'].get(name) == 0 and value or (problem['priors'].get(name) == 1 and (not value)) for name, value in root_values.items())):
                continue
            admissible = False
            for world in CB_worlds(equations, root_values, {}, deadline):
                factual_models = True
                diagnostics['factual_worlds_examined'] += 1
                if all((world[name] == value for name, value in problem['factual'].items())):
                    admissible = True
                    break
            if not admissible:
                continue
            diagnostics['factual_root_contexts'] += 1
            has_counterfactual_model = False
            for world in CB_worlds(equations, root_values, problem['intervention'], deadline):
                has_counterfactual_model = True
                diagnostics['counterfactual_worlds_examined'] += 1
                if not all((world[name] == value for name, value in problem['counterfactual'].items())):
                    continue
                diagnostics['conditioned_worlds'] += 1
                outcomes.add(world[target_name] == target_value)
                if len(trace) < 3:
                    trace.append(f"Roots {root_values}; intervention {problem['intervention']}; target {target_name}={world[target_name]}; observation timing={observation_mode}.")
            if not has_counterfactual_model:
                return answer('cyclic_no_counterfactual_model' if cyclic else 'no_counterfactual_model', 'An admissible factual root context has no post-intervention fixed point.', failed_root_context=root_values)
        if not factual_models:
            return answer('cyclic_no_factual_model' if cyclic else 'no_factual_model', 'The structural equations have no factual fixed point on positive root support.')
        if not diagnostics['factual_root_contexts']:
            return answer('conflicting_factual_evidence', 'No factual model satisfies the supplied observations and/or instead-of condition. Evidence was not silently moved into the counterfactual world.')
        if not outcomes:
            return answer('conflicting_counterfactual_evidence', 'Post-intervention models exist, but none satisfy the counterfactual-world observations.')
        if len(outcomes) != 1:
            return answer('ambiguous_counterfactual_outcome', 'Admissible conditioned counterfactual worlds disagree on the target.', status='unknown')
        flag = True in outcomes
        return answer('unanimous_counterfactual_outcome', 'All admissible conditioned counterfactual models agree on the target.', flag=flag, status='true' if flag else 'false', resolved=True)
    except CB_Diagnostic as exc:
        return answer(exc.code, str(exc), **exc.details)
    except RT_Timeout:
        return answer('reasoning_timeout', 'Causal reasoning time limit reached.', status='timeout')
    except (CB_Unsupported, TypeError, ValueError, RecursionError) as exc:
        return answer('unsupported_causal_grammar', str(exc))


In [8]:
INPUT_ALIASES = {'id': ('id', 'question_id', 'example_id', 'example-id', 'problem_id', 'query_id', 'qid', 'record_id', 'uid'), 'premises': ('premises', 'premises-FOL', 'premises_fol', 'context'), 'query': ('query', 'statement', 'conclusion-FOL', 'conclusion_fol', 'question'), 'context': ('context',), 'given_info': ('given_info',), 'question': ('question',)}

def decode_premises(value):
    if isinstance(value, str):
        if value.lstrip().startswith('['):
            value = json.loads(value)
        else:
            value = [line.strip() for line in value.replace('\\n', '\n').splitlines() if line.strip()]
    if not isinstance(value, (list, tuple)) or not all((isinstance(item, str) for item in value)):
        raise ValueError('Premises must be a list of symbolic formula strings, a JSON list string, or one formula per line')
    return list(value)

def input_field(record, name, fields=None, decoder=None):
    if fields and name in fields:
        keys = (fields[name],)
    else:
        keys = INPUT_ALIASES[name]
    present = [key for key in keys if key in record and record[key] is not None]
    if not present:
        raise ValueError(f'Missing {name}; expected one of {keys}')
    values = [decoder(record[key]) if decoder else record[key] for key in present]
    if any((value != values[0] for value in values[1:])):
        raise ValueError(f'Conflicting {name} fields {present}; select the intended column explicitly in TA_FIELDS')
    return values[0]

def adapt_problem(record, mode='fol', fields=None, causal_observation_mode='factual'):
    if not isinstance(record, dict):
        raise ValueError('Each record must be a dictionary')
    if mode == 'fol':
        has_premises = any((key in record and record[key] is not None for key in ((fields or {}).get('premises'), *INPUT_ALIASES['premises']) if key is not None))
        if has_premises:
            premises = input_field(record, 'premises', fields, decode_premises)
        elif 'facts' in record or 'rules' in record:
            premises = decode_premises(record.get('facts', [])) + decode_premises(record.get('rules', []))
        else:
            raise ValueError('Formal input needs premises, or facts/rules containing symbolic formulas')
        query = input_field(record, 'query', fields)
        if not isinstance(query, str) or not query.strip():
            raise ValueError('The formal query must be a nonempty formula string')
        return {'premises': premises, 'query': query}
    if mode == 'ruletaker':
        return {'context': input_field(record, 'context', fields), 'query': input_field(record, 'query', fields)}
    if mode == 'causal':
        return {'given_info': input_field(record, 'given_info', fields), 'question': input_field(record, 'question', fields), 'observation_mode': record.get('observation_mode', causal_observation_mode)}
    raise ValueError('mode must be fol, ruletaker, or causal')

def _predict_f1(record, mode='fol', semantics='classical', timeout=2.0, fields=None, causal_observation_mode='factual'):
    started = perf_counter()
    engine = {'fol': 'fol', 'ruletaker': 'ground_horn', 'causal': 'counterbench'}.get(mode, 'unsupported')
    problem = None
    try:
        problem = adapt_problem(record, mode, fields, causal_observation_mode)
        if mode == 'fol':
            if semantics != 'classical':
                raise ValueError('fol mode requires classical semantics')
            result = solve_fol(problem['premises'], problem['query'], timeout)
        elif mode == 'ruletaker':
            result = solve_ruletaker(problem['context'], problem['query'], semantics, timeout)
        else:
            if semantics != 'classical':
                raise ValueError('causal mode requires classical semantics')
            result = solve_causal(problem['given_info'], problem['question'], timeout, observation_mode=problem['observation_mode'])
        if not isinstance(result.get('flag'), bool):
            raise ValueError('The engine did not produce a Boolean flag')
    except Exception as error:
        result = {'flag': False, 'status': 'unsupported', 'resolved': False, 'trace': [], 'category': 'input_schema', 'reason': f'{type(error).__name__}: {error}'}
    if 'category' not in result:
        result['category'] = result.get('reason_code', 'established' if result.get('resolved') else result.get('status', 'unsupported'))
    if mode == 'causal':
        result['observation_mode'] = result.get('diagnostics', {}).get('observation_mode', causal_observation_mode)
    if problem is not None:
        result['input'] = problem
    return dict(result, engine=engine, seconds=perf_counter() - started)
AUTO_FOL_ALIASES = {'premises': ('premises', 'premises-FOL', 'premises_fol', 'context'), 'query': ('query', 'statement', 'conclusion-FOL', 'conclusion_fol', 'question', 'conclusion', 'hypothesis', 'goal'), 'kb': ('knowledge_base', 'kb'), 'facts': ('facts',), 'rules': ('rules',), 'clauses': ('clauses',)}
AUTO_FOL_REFERENCE_FIELDS = {'label', 'labels', 'answer', 'answers', 'gold', 'gold_label', 'target', 'flag', 'prediction', 'predictions', 'predicted_label', 'predicted_labels', 'predicted_flag', 'reasoning', 'explanation', 'explanations', 'expected', 'expected_answer', 'expected_label', 'ground_truth', 'reference_answer'}
AUTO_FOL_FIELD_ROLES = set(AUTO_FOL_ALIASES) | {'id', 'context', 'given_info', 'question'}
AUTO_FOL_OPERATORS = {'and': 'and', '∧': 'and', '&': 'and', 'or': 'or', '∨': 'or', '|': 'or', 'not': 'not', '¬': 'not', '~': 'not', '!': 'not', 'implies': 'implies', '->': 'implies', '→': 'implies', 'iff': 'iff', '<->': 'iff', '↔': 'iff', 'xor': 'xor', '⊕': 'xor', 'forall': 'forall', '∀': 'forall', 'exists': 'exists', '∃': 'exists', 'atom': 'atom', 'literal': 'literal', 'rule': 'rule', 'horn': 'rule', 'eq': 'eq', 'equals': 'eq', '=': 'eq', 'neq': 'neq', '!=': 'neq', '≠': 'neq'}
AUTO_FOL_RESERVED_WORDS = {'forall', 'exists', 'not', 'and', 'or', 'implies', 'iff', 'xor'}
AUTO_FOL_SIMPLE_NAME = re.compile("\\??\\w+(?:[-']\\w+)*")
AUTO_FOL_SYMBOL = re.compile("\\w+(?:[-']\\w+)*(?:\\.\\w+(?:[-']\\w+)*)*")
AUTO_FOL_QUOTED = re.compile('(?:\'(?:[^\'\\\\]|\\\\.)*\'|"(?:[^"\\\\]|\\\\.)*")')
AUTO_FOL_AST_KEYS = {'type', 'op', 'operator', 'predicate', 'pred', 'args', 'arguments', 'negated', 'negative', 'atom', 'name', 'variable', 'variables', 'var', 'vars', 'constant', 'function', 'body', 'operand', 'operands', 'left', 'right', 'if', 'then', 'antecedent', 'antecedents', 'premises', 'consequent', 'conclusion', 'head', 'facts', 'rules', 'clauses', 'literals'}
AUTO_FOL_MAX_DEPTH = 64
AUTO_FOL_MAX_NODES = 10000
AUTO_FOL_MAX_STRING = 200000
AUTO_FOL_MAX_TEXT = 2000000

class AUTO_FOL_Error(ValueError):
    pass

def AUTO_FOL_keys(node, allowed, location):
    unknown = set(node) - set(allowed)
    if unknown:
        raise AUTO_FOL_Error(f'{location}: unsupported keys {sorted(map(str, unknown))}')

def AUTO_FOL_pick(node, keys, location, required=True, convert=None):
    present = [key for key in keys if key in node]
    if not present:
        if required:
            raise AUTO_FOL_Error(f'{location}: missing one of {tuple(keys)}')
        return None
    values = [convert(node[key]) if convert else node[key] for key in present]
    if any((value != values[0] for value in values[1:])):
        raise AUTO_FOL_Error(f'{location}: conflicting fields {present}')
    return values[0]

def AUTO_FOL_fields(fields):
    if fields is None:
        return {}
    if not isinstance(fields, dict):
        raise AUTO_FOL_Error('fields must be a dictionary')
    if set(fields) - AUTO_FOL_FIELD_ROLES:
        raise AUTO_FOL_Error(f'Unsupported field-map roles: {sorted(map(str, set(fields) - AUTO_FOL_FIELD_ROLES))}')
    for role in AUTO_FOL_ALIASES:
        if role in fields:
            key = fields[role]
            if not isinstance(key, str) or not key.strip():
                raise AUTO_FOL_Error(f'fields[{role!r}] must name a column')
            if key.lower() in AUTO_FOL_REFERENCE_FIELDS:
                raise AUTO_FOL_Error('Reference-label columns cannot supply a formal problem field')
    return fields

def AUTO_FOL_present(record, role, fields):
    keys = (fields[role],) if role in fields else AUTO_FOL_ALIASES[role]
    return [(key, record[key]) for key in keys if key in record and record[key] is not None]

def AUTO_FOL_split_lines(text):
    lines, start, quote, escaped = ([], 0, None, False)
    for index, character in enumerate(text):
        if quote is not None:
            if escaped:
                escaped = False
            elif character == '\\':
                escaped = True
            elif character == quote:
                quote = None
        elif character in {"'", '"'}:
            inside_name = character == "'" and index > 0 and (index + 1 < len(text)) and (text[index - 1].isalnum() or text[index - 1] == '_') and (text[index + 1].isalnum() or text[index + 1] == '_')
            if not inside_name:
                quote = character
        elif character in '\r\n':
            value = text[start:index].strip()
            if value:
                lines.append(value)
            start = index + 1
    value = text[start:].strip()
    if value:
        lines.append(value)
    return lines

def AUTO_FOL_json_value(value):
    if isinstance(value, str) and value.lstrip().startswith(('[', '{')):
        if len(value) > AUTO_FOL_MAX_STRING:
            raise AUTO_FOL_Error('Serialized formal input exceeds the text limit')
        try:
            return json.loads(value)
        except (json.JSONDecodeError, RecursionError) as error:
            raise AUTO_FOL_Error(f'Malformed serialized formal input: {error}') from error
    return value

class AUTO_FOL_Serializer:

    def __init__(self):
        self.nodes, self.text_size, self.used, self.counter = (0, 0, set(), 0)

    def touch(self, depth):
        self.nodes += 1
        if depth > AUTO_FOL_MAX_DEPTH or self.nodes > AUTO_FOL_MAX_NODES:
            raise AUTO_FOL_Error('Formal AST exceeds the supported depth/node limits')

    def scan(self, value, depth=0, ancestors=frozenset()):
        self.touch(depth)
        if isinstance(value, str):
            if len(value) > AUTO_FOL_MAX_STRING:
                raise AUTO_FOL_Error('Formal string exceeds its length limit')
            self.text_size += len(value)
            if self.text_size > AUTO_FOL_MAX_TEXT:
                raise AUTO_FOL_Error('Formal input exceeds its total text limit')
            self.used.update((name.lstrip('?') for name in AUTO_FOL_SIMPLE_NAME.findall(value)))
        elif isinstance(value, (dict, list, tuple)):
            if id(value) in ancestors:
                raise AUTO_FOL_Error('Cyclic formal ASTs are unsupported')
            if isinstance(value, dict):
                AUTO_FOL_keys(value, AUTO_FOL_AST_KEYS, 'formal AST')
                children = value.values()
            else:
                children = value
            for child in children:
                self.scan(child, depth + 1, ancestors | {id(value)})
        elif value is not None and (not isinstance(value, (bool, int, float))):
            raise AUTO_FOL_Error(f'Unsupported formal value type {type(value).__name__}')

    def fresh(self):
        while True:
            name = 'AUTO_FOL_bound' + str(self.counter)
            self.counter += 1
            if name not in self.used:
                self.used.add(name)
                return name

    def standalone(self, value, depth=0):
        self.used = set(self.source_names)
        self.counter = 0
        return self.formula(value, depth=depth)

    @staticmethod
    def variable_name(value):
        if not isinstance(value, str) or not AUTO_FOL_SIMPLE_NAME.fullmatch(value):
            raise AUTO_FOL_Error('A variable must be a simple name, optionally prefixed with ?')
        name = value[1:] if value.startswith('?') else value
        if name.isdigit() or name.lower() in AUTO_FOL_RESERVED_WORDS:
            raise AUTO_FOL_Error(f'Invalid variable name {value!r}')
        return name

    @staticmethod
    def symbol(value):
        if not isinstance(value, str) or not (AUTO_FOL_SYMBOL.fullmatch(value) or AUTO_FOL_QUOTED.fullmatch(value)):
            raise AUTO_FOL_Error(f'Unsupported predicate/function name {value!r}')
        if value.split('.', 1)[0].lower() in AUTO_FOL_RESERVED_WORDS:
            raise AUTO_FOL_Error(f'Reserved operator cannot be a predicate/function name: {value!r}')
        return value

    @staticmethod
    def constant(value):
        if isinstance(value, bool) or not isinstance(value, (str, int, float)):
            raise AUTO_FOL_Error('A constant must be a string or finite number')
        if isinstance(value, float) and (not math.isfinite(value)):
            raise AUTO_FOL_Error('Nonfinite constants are unsupported')
        value = str(value)
        if AUTO_FOL_QUOTED.fullmatch(value):
            return value
        if AUTO_FOL_SYMBOL.fullmatch(value) and value.split('.', 1)[0].lower() not in AUTO_FOL_RESERVED_WORDS:
            return value
        return json.dumps(value, ensure_ascii=False)

    def term(self, value, bound, depth):
        self.touch(depth)
        if isinstance(value, str):
            if value.startswith('?'):
                name = self.variable_name(value)
                return '?' + bound.get(name, name)
            if value in bound:
                return '?' + bound[value]
            return self.constant(value)
        if isinstance(value, (int, float)) and (not isinstance(value, bool)):
            return self.constant(value)
        if not isinstance(value, dict):
            raise AUTO_FOL_Error('Unsupported term; use a name or explicit term dictionary')
        kind = AUTO_FOL_pick(value, ('type',), 'term', required=False)
        if kind is not None and (not isinstance(kind, str) or kind not in {'variable', 'constant', 'function'}):
            raise AUTO_FOL_Error(f'Unsupported term type {kind!r}')
        markers = {'variable'} if 'variable' in value or 'var' in value else set()
        markers |= {'constant'} if 'constant' in value else set()
        markers |= {'function'} if 'function' in value else set()
        if len(markers) > 1 or (kind is not None and markers and (markers != {kind})):
            raise AUTO_FOL_Error('Conflicting explicit term types/markers')
        if 'variable' in value or 'var' in value or kind == 'variable':
            AUTO_FOL_keys(value, {'type', 'variable', 'var', 'name'}, 'variable term')
            name = self.variable_name(AUTO_FOL_pick(value, ('variable', 'var', 'name'), 'variable term'))
            return '?' + bound.get(name, name)
        if 'constant' in value or kind == 'constant':
            AUTO_FOL_keys(value, {'type', 'constant', 'name'}, 'constant term')
            return self.constant(AUTO_FOL_pick(value, ('constant', 'name'), 'constant term'))
        if 'function' in value or kind == 'function':
            AUTO_FOL_keys(value, {'type', 'function', 'name', 'args', 'arguments'}, 'function term')
            name = self.symbol(AUTO_FOL_pick(value, ('function', 'name'), 'function term'))
            args = AUTO_FOL_pick(value, ('args', 'arguments'), 'function term', required=False)
            if args is None:
                args = []
            if not isinstance(args, (list, tuple)):
                raise AUTO_FOL_Error('Function arguments must be a list')
            return name + '(' + ', '.join((self.term(a, bound, depth + 1) for a in args)) + ')'
        raise AUTO_FOL_Error('Term dictionaries need variable, constant or function markers')

    @staticmethod
    def operator(value):
        if not isinstance(value, str) or value.lower() not in AUTO_FOL_OPERATORS:
            raise AUTO_FOL_Error(f'Unsupported formal AST operator {value!r}')
        return AUTO_FOL_OPERATORS[value.lower()]

    @staticmethod
    def identity(conjunction):
        name = 'AUTO_FOL_Identity'
        return f'({name} or not {name})' if conjunction else f'({name} and not {name})'

    def combine(self, values, operator, bound, depth):
        if not isinstance(values, (list, tuple)):
            raise AUTO_FOL_Error(f'{operator} operands must be a list')
        if not values:
            return self.identity(operator == 'and')
        rendered = [self.formula(v, bound, depth + 1) for v in values]
        return rendered[0] if len(rendered) == 1 else '(' + (' ' + operator + ' ').join(('(' + text + ')' for text in rendered)) + ')'

    def formula(self, value, bound=None, depth=0):
        self.touch(depth)
        bound = {} if bound is None else bound
        if isinstance(value, str):
            if not value.strip():
                raise AUTO_FOL_Error('A formal formula cannot be empty')
            if bound:
                raise AUTO_FOL_Error('Within an AST quantifier use structured formulas, not raw formula-string templates')
            return value.strip()
        if not isinstance(value, dict):
            raise AUTO_FOL_Error('A formula must be a formal string or explicit AST dictionary')
        op = AUTO_FOL_pick(value, ('op', 'operator', 'type'), 'formula node', required=False, convert=self.operator)
        if 'predicate' in value or 'pred' in value or 'atom' in value or (op in {'atom', 'literal'}):
            if op not in {None, 'atom', 'literal'}:
                raise AUTO_FOL_Error('A predicate/atom marker conflicts with its explicit operator')
            AUTO_FOL_keys(value, {'op', 'operator', 'type', 'predicate', 'pred', 'name', 'args', 'arguments', 'negated', 'negative', 'atom'}, 'atom')
            negative = AUTO_FOL_pick(value, ('negated', 'negative'), 'atom polarity', required=False)
            if negative is None:
                negative = False
            if type(negative) is not bool:
                raise AUTO_FOL_Error('Atom negated must be a Boolean')
            if 'atom' in value:
                if any((k in value for k in ('predicate', 'pred', 'name', 'args', 'arguments'))):
                    raise AUTO_FOL_Error('A wrapped atom cannot also contain an independent predicate')
                text = self.literal(value['atom'], bound, depth + 1)
            else:
                raw_name = AUTO_FOL_pick(value, ('predicate', 'pred', 'name'), 'atom predicate')
                args = AUTO_FOL_pick(value, ('args', 'arguments'), 'atom arguments', required=False)
                if args is None:
                    args = []
                if not isinstance(args, (list, tuple)):
                    raise AUTO_FOL_Error('Atom arguments must be a list')
                if isinstance(raw_name, str) and raw_name in {'=', '!=', '≠'}:
                    if len(args) != 2:
                        raise AUTO_FOL_Error('Equality atoms require two arguments')
                    text = '(' + self.term(args[0], bound, depth + 1) + ' = ' + self.term(args[1], bound, depth + 1) + ')'
                    negative = negative != (raw_name != '=')
                else:
                    name = self.symbol(raw_name)
                    text = name + ('(' + ', '.join((self.term(a, bound, depth + 1) for a in args)) + ')' if args else '')
            return f'not ({text})' if negative else text
        rule_keys = {'if', 'then', 'antecedent', 'antecedents', 'consequent', 'head'}
        if op == 'rule' or set(value) & rule_keys or {'premises', 'conclusion'} <= set(value):
            if op not in {None, 'rule'}:
                raise AUTO_FOL_Error('A rule schema conflicts with its explicit operator')
            return self.rule(value, bound, depth + 1)
        if 'clauses' in value:
            AUTO_FOL_keys(value, {'clauses'}, 'CNF formula')
            clauses = self.cnf(value['clauses'], bound, depth + 1)
            return '(' + ' and '.join(clauses) + ')' if clauses else self.identity(True)
        if op in {'eq', 'neq'}:
            AUTO_FOL_keys(value, {'op', 'operator', 'type', 'args', 'arguments', 'left', 'right'}, 'equality')
            operands = self.operands(value, depth, binary=True)
            return '(' + self.term(operands[0], bound, depth + 1) + (' != ' if op == 'neq' else ' = ') + self.term(operands[1], bound, depth + 1) + ')'
        if op in {'forall', 'exists'}:
            AUTO_FOL_keys(value, {'op', 'operator', 'type', 'variable', 'variables', 'var', 'vars', 'body'}, 'quantifier')
            declarations = AUTO_FOL_pick(value, ('variables', 'vars', 'variable', 'var'), 'quantifier variables', convert=lambda x: list(x) if isinstance(x, (list, tuple)) else [x])
            if not declarations:
                raise AUTO_FOL_Error('A quantifier needs at least one variable')
            names = [self.variable_name(v) for v in declarations]
            if len(set(names)) != len(names):
                raise AUTO_FOL_Error('Duplicate variables in one quantifier declaration')
            new_bound, emitted = (dict(bound), [])
            for name in names:
                fresh = self.fresh()
                new_bound[name] = fresh
                emitted.append(fresh)
            body = self.formula(AUTO_FOL_pick(value, ('body',), 'quantifier body'), new_bound, depth + 1)
            return f"({op} {', '.join(emitted)}. ({body}))"
        if op == 'not':
            AUTO_FOL_keys(value, {'op', 'operator', 'type', 'body', 'operand', 'args', 'arguments', 'operands'}, 'negation')
            direct = AUTO_FOL_pick(value, ('body', 'operand'), 'negation', required=False)
            lists = AUTO_FOL_pick(value, ('args', 'arguments', 'operands'), 'negation operands', required=False)
            if lists is not None:
                if not isinstance(lists, (list, tuple)) or len(lists) != 1:
                    raise AUTO_FOL_Error('not needs exactly one operand')
                if direct is not None and direct != lists[0]:
                    raise AUTO_FOL_Error('Conflicting negation operands')
                direct = lists[0]
            if direct is None:
                raise AUTO_FOL_Error('not needs an operand')
            return 'not (' + self.formula(direct, bound, depth + 1) + ')'
        if op in {'and', 'or', 'implies', 'iff', 'xor'}:
            AUTO_FOL_keys(value, {'op', 'operator', 'type', 'args', 'arguments', 'operands', 'left', 'right'}, 'Boolean node')
            values = self.operands(value, depth, binary=op in {'implies', 'iff', 'xor'})
            return self.combine(values, op, bound, depth + 1)
        raise AUTO_FOL_Error('Unrecognized formal AST; an explicit supported operator or predicate is required')

    def operands(self, value, depth, binary=False):
        listed = AUTO_FOL_pick(value, ('args', 'arguments', 'operands'), 'operands', required=False)
        if listed is not None and (not isinstance(listed, (list, tuple))):
            raise AUTO_FOL_Error('Operands must be an explicit list')
        paired = None
        if 'left' in value or 'right' in value:
            if 'left' not in value or 'right' not in value:
                raise AUTO_FOL_Error('Both left and right are required')
            paired = [value['left'], value['right']]
        if listed is not None and paired is not None and (list(listed) != paired):
            raise AUTO_FOL_Error('Conflicting list and left/right operands')
        values = listed if listed is not None else paired
        if not isinstance(values, (list, tuple)):
            raise AUTO_FOL_Error('Explicit operand list or left/right pair required')
        if binary and len(values) != 2:
            raise AUTO_FOL_Error('This operator needs exactly two operands')
        return values

    def rule(self, value, bound, depth):
        AUTO_FOL_keys(value, {'op', 'operator', 'type', 'if', 'then', 'antecedent', 'antecedents', 'body', 'premises', 'consequent', 'conclusion', 'head'}, 'Horn rule')
        body = AUTO_FOL_pick(value, ('if', 'antecedent', 'antecedents', 'body', 'premises'), 'rule body')
        head = AUTO_FOL_pick(value, ('then', 'consequent', 'conclusion', 'head'), 'rule head')
        conclusion = self.formula(head, bound, depth + 1)
        if isinstance(body, (list, tuple)):
            if not body:
                return conclusion
            premise = self.combine(body, 'and', bound, depth + 1)
        else:
            premise = self.formula(body, bound, depth + 1)
        return f'(({premise}) -> ({conclusion}))'

    def literal(self, value, bound, depth):
        if isinstance(value, dict):
            op = AUTO_FOL_pick(value, ('op', 'operator', 'type'), 'literal', required=False, convert=self.operator)
            if op not in {None, 'atom', 'literal', 'not', 'eq', 'neq'}:
                raise AUTO_FOL_Error('CNF clauses may contain literals only')
        text = self.formula(value, bound, depth + 1)
        stripped = re.sub('(?:\'(?:[^\'\\\\]|\\\\.)*\'|"(?:[^"\\\\]|\\\\.)*")', '', text)
        if re.search('[∀∃∧∨⊕→↔⇒⇔⟷&|]|<->|->|=>|\\b(?:forall|exists|and|or|implies|iff|xor)\\b', stripped, re.I):
            raise AUTO_FOL_Error('CNF clauses may contain literals only')
        return text

    def cnf(self, clauses, bound=None, depth=0):
        self.touch(depth)
        clauses = AUTO_FOL_json_value(clauses)
        bound = {} if bound is None else bound
        if not isinstance(clauses, (list, tuple)):
            raise AUTO_FOL_Error('clauses must be a list of literal lists')
        result = []
        for clause in clauses:
            if isinstance(clause, dict):
                AUTO_FOL_keys(clause, {'literals'}, 'CNF clause')
                clause = AUTO_FOL_pick(clause, ('literals',), 'CNF clause')
            if not isinstance(clause, (list, tuple)):
                raise AUTO_FOL_Error('Every CNF clause must be an explicit literal list')
            literals = [self.literal(v, bound, depth + 1) for v in clause]
            result.append('(' + ' or '.join(literals) + ')' if literals else self.identity(False))
        return result

    def premises(self, value, depth=0):
        value = AUTO_FOL_json_value(value)
        if isinstance(value, str):
            return AUTO_FOL_split_lines(value)
        if isinstance(value, dict):
            return [self.standalone(value, depth=depth + 1)]
        if not isinstance(value, (list, tuple)):
            raise AUTO_FOL_Error('Premises must be a formula list, AST, or formula-per-line string')
        return [self.standalone(v, depth=depth + 1) for v in value if not isinstance(v, str) or v.strip()]

def AUTO_FOL_problem_inputs(record, fields):
    if not isinstance(record, dict):
        raise AUTO_FOL_Error('A formal problem must be a dictionary')
    query = AUTO_FOL_present(record, 'query', fields)
    if not query:
        raise AUTO_FOL_Error('Missing formal query')
    if 'premises' in fields:
        providers = [('premises', value) for _, value in AUTO_FOL_present(record, 'premises', fields)]
        if not providers:
            raise AUTO_FOL_Error('The configured premises column is missing')
    else:
        providers = [('premises', value) for _, value in AUTO_FOL_present(record, 'premises', fields)]
        providers += [('kb', value) for _, value in AUTO_FOL_present(record, 'kb', fields)]
        components = {}
        for role in ('facts', 'rules', 'clauses'):
            present = AUTO_FOL_present(record, role, fields)
            if present:
                components[role] = AUTO_FOL_pick(dict(present), tuple((k for k, _ in present)), role)
        if components:
            providers.append(('kb', components))
    if not providers:
        raise AUTO_FOL_Error('Missing explicit premises, facts/rules, clauses, or knowledge base')
    return (providers, query)

def normalize_formal_problem(record, fields=None):
    fields = AUTO_FOL_fields(fields)
    providers, queries = AUTO_FOL_problem_inputs(record, fields)
    serializer = AUTO_FOL_Serializer()

    def decode_provider(role, value):
        value = AUTO_FOL_json_value(value)
        if role == 'kb' and isinstance(value, dict):
            AUTO_FOL_keys(value, {'premises', 'facts', 'rules', 'clauses'}, 'knowledge-base container')
            value = {key: AUTO_FOL_json_value(item) for key, item in value.items()}
        return value
    decoded_providers = [(role, decode_provider(role, value)) for role, value in providers]
    decoded_queries = [(key, AUTO_FOL_json_value(value)) for key, value in queries]
    for _, value in decoded_providers + decoded_queries:
        serializer.scan(value)
    serializer.source_names = frozenset(serializer.used)
    serializer.nodes = 0

    def knowledge_base(value):
        if not isinstance(value, dict):
            return serializer.premises(value)
        AUTO_FOL_keys(value, {'premises', 'facts', 'rules', 'clauses'}, 'knowledge-base container')
        candidates = []
        if 'premises' in value:
            candidates.append(serializer.premises(value['premises']))
        if any((role in value for role in ('facts', 'rules', 'clauses'))):
            combined = []
            for role in ('facts', 'rules'):
                if role in value:
                    combined += serializer.premises(value[role])
            if 'clauses' in value:
                combined += serializer.cnf(value['clauses'])
            candidates.append(combined)
        if not candidates:
            return []
        if any((v != candidates[0] for v in candidates[1:])):
            raise AUTO_FOL_Error('Conflicting KB premises and component fields')
        return candidates[0]
    premise_options = [knowledge_base(value) if role == 'kb' else serializer.premises(value) for role, value in decoded_providers]
    if any((value != premise_options[0] for value in premise_options[1:])):
        raise AUTO_FOL_Error('Conflicting premise aliases/containers; select the intended column in fields')
    query_options = [serializer.standalone(value) for _, value in decoded_queries]
    if any((value != query_options[0] for value in query_options[1:])):
        raise AUTO_FOL_Error('Conflicting query aliases; select the intended column in fields')
    result = {'premises': premise_options[0], 'query': query_options[0]}
    if sum(map(len, result['premises'])) + len(result['query']) > AUTO_FOL_MAX_TEXT:
        raise AUTO_FOL_Error('Normalized formulas exceed the total text limit')
    return result

def AUTO_FOL_looks_formal(value, depth=0):
    if depth > AUTO_FOL_MAX_DEPTH:
        return False
    if isinstance(value, dict):
        return bool(set(value) & {'predicate', 'pred', 'op', 'operator', 'facts', 'rules', 'clauses', 'premises', 'if', 'then'} or value.get('type') in AUTO_FOL_OPERATORS)
    if isinstance(value, (list, tuple)):
        return len(value) <= AUTO_FOL_MAX_NODES and all((AUTO_FOL_looks_formal(v, depth + 1) for v in value))
    if not isinstance(value, str) or not value.strip() or len(value) > AUTO_FOL_MAX_STRING:
        return False
    text = value.strip()
    if text.startswith(('[', '{')):
        return True
    if AUTO_FOL_SYMBOL.fullmatch(text) or AUTO_FOL_QUOTED.fullmatch(text):
        return True
    lines = AUTO_FOL_split_lines(text)
    if len(lines) > 1:
        return len(lines) <= AUTO_FOL_MAX_NODES and all((AUTO_FOL_looks_formal(line, depth + 1) for line in lines))
    if re.search('[()∀∃¬∧∨⊕→↔&|=]|->|<->|\\?\\w|\\b(?:forall|exists)\\s+\\??\\w', text, re.I):
        return True
    atom = "\\w+(?:[-']\\w+)*(?:\\.\\w+(?:[-']\\w+)*)*"
    if re.fullmatch(f'(?:not\\s+)?{atom}(?:\\s+(?:and|or|implies|iff|xor)\\s+(?:not\\s+)?{atom})*\\.?', text, re.I):
        return True
    return False

def is_formal_record(record, fields=None):
    try:
        fields = AUTO_FOL_fields(fields)
        providers, queries = AUTO_FOL_problem_inputs(record, fields)
        explicit = 'premises' in fields or 'query' in fields or any((key in record for key in ('premises-FOL', 'premises_fol', 'conclusion-FOL', 'conclusion_fol', 'clauses', 'knowledge_base', 'kb')))
        if explicit:
            return True

        def provider_eligible(role, value):
            if role == 'kb' and isinstance(value, dict):
                return all((AUTO_FOL_looks_formal(part) for part in value.values()))
            return AUTO_FOL_looks_formal(value)
        return any((provider_eligible(role, value) for role, value in providers)) and any((AUTO_FOL_looks_formal(value) for _, value in queries))
    except (AUTO_FOL_Error, TypeError, ValueError, RecursionError):
        return False

class AUTO_CB_Error(ValueError):

    def __init__(self, code, message, **details):
        super().__init__(message)
        self.code, self.details = (code, details)
AUTO_CB_ALIASES = {'equations': ('equations', 'structural_equations'), 'query': ('query', 'target', 'statement', 'query_expression'), 'interventions': ('interventions', 'intervention', 'do'), 'observations': ('observations', 'evidence'), 'factual': ('factual', 'factual_evidence'), 'counterfactual': ('counterfactual', 'counterfactual_evidence', 'counterfactual_observations'), 'priors': ('priors', 'root_priors', 'marginals', 'bernoulli'), 'roots': ('roots', 'exogenous', 'exogenous_variables'), 'timing': ('observation_mode', 'observation_timing', 'evidence_timing')}
AUTO_CB_OPS = {'not', 'and', 'or', 'xor', 'iff', 'implies', 'var', 'atom', 'const', 'constant'}
AUTO_CB_TOKEN = re.compile('\\s*(<->|<=>|->|=>|&&|\\|\\||[¬∧∨⊕→⇒↔⇔!~&|^(),=]|"(?:[^"\\\\]|\\\\.)*"|\'(?:[^\'\\\\]|\\\\.)*\'|\\w+(?:[.\'-]\\w+)*)')
AUTO_CB_SYMBOL_OPS = {'~': 'not', '!': 'not', '¬': 'not', '&': 'and', '&&': 'and', '∧': 'and', '|': 'or', '||': 'or', '∨': 'or', '^': 'xor', '⊕': 'xor', '->': 'implies', '=>': 'implies', '→': 'implies', '⇒': 'implies', '<->': 'iff', '<=>': 'iff', '↔': 'iff', '⇔': 'iff', '=': 'iff'}

def AUTO_CB_check(context):
    if time.perf_counter() >= context['deadline']:
        raise AUTO_CB_Error('normalization_timeout', 'Structured causal input processing exceeded its time budget.')
    context['nodes'] += 1
    if context['nodes'] > 20000:
        raise AUTO_CB_Error('input_limit', 'Structured causal input exceeds the bounded node limit.')

def AUTO_CB_solve_check(deadline):
    if time.perf_counter() >= deadline:
        raise AUTO_CB_Error('reasoning_timeout', 'Structured causal reasoning time limit reached.')

def AUTO_CB_name(value):
    if not isinstance(value, str) or not value or value != value.strip() or (len(value) > 128):
        raise AUTO_CB_Error('invalid_variable_name', 'Variable labels must be nonempty strings, at most 128 characters, without outer whitespace.')
    if any((ord(character) < 32 for character in value)):
        raise AUTO_CB_Error('invalid_variable_name', 'Variable labels cannot contain control characters.')
    return value

def AUTO_CB_boolean(value):
    if isinstance(value, bool):
        return value
    if isinstance(value, int) and value in (0, 1):
        return bool(value)
    if isinstance(value, str) and value.strip().lower() in {'true', 'false', '0', '1'}:
        return value.strip().lower() in {'true', '1'}
    raise AUTO_CB_Error('invalid_boolean_assignment', 'A Boolean assignment must be true/false or 0/1.')

def AUTO_CB_decode(value):
    if isinstance(value, str) and value.lstrip().startswith(('{', '[')):
        if len(value) > 200000:
            raise AUTO_CB_Error('input_limit', 'JSON-encoded causal field exceeds 200,000 characters.')
        try:
            return json.loads(value)
        except (json.JSONDecodeError, RecursionError) as error:
            raise AUTO_CB_Error('invalid_json_causal_field', 'A causal field beginning with {/[ must contain valid JSON: ' + str(error)) from error
    return value

def AUTO_CB_pick(record, names, default=None, required=False):
    present = [name for name in names if name in record and record[name] is not None]
    if not present:
        if required:
            raise AUTO_CB_Error('missing_causal_field', 'Missing structured causal field: ' + '/'.join(names))
        return (default, False)
    value = record[present[0]]
    if any((record[name] != value for name in present[1:])):
        raise AUTO_CB_Error('conflicting_causal_aliases', 'Conflicting aliases: ' + ', '.join(present))
    return (value, True)

def AUTO_CB_expand(operator, arguments):
    if operator == 'not' and len(arguments) == 1:
        return ('not', arguments[0])
    if operator in {'and', 'or'}:
        return (operator, tuple(arguments))
    if operator not in {'xor', 'iff', 'implies'} or len(arguments) != 2:
        raise AUTO_CB_Error('unsupported_boolean_operator', 'Unsupported Boolean operator or arity: ' + str(operator))
    left, right = arguments
    if operator == 'implies':
        return ('or', (('not', left), right))
    if operator == 'iff':
        return ('or', (('and', (left, right)), ('and', (('not', left), ('not', right)))))
    return ('or', (('and', (left, ('not', right))), ('and', (('not', left), right))))

def AUTO_CB_string_expression(text, context):
    if not text.strip() or len(text) > 20000:
        raise AUTO_CB_Error('invalid_boolean_expression', 'Boolean expression text must be nonempty and at most 20,000 characters.')
    tokens, position = ([], 0)
    text = text.strip()
    while position < len(text):
        AUTO_CB_check(context)
        match = AUTO_CB_TOKEN.match(text, position)
        if not match:
            raise AUTO_CB_Error('unsupported_boolean_syntax', 'Unsupported Boolean syntax near: ' + repr(text[position:position + 30]))
        tokens.append(match[1])
        position = match.end()
    index = 0

    def peek():
        if index >= len(tokens):
            return None
        raw = tokens[index]
        return AUTO_CB_SYMBOL_OPS.get(raw, raw.lower() if raw.lower() in AUTO_CB_OPS else raw)

    def take(expected=None):
        nonlocal index
        AUTO_CB_check(context)
        token = peek()
        if token is None or (expected is not None and token != expected):
            raise AUTO_CB_Error('unsupported_boolean_syntax', 'Expected ' + str(expected or 'a Boolean token') + ', found ' + repr(token))
        raw = tokens[index]
        index += 1
        return raw

    def unary(depth):
        AUTO_CB_check(context)
        if depth > 64:
            raise AUTO_CB_Error('input_limit', 'Boolean expression nesting exceeds 64 levels.')
        token = peek()
        if token == 'not':
            take()
            return ('not', unary(depth + 1))
        if token == '(':
            take()
            value = expression(0, depth + 1)
            take(')')
            return value
        raw = take()
        if raw.startswith('"'):
            return ('var', AUTO_CB_name(json.loads(raw)))
        if raw.startswith("'"):
            inner = raw[1:-1]
            if re.search("\\\\(?!['\\\\])", inner):
                raise AUTO_CB_Error('unsupported_boolean_syntax', 'Single-quoted variable labels support only escaped quote/backslash.')
            return ('var', AUTO_CB_name(re.sub("\\\\(['\\\\])", '\\1', inner)))
        if raw.lower() in {'true', 'false', '0', '1'}:
            return ('and', ()) if AUTO_CB_boolean(raw) else ('or', ())
        operator = AUTO_CB_SYMBOL_OPS.get(raw, raw.lower())
        if operator in {'and', 'or', 'xor', 'iff', 'implies'} and peek() == '(':
            take('(')
            arguments = []
            if peek() != ')':
                arguments.append(expression(0, depth + 1))
                while peek() == ',':
                    take(',')
                    arguments.append(expression(0, depth + 1))
            take(')')
            return AUTO_CB_expand(operator, arguments)
        if operator in AUTO_CB_OPS or raw in {',', ')'}:
            raise AUTO_CB_Error('unsupported_boolean_syntax', 'Unexpected Boolean token: ' + raw)
        return ('var', AUTO_CB_name(raw))

    def expression(level=0, depth=0):
        precedence = (('iff',), ('implies',), ('or', 'xor'), ('and',))
        if level == len(precedence):
            return unary(depth)
        left = expression(level + 1, depth)
        while peek() in precedence[level]:
            operator = AUTO_CB_SYMBOL_OPS.get(tokens[index], tokens[index].lower())
            take()
            right = expression(level if operator == 'implies' else level + 1, depth + 1)
            left = AUTO_CB_expand(operator, [left, right])
        return left
    result = expression()
    if index != len(tokens):
        raise AUTO_CB_Error('unsupported_boolean_syntax', 'Unexpected trailing Boolean token: ' + tokens[index])
    return result

def AUTO_CB_expression(value, context, depth=0):
    AUTO_CB_check(context)
    value = AUTO_CB_decode(value)
    if depth > 64:
        raise AUTO_CB_Error('input_limit', 'Structured Boolean expression nesting exceeds 64 levels.')
    if isinstance(value, bool) or (isinstance(value, int) and value in (0, 1)):
        return ('and', ()) if AUTO_CB_boolean(value) else ('or', ())
    if isinstance(value, str):
        return AUTO_CB_string_expression(value, context)
    if isinstance(value, dict):
        var, has_var = AUTO_CB_pick(value, ('var', 'atom', 'variable'))
        if has_var and (not set(value) & {'op', 'operator', 'type'}):
            if set(value) - {'var', 'atom', 'variable', 'negated', 'negative', 'args', 'arguments'}:
                raise AUTO_CB_Error('unsupported_boolean_ast', 'Unexpected fields in Boolean variable node.')
            arguments, _ = AUTO_CB_pick(value, ('args', 'arguments'), ())
            if AUTO_CB_decode(arguments) not in ([], ()):
                raise AUTO_CB_Error('unsupported_boolean_ast', 'Boolean variable nodes cannot have arguments.')
            negative, _ = AUTO_CB_pick(value, ('negated', 'negative'), False)
            expression = ('var', AUTO_CB_name(var))
            return ('not', expression) if AUTO_CB_boolean(negative) else expression
        constant, has_constant = AUTO_CB_pick(value, ('const', 'constant'))
        if has_constant:
            if set(value) - {'const', 'constant'}:
                raise AUTO_CB_Error('unsupported_boolean_ast', 'Unexpected fields in Boolean constant node.')
            return AUTO_CB_expression(AUTO_CB_boolean(constant), context, depth + 1)
        operator, explicit = AUTO_CB_pick(value, ('op', 'operator', 'type'))
        shorthand = not explicit and len(value) == 1
        if shorthand:
            operator, arguments = next(iter(value.items()))
        elif explicit:
            operator = str(operator).lower()
            if operator in {'var', 'atom'}:
                name, _ = AUTO_CB_pick(value, ('name', 'variable', 'symbol', 'predicate'), required=True)
                negative, _ = AUTO_CB_pick(value, ('negative', 'negated'), False)
                if set(value) - {'op', 'operator', 'type', 'name', 'variable', 'symbol', 'predicate', 'negative', 'negated', 'args', 'arguments'}:
                    raise AUTO_CB_Error('unsupported_boolean_ast', 'A Boolean atom cannot carry arguments or unknown fields.')
                arguments, _ = AUTO_CB_pick(value, ('args', 'arguments'), ())
                if AUTO_CB_decode(arguments) not in ([], ()):
                    raise AUTO_CB_Error('unsupported_boolean_ast', 'Only zero-arity atoms are Boolean variables; relational atoms need an explicit adapter.')
                node = ('var', AUTO_CB_name(name))
                return ('not', node) if AUTO_CB_boolean(negative) else node
            if operator in {'const', 'constant'}:
                if set(value) - {'op', 'operator', 'type', 'value'}:
                    raise AUTO_CB_Error('unsupported_boolean_ast', 'Unexpected fields in Boolean constant node.')
                return AUTO_CB_expression(AUTO_CB_boolean(value.get('value')), context, depth + 1)
            arguments, has_args = AUTO_CB_pick(value, ('args', 'operands', 'children'))
            has_unary = bool(set(value) & {'arg', 'operand', 'body'})
            has_binary = bool(set(value) & {'left', 'right'})
            if sum((has_args, has_unary, has_binary)) > 1:
                raise AUTO_CB_Error('ambiguous_boolean_operands', 'Use exactly one Boolean operand encoding; args, body and left/right cannot coexist.')
            if not has_args and operator == 'not':
                argument, _ = AUTO_CB_pick(value, ('arg', 'operand', 'body'), required=True)
                arguments = [argument]
            elif not has_args and 'left' in value and ('right' in value):
                arguments = [value['left'], value['right']]
            if set(value) - {'op', 'operator', 'type', 'args', 'operands', 'children', 'arg', 'operand', 'body', 'left', 'right'}:
                raise AUTO_CB_Error('unsupported_boolean_ast', 'Unknown Boolean operator fields were not discarded.')
        else:
            raise AUTO_CB_Error('unsupported_boolean_ast', 'Boolean AST must identify its operator explicitly.')
        operator = AUTO_CB_SYMBOL_OPS.get(str(operator), str(operator).lower())
        arguments = AUTO_CB_decode(arguments)
        if operator == 'not' and (shorthand or not isinstance(arguments, (list, tuple))):
            arguments = [arguments]
        if not isinstance(arguments, (list, tuple)):
            raise AUTO_CB_Error('unsupported_boolean_ast', 'Boolean operator operands must be a list.')
        return AUTO_CB_expand(operator, [AUTO_CB_expression(argument, context, depth + 1) for argument in arguments])
    if isinstance(value, (list, tuple)) and value and isinstance(value[0], str):
        operator = AUTO_CB_SYMBOL_OPS.get(value[0], value[0].lower())
        if operator in {'var', 'atom'} and len(value) == 2:
            return ('var', AUTO_CB_name(value[1]))
        if operator in {'const', 'constant'} and len(value) == 2:
            return AUTO_CB_expression(AUTO_CB_boolean(value[1]), context, depth + 1)
        arguments = list(value[1:])
        if operator in {'and', 'or', 'xor', 'iff', 'implies'} and len(arguments) == 1 and isinstance(arguments[0], (list, tuple)):
            body = arguments[0]
            is_node = bool(body) and isinstance(body[0], str) and (body[0].lower() in AUTO_CB_OPS)
            if not is_node:
                arguments = list(body)
        return AUTO_CB_expand(operator, [AUTO_CB_expression(argument, context, depth + 1) for argument in arguments])
    raise AUTO_CB_Error('unsupported_boolean_ast', 'Unsupported Boolean expression representation.')

def AUTO_CB_symbols(expression, context):
    pending, symbols, count = ([expression], set(), 0)
    while pending:
        AUTO_CB_check(context)
        node = pending.pop()
        count += 1
        if count > 4096:
            raise AUTO_CB_Error('input_limit', 'Expanded Boolean expression exceeds 4,096 nodes.')
        if node[0] == 'var':
            symbols.add(node[1])
        elif node[0] == 'not':
            pending.append(node[1])
        else:
            pending.extend(node[1])
    return symbols

def AUTO_CB_merge(destination, additions, role):
    for name, value in additions.items():
        if name in destination and destination[name] != value:
            raise AUTO_CB_Error('conflicting_' + role, 'Conflicting ' + role + ' assignments for ' + name)
        destination[name] = value

def AUTO_CB_assignments(value, context, role):
    AUTO_CB_check(context)
    value = AUTO_CB_decode(value)
    if value is None:
        return {}
    if isinstance(value, dict):
        name, explicit = AUTO_CB_pick(value, ('variable', 'var', 'name'))
        if explicit and any((key in value for key in ('value', 'state'))):
            state, _ = AUTO_CB_pick(value, ('value', 'state'), required=True)
            if set(value) - {'variable', 'var', 'name', 'value', 'state'}:
                raise AUTO_CB_Error('unsupported_assignment', 'Unknown assignment fields were not discarded.')
            return {AUTO_CB_name(name): AUTO_CB_boolean(state)}
        is_ast = bool(set(value) & {'var', 'atom', 'variable', 'not', 'and', 'op', 'operator', 'type'})
        if not is_ast:
            return {AUTO_CB_name(name): AUTO_CB_boolean(state) for name, state in value.items()}
    if isinstance(value, (list, tuple)):
        if len(value) == 2 and isinstance(value[0], str) and (value[0].lower() not in AUTO_CB_OPS):
            try:
                state = AUTO_CB_boolean(value[1])
            except AUTO_CB_Error:
                pass
            else:
                return {AUTO_CB_name(value[0]): state}
        is_ast = bool(value) and isinstance(value[0], str) and (value[0].lower() in AUTO_CB_OPS)
        if not is_ast:
            merged = {}
            for item in value:
                AUTO_CB_merge(merged, AUTO_CB_assignments(item, context, role), role)
            return merged
    expression = AUTO_CB_expression(value, context)
    result, pending = ({}, [expression])
    while pending:
        AUTO_CB_check(context)
        node = pending.pop()
        if node[0] == 'and':
            pending.extend(node[1])
        elif node[0] == 'var':
            AUTO_CB_merge(result, {node[1]: True}, role)
        elif node[0] == 'not' and node[1][0] == 'var':
            AUTO_CB_merge(result, {node[1][1]: False}, role)
        else:
            raise AUTO_CB_Error('unsupported_assignment', role + ' must be variable assignments or a conjunction of literals.')
    return result

def AUTO_CB_prior(value):
    value = AUTO_CB_decode(value)
    if isinstance(value, dict):
        family, has_family = AUTO_CB_pick(value, ('type', 'family'), 'bernoulli')
        if has_family and str(family).lower() not in {'bern', 'bernoulli'}:
            raise AUTO_CB_Error('unsupported_prior_distribution', 'Only Bernoulli marginals are supported.')
        probability, present = AUTO_CB_pick(value, ('p', 'probability', 'bernoulli', 'Bernoulli', 'Bern'))
        if not present and set(value) == {'distribution'}:
            return AUTO_CB_prior(value['distribution'])
        if not present or set(value) - {'type', 'family', 'p', 'probability', 'bernoulli', 'Bernoulli', 'Bern'}:
            raise AUTO_CB_Error('unsupported_prior_distribution', 'Use a scalar marginal or an explicit Bernoulli p value.')
        value = probability
    if isinstance(value, bool) or not isinstance(value, (str, int, float, Fraction)):
        raise AUTO_CB_Error('invalid_prior', 'Bernoulli probabilities must be numeric marginals, not Boolean assignments.')
    text = str(value).strip()
    match = re.fullmatch('Bern(?:oulli)?\\(\\s*(.*?)\\s*\\)', text, re.I)
    if match:
        text = match[1]
    if len(text) > 128:
        raise AUTO_CB_Error('input_limit', 'Probability text is too long.')
    exponent = re.search('[eE]([+-]?\\d+)$', text)
    if exponent and abs(int(exponent[1])) > 1000:
        raise AUTO_CB_Error('input_limit', 'Probability exponent exceeds the supported bound.')
    try:
        probability = Fraction(text)
    except (ValueError, ZeroDivisionError, OverflowError):
        raise AUTO_CB_Error('invalid_prior', 'Invalid exact Bernoulli marginal: ' + text)
    if not 0 <= probability <= 1:
        raise AUTO_CB_Error('invalid_prior', 'Bernoulli marginals must be between zero and one.')
    return probability

def is_structured_causal_record(record):
    return isinstance(record, dict) and (any((key in record for key in AUTO_CB_ALIASES['equations'])) or 'causal_model' in record)

def AUTO_CB_normalize(record, fields, context):
    AUTO_CB_check(context)
    if not isinstance(record, dict):
        raise AUTO_CB_Error('invalid_causal_record', 'A structured causal record must be a dictionary.')
    if len(record) > 256:
        raise AUTO_CB_Error('input_limit', 'Too many fields in a structured causal record.')
    selected = dict(record)
    if 'causal_model' in selected:
        model = AUTO_CB_decode(selected.pop('causal_model'))
        if not isinstance(model, dict):
            raise AUTO_CB_Error('invalid_causal_model', 'causal_model must be a dictionary.')
        if len(model) > 128:
            raise AUTO_CB_Error('input_limit', 'Too many fields in causal_model.')
        for key, value in model.items():
            if key in selected and selected[key] != value:
                raise AUTO_CB_Error('conflicting_causal_envelope', 'Conflicting outer/model field: ' + str(key))
            selected[key] = value
    for key in ('facts', 'rules', 'premises', 'context', 'constraints', 'assumptions', 'knowledge_base', 'kb', 'clauses', 'premises-FOL', 'premises_fol', 'conditions', 'given_info'):
        if key in selected:
            raise AUTO_CB_Error('mixed_causal_logical_schema', "Additional semantic field '" + key + "' cannot be silently discarded beside structural equations.")
    for key in ('noise', 'noise_model', 'noise_terms', 'joint_distribution', 'root_joint_distribution', 'exogenous_distribution', 'root_distribution', 'joint_prior', 'equilibrium_selection'):
        if key in selected and selected[key] not in (None, {}, []):
            raise AUTO_CB_Error('unsupported_causal_model', 'Unsupported causal model field: ' + key)
    kind, has_kind = AUTO_CB_pick(selected, ('model_type', 'model_kind'))
    if has_kind and str(kind).lower() not in {'causal', 'scm', 'boolean', 'boolean_scm', 'deterministic_boolean', 'deterministic_scm', 'structural_causal'}:
        raise AUTO_CB_Error('unsupported_causal_model', 'Only deterministic Boolean structural models are supported.')
    semantics, has_semantics = AUTO_CB_pick(selected, ('semantics', 'causal_semantics'))
    if has_semantics and str(semantics).lower() not in {'classical', 'causal', 'scm', 'structural', 'deterministic_boolean', 'boolean_scm'}:
        raise AUTO_CB_Error('unsupported_causal_semantics', 'The supplied causal semantics are not supported by the deterministic Boolean model.')

    def field(name, default=None, required=False):
        names = (fields[name],) if fields and name in fields else AUTO_CB_ALIASES[name]
        if not all((isinstance(item, str) for item in names)):
            raise AUTO_CB_Error('invalid_field_mapping', 'Causal field mappings must name string keys.')
        reference_fields = {'gold', 'gold_label', 'label', 'labels', 'answer', 'answers', 'flag', 'prediction', 'predictions', 'predicted_label', 'predicted_labels', 'predicted_flag', 'reasoning', 'explanation', 'explanations', 'expected', 'expected_answer', 'expected_label', 'ground_truth', 'reference_answer'}
        if any((key.casefold() in reference_fields for key in names)):
            raise AUTO_CB_Error('reference_field_not_input', 'Reference labels and supplied explanations cannot be mapped to causal problem fields.')
        present = [key for key in names if key in selected and selected[key] is not None]
        if not present:
            if required:
                raise AUTO_CB_Error('missing_causal_field', 'Missing structured causal field: ' + '/'.join(names))
            return (default, False)
        values = [AUTO_CB_decode(selected[key]) for key in present]
        if any((value != values[0] for value in values[1:])):
            raise AUTO_CB_Error('conflicting_causal_aliases', 'Conflicting aliases: ' + ', '.join(present))
        return (values[0], True)
    raw_equations, _ = field('equations', required=True)
    if isinstance(raw_equations, (dict, list, tuple)) and len(raw_equations) > 128:
        raise AUTO_CB_Error('input_limit', 'Too many structural equations.')
    if isinstance(raw_equations, dict):
        entries = list(raw_equations.items())
    elif isinstance(raw_equations, (list, tuple)):
        entries = []
        for item in raw_equations:
            AUTO_CB_check(context)
            item = AUTO_CB_decode(item)
            if isinstance(item, str):
                match = re.fullmatch("\\s*([\\w]+(?:[.'-]\\w+)*)\\s*(?::=|=(?!=|>))\\s*(.+)\\s*", item, re.S)
                if not match:
                    raise AUTO_CB_Error('unsupported_equation', "Equation strings must use 'variable := expression' or 'variable = expression'; ordinary arrows are not assignments.")
                entries.append((match[1], match[2]))
            elif isinstance(item, (list, tuple)) and len(item) == 2:
                entries.append((item[0], item[1]))
            elif isinstance(item, dict):
                name, _ = AUTO_CB_pick(item, ('variable', 'var', 'name', 'target', 'lhs'), required=True)
                rhs, _ = AUTO_CB_pick(item, ('expression', 'expr', 'rhs', 'value'), required=True)
                if set(item) - {'variable', 'var', 'name', 'target', 'lhs', 'expression', 'expr', 'rhs', 'value'}:
                    raise AUTO_CB_Error('unsupported_equation', 'Unknown equation fields were not discarded.')
                entries.append((name, rhs))
            else:
                raise AUTO_CB_Error('unsupported_equation', 'Unsupported structural equation representation.')
    else:
        raise AUTO_CB_Error('invalid_equations', 'Structural equations must be a mapping or a list of assignments.')
    if len(entries) > 128:
        raise AUTO_CB_Error('input_limit', 'Too many structural equations.')
    equations, dependencies = ({}, set())
    for name, value in entries:
        AUTO_CB_check(context)
        name = AUTO_CB_name(name)
        expression = AUTO_CB_expression(value, context)
        dependencies.update(AUTO_CB_symbols(expression, context))
        if name in equations and equations[name] != expression:
            raise AUTO_CB_Error('multiple_structural_equations', 'Different equations supplied for ' + name)
        equations[name] = expression
    priors_raw, _ = field('priors', {})
    if not isinstance(priors_raw, dict):
        raise AUTO_CB_Error('invalid_prior_schema', 'Priors must map variable names to Bernoulli marginal probabilities.')
    if len(priors_raw) > 128:
        raise AUTO_CB_Error('input_limit', 'Too many marginal-prior declarations.')
    priors = {AUTO_CB_name(name): AUTO_CB_prior(value) for name, value in priors_raw.items()}
    roots_raw, roots_explicit = field('roots', [])
    if isinstance(roots_raw, (dict, list, tuple)) and len(roots_raw) > 128:
        raise AUTO_CB_Error('input_limit', 'Too many exogenous-root declarations.')
    roots = set()
    if isinstance(roots_raw, dict):
        for name, declaration in roots_raw.items():
            roots.add(AUTO_CB_name(name))
            if declaration is not None:
                probability = AUTO_CB_prior(declaration)
                if name in priors and priors[name] != probability:
                    raise AUTO_CB_Error('conflicting_root_prior', 'Conflicting root marginal for ' + name)
                priors[name] = probability
    elif isinstance(roots_raw, (list, tuple)):
        roots = {AUTO_CB_name(name) for name in roots_raw}
    else:
        raise AUTO_CB_Error('ambiguous_exogenous_model', 'Declare roots as names or a mapping to explicit marginal distributions.')
    if roots & equations.keys():
        raise AUTO_CB_Error('ambiguous_exogenous_model', 'A variable cannot be both an exogenous root and equation-defined.')
    inferred = (dependencies | set(priors)) - equations.keys()
    if roots_explicit and (not inferred <= roots):
        raise AUTO_CB_Error('incomplete_exogenous_declaration', 'Explicit roots omit model root variables: ' + ', '.join(sorted(inferred - roots)))
    roots |= inferred
    variables = set(equations) | dependencies | roots | set(priors)
    if 'variables' in selected:
        declared = AUTO_CB_decode(selected['variables'])
        if not isinstance(declared, (list, tuple)):
            raise AUTO_CB_Error('unsupported_variable_declaration', 'variables must be a list of Boolean variable names.')
        declared = {AUTO_CB_name(name) for name in declared}
        if declared != variables:
            raise AUTO_CB_Error('ambiguous_variable_declaration', 'Declared variables must match equation variables, roots and priors exactly.', missing=sorted(variables - declared), unexplained=sorted(declared - variables))
    if not variables or len(variables) > 16:
        raise AUTO_CB_Error('variable_limit', 'A model must contain between one and sixteen declared Boolean variables.')
    mode, mode_explicit = field('timing', 'factual')
    if not isinstance(mode, str) or mode.lower() not in {'factual', 'counterfactual'}:
        raise AUTO_CB_Error('invalid_observation_mode', 'Observation timing must be factual or counterfactual.')
    mode = mode.lower()
    observed_raw, _ = field('observations', {})
    observed = AUTO_CB_assignments(observed_raw, context, 'observations')
    factual = AUTO_CB_assignments(field('factual', {})[0], context, 'factual_evidence')
    counterfactual = AUTO_CB_assignments(field('counterfactual', {})[0], context, 'counterfactual_evidence')
    intervention = AUTO_CB_assignments(field('interventions', {})[0], context, 'interventions')
    query_raw, _ = field('query', required=True)
    if isinstance(query_raw, dict) and (not query_raw):
        raise AUTO_CB_Error('unsupported_query', 'An empty query object is ambiguous; use an explicit Boolean constant if intended.')
    query_ast = isinstance(query_raw, dict) and bool(set(query_raw) & {'op', 'operator', 'type', 'const', 'constant'})
    if isinstance(query_raw, dict) and (not query_ast) and any((key in query_raw for key in ('variable', 'name', 'value', 'state'))):
        assignments = AUTO_CB_assignments(query_raw, context, 'query')
        query = ('and', tuple((('var', name) if value else ('not', ('var', name)) for name, value in assignments.items())))
    elif isinstance(query_raw, dict) and (not set(query_raw) & {'var', 'atom', 'const', 'constant', 'not', 'and', 'or', 'xor', 'iff', 'implies', 'op', 'operator', 'type'}):
        assignments = AUTO_CB_assignments(query_raw, context, 'query')
        query = ('and', tuple((('var', name) if value else ('not', ('var', name)) for name, value in assignments.items())))
    else:
        query = AUTO_CB_expression(query_raw, context)
    query_variables = AUTO_CB_symbols(query, context)
    for role, names in (('intervention', set(intervention)), ('observation', set(observed)), ('factual_evidence', set(factual)), ('counterfactual_evidence', set(counterfactual)), ('query', query_variables)):
        if not names <= variables:
            raise AUTO_CB_Error('undefined_' + role + '_variable', 'Undeclared ' + role + ' variable(s): ' + ', '.join(sorted(names - variables)))
    return {'format': 'AUTO_CB_boolean_scm_v1', 'equations': equations, 'variables': sorted(variables), 'roots': sorted(roots), 'priors': {name: str(value) for name, value in priors.items()}, 'intervention': intervention, 'observations': observed, 'factual': factual, 'counterfactual': counterfactual, 'query_expression': query, 'observation_mode': mode, 'observation_mode_explicit': mode_explicit, 'normalization': {'case_sensitive_names': True, 'equation_semantics': 'deterministic Boolean assignment', 'root_policy': 'Explicit roots, or otherwise equation dependencies without defining equations.', 'prior_semantics': 'unconditional Bernoulli marginals; F1 exact supported analysis', 'logical_arrows_alone_are_not_causal': True}}

def normalize_causal_problem(record, fields=None):
    context = {'deadline': time.perf_counter() + 0.25, 'nodes': 0}
    try:
        return AUTO_CB_normalize(record, fields, context)
    except AUTO_CB_Error:
        raise
    except (TypeError, ValueError, RecursionError, OverflowError) as error:
        raise AUTO_CB_Error('unsupported_causal_schema', str(error)) from error

def AUTO_CB_safe(value):
    if isinstance(value, Fraction):
        return str(value)
    if isinstance(value, dict):
        return {str(key): AUTO_CB_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set, frozenset)):
        return [AUTO_CB_safe(item) for item in value]
    if value is None or isinstance(value, (str, bool, int)):
        return value
    if isinstance(value, float) and math.isfinite(value):
        return value
    return str(value)

def solve_structured_causal(problem, timeout=2.0):
    start, trace, selected = (time.perf_counter(), [], {})
    diagnostics = {'format': 'structured Boolean SCM', 'factual_root_contexts': 0, 'factual_worlds_examined': 0, 'counterfactual_worlds_examined': 0, 'conditioned_worlds': 0}

    def answer(code, reason, flag=False, status='unsupported', resolved=False, **details):
        safe = AUTO_CB_safe({**diagnostics, **details})
        return {'flag': bool(flag), 'resolved': bool(resolved), 'status': status, 'category': code, 'reason_code': code, 'reason': reason, 'trace': list(trace), 'seconds': time.perf_counter() - start, 'engine': 'counterbench', 'details': safe, 'diagnostics': safe, 'input': AUTO_CB_safe(selected), 'proof_kind': 'unanimous_counterfactual_models' if resolved else 'none'}
    try:
        timeout = float(timeout)
        if not math.isfinite(timeout) or timeout < 0:
            raise AUTO_CB_Error('invalid_timeout', 'timeout must be finite and nonnegative.')
        deadline = start + timeout
        context = {'deadline': deadline, 'nodes': 0}
        AUTO_CB_check(context)
        required = ('CB_worlds', 'CB_resolve_marginals', 'CB_v3_has_cycle', 'CB_evaluate')
        if not all((callable(globals().get(name)) for name in required)):
            raise AUTO_CB_Error('causal_engine_unavailable', 'Load the unchanged F1 causal helper definitions before this adapter.')
        selected = AUTO_CB_normalize(problem, None, context)
        if isinstance(problem, dict) and problem.get('format') == 'AUTO_CB_boolean_scm_v1':
            explicit = problem.get('observation_mode_explicit', True)
            if isinstance(explicit, bool):
                selected['observation_mode_explicit'] = explicit
        mode = selected['observation_mode']
        factual, counterfactual = (dict(selected['factual']), dict(selected['counterfactual']))
        AUTO_CB_merge(factual if mode == 'factual' else counterfactual, selected['observations'], 'factual_evidence' if mode == 'factual' else 'counterfactual_evidence')
        internal = {'equations': selected['equations'], 'variables': set(selected['variables']), 'priors': {name: Fraction(value) for name, value in selected['priors'].items()}, 'factual': factual, 'counterfactual': counterfactual, 'intervention': selected['intervention']}
        internal, marginal_info = CB_resolve_marginals(internal, deadline)
        equations, variables = (internal['equations'], internal['variables'])
        cyclic = CB_v3_has_cycle(equations)
        diagnostics.update(observation_mode=mode, equation_interpretation='deterministic_assignment', factual_evidence=factual, counterfactual_evidence=counterfactual, interventions=internal['intervention'], query_expression=selected['query_expression'], marginal_prior_analysis=marginal_info, has_cycle=cyclic, root_support='F1 Cartesian positive-support reasoning; no probability estimate')
        roots = sorted(variables - equations.keys())
        outcomes, factual_models = (set(), False)
        for bits in itertools.product((False, True), repeat=len(roots)):
            AUTO_CB_solve_check(deadline)
            root_values = dict(zip(roots, bits))
            if any((internal['priors'].get(name) == 0 and value or (internal['priors'].get(name) == 1 and (not value)) for name, value in root_values.items())):
                continue
            admissible = False
            for world in CB_worlds(equations, root_values, {}, deadline):
                factual_models = True
                diagnostics['factual_worlds_examined'] += 1
                if all((world[name] == value for name, value in factual.items())):
                    admissible = True
                    break
            if not admissible:
                continue
            diagnostics['factual_root_contexts'] += 1
            has_model = False
            for world in CB_worlds(equations, root_values, internal['intervention'], deadline):
                has_model = True
                diagnostics['counterfactual_worlds_examined'] += 1
                if not all((world[name] == value for name, value in counterfactual.items())):
                    continue
                diagnostics['conditioned_worlds'] += 1
                outcome = bool(CB_evaluate(selected['query_expression'], world))
                outcomes.add(outcome)
                if len(trace) < 3:
                    trace.append(f"Roots {root_values}; do {internal['intervention']}; query={outcome}; evidence timing={mode}.")
            if not has_model:
                return answer('cyclic_no_counterfactual_model' if cyclic else 'no_counterfactual_model', 'An admissible factual root context has no post-intervention fixed point.')
        if not factual_models:
            return answer('cyclic_no_factual_model' if cyclic else 'no_factual_model', 'No factual model exists on positive root support.')
        if not diagnostics['factual_root_contexts']:
            return answer('conflicting_factual_evidence', 'No factual model satisfies the supplied factual evidence.')
        if not outcomes:
            return answer('conflicting_counterfactual_evidence', 'No post-intervention model satisfies the counterfactual evidence.')
        if len(outcomes) != 1:
            return answer('ambiguous_counterfactual_outcome', 'Admissible counterfactual models disagree on the query.', status='unknown')
        flag = True in outcomes
        return answer('unanimous_counterfactual_outcome', 'All admissible conditioned models agree on the query.', flag=flag, status='true' if flag else 'false', resolved=True)
    except Exception as error:
        if isinstance(error, AUTO_CB_Error):
            return answer(error.code, str(error), status='timeout' if error.code.endswith('_timeout') else 'unsupported', **error.details)
        diagnostic_type, timeout_type = (globals().get('CB_Diagnostic'), globals().get('RT_Timeout'))
        if isinstance(diagnostic_type, type) and isinstance(error, diagnostic_type):
            return answer(error.code, str(error), **error.details)
        if isinstance(timeout_type, type) and isinstance(error, timeout_type):
            return answer('reasoning_timeout', 'Structured causal reasoning time limit reached.', status='timeout')
        return answer('unsupported_causal_schema', type(error).__name__ + ': ' + str(error))
AUTO_TYPED_FOL_PREMISE_COLUMNS = ('premises-FOL', 'premises_fol')
AUTO_TYPED_FOL_QUERY_COLUMNS = ('conclusion-FOL', 'conclusion_fol')
AUTO_TYPED_FOL_REFERENCE_COLUMNS = {'label', 'labels', 'answer', 'answers', 'gold', 'gold_label', 'target', 'flag', 'prediction', 'predictions', 'predicted_label', 'predicted_labels', 'predicted_flag', 'reasoning', 'explanation', 'explanations', 'expected', 'expected_answer', 'expected_label', 'ground_truth', 'reference_answer'}

def AUTO_TYPED_FOL_symbolic_string(value):
    if not isinstance(value, str) or not value.strip():
        return False
    text = value.strip()
    if text.startswith(('[', '{')):
        return True
    if re.search('[∀∃¬∧∨⊕→⇒↔⇔⟷≠&|=]|<->|->|—>|=>|\\?\\w', text):
        return True
    probe = text.strip('() \t\r\n')
    if re.match('^[!~]', probe):
        return True
    name = "\\w+(?:[-']\\w+)*(?:\\.\\w+(?:[-']\\w+)*)*"
    prefix = '(?:(?:not\\s+)|[!~]\\s*)*'
    if re.match(f'^{prefix}{name}\\s*\\(', probe, re.I):
        return True
    if re.match('^(?:forall|exists)\\s+\\??\\w', probe, re.I):
        return True
    if re.fullmatch(f'{prefix}{name}(?:\\s+(?:and|or|implies|iff|xor)\\s+{prefix}{name})*\\.?', probe, re.I):
        return True
    if re.fullmatch('(?:\'(?:[^\'\\\\]|\\\\.)*\'|"(?:[^"\\\\]|\\\\.)*")', probe):
        return True
    return False

def prefer_explicit_formal_columns(record, fields=None):
    if not isinstance(record, dict):
        raise ValueError('A problem record must be a dictionary')
    if fields is not None and (not isinstance(fields, dict)):
        raise ValueError('Column mappings must be a dictionary')
    fields = {} if fields is None else fields
    for role in ('premises', 'query'):
        if role in fields:
            name = fields[role]
            if not isinstance(name, str) or not name.strip():
                raise ValueError(f'Column mapping for {role} must name a field')
            if name.lower() in AUTO_TYPED_FOL_REFERENCE_COLUMNS:
                raise ValueError('Reference-label columns cannot supply a formal problem field')
    copied = record.copy()
    premise_columns = [key for key in AUTO_TYPED_FOL_PREMISE_COLUMNS if key in record and record[key] is not None]
    query_columns = [key for key in AUTO_TYPED_FOL_QUERY_COLUMNS if key in record and record[key] is not None]
    pair = any((isinstance(record[key], (str, list, tuple, dict)) for key in premise_columns)) and any((isinstance(record[key], (str, dict)) for key in query_columns))
    notes = {'typed_formal_pair_present': bool(pair), 'applied': False, 'suppressed_parallel_text_fields': [], 'preserved_symbolic_alternatives': [], 'formal_premise_columns': premise_columns, 'formal_query_columns': query_columns, 'explicit_role_overrides': [role for role in ('premises', 'query') if role in fields], 'original_record_unchanged': True}
    if not pair:
        return (copied, notes)
    for role, alternate in (('premises', 'premises'), ('query', 'conclusion')):
        if role in fields or alternate not in record:
            continue
        value = record[alternate]
        if not isinstance(value, str):
            continue
        if AUTO_TYPED_FOL_symbolic_string(value):
            notes['preserved_symbolic_alternatives'].append(alternate)
        else:
            copied.pop(alternate, None)
            notes['suppressed_parallel_text_fields'].append(alternate)
    notes['applied'] = bool(notes['suppressed_parallel_text_fields'])
    return (copied, notes)
AUTO_REPRESENTATION_KEYS = ('representation', 'logical_format', 'logic_type', 'input_format', 'format')
AUTO_FORMAT_ALIASES = {'fol': 'fol', 'first_order_logic': 'fol', 'first-order-logic': 'fol', 'first order logic': 'fol', 'symbolic': 'fol', 'propositional': 'fol', 'horn': 'fol', 'cnf': 'fol', 'clauses': 'fol', 'ast': 'fol', 'causal': 'causal', 'scm': 'causal', 'structural_causal_model': 'causal', 'structural causal model': 'causal', 'auto_cb_boolean_scm_v1': 'causal'}
AUTO_SEMANTIC_KEYS = {'premises', 'premises-FOL', 'premises_fol', 'context', 'facts', 'rules', 'clauses', 'query', 'statement', 'conclusion-FOL', 'conclusion_fol', 'question', 'equations', 'structural_equations', 'causal_model', 'interventions', 'intervention', 'do', 'observations', 'evidence', 'factual_evidence', 'counterfactual_evidence', 'observation_mode', 'roots', 'exogenous', 'variables', 'priors', 'given_info', 'constraints', 'assumptions'} | {alias for aliases in AUTO_FOL_ALIASES.values() for alias in aliases} | {alias for aliases in AUTO_CB_ALIASES.values() for alias in aliases}

def AUTO_clean_blank_cells(record):
    known = AUTO_SEMANTIC_KEYS | set(AUTO_REPRESENTATION_KEYS) | {'semantics', 'logic_semantics'}
    blank = {key for key in known if key in record and isinstance(record[key], str) and (not record[key].strip())}
    output = {key: value for key, value in record.items() if key not in blank}
    knowledge = {alias for role, aliases in AUTO_FOL_ALIASES.items() if role != 'query' for alias in aliases}
    has_envelope = any((isinstance(output.get(key), dict) for key in ('problem', 'instance', 'input')))
    has_causal = any((key in output for key in ('equations', 'structural_equations', 'causal_model')))
    if blank & knowledge and (not knowledge.intersection(output)) and (not has_envelope) and (not has_causal):
        output['premises'] = []
    return output

def AUTO_unwrap_record(record, fields=None):
    if not isinstance(record, dict):
        raise ValueError('Each problem must be a dictionary')
    record = AUTO_clean_blank_cells(record)
    envelopes = [(key, record[key]) for key in ('problem', 'instance', 'input') if key in record and isinstance(record[key], dict)]
    if not envelopes:
        return record
    payload = AUTO_clean_blank_cells(envelopes[0][1])
    if any((AUTO_clean_blank_cells(value) != payload for _, value in envelopes[1:])):
        raise ValueError('Conflicting problem envelopes; provide one converted problem per record')
    legacy_metadata = ('type', *AUTO_CB_ALIASES['timing']) if isinstance(payload.get('given_info'), str) and isinstance(payload.get('question'), str) else ()
    if legacy_metadata and (fields or {}).get('timing'):
        legacy_metadata += (fields['timing'],)
    metadata_keys = tuple(dict.fromkeys(AUTO_REPRESENTATION_KEYS + ('observation_mode', 'semantics', 'logic_semantics') + legacy_metadata))
    overlapping = [key for key in AUTO_SEMANTIC_KEYS if key in record and key not in metadata_keys]
    if overlapping:
        raise ValueError('Both outer and nested problem fields are present: ' + ', '.join(sorted(overlapping)))
    output = dict(payload)
    for key in metadata_keys:
        if key in record:
            if key in output and output[key] != record[key]:
                raise ValueError(f'Conflicting outer and nested {key!r}')
            output[key] = record[key]
    return output

def AUTO_representation(record):
    choices = []
    for key in AUTO_REPRESENTATION_KEYS:
        if key not in record or record[key] is None:
            continue
        value = record[key]
        if not isinstance(value, str):
            raise ValueError(f'{key} must name a supported representation')
        normalized = value.strip().lower()
        if normalized in {'json', 'jsonl', 'csv', 'auto'}:
            continue
        if normalized not in AUTO_FORMAT_ALIASES:
            raise ValueError(f'Unsupported explicit representation {value!r}')
        choices.append(AUTO_FORMAT_ALIASES[normalized])
    if len(set(choices)) > 1:
        raise ValueError('Conflicting representation metadata')
    return choices[0] if choices else None

def AUTO_validate_semantics(record, requested):
    aliases = {'classical': 'classical', 'classical_logic': 'classical', 'ground_horn': 'ground_horn'}
    choices = []
    for key in ('semantics', 'logic_semantics'):
        if key not in record or record[key] is None:
            continue
        raw = record[key]
        value = raw.strip().lower() if isinstance(raw, str) else None
        if value not in aliases:
            raise ValueError(f'Unsupported declared semantics {raw!r}; it cannot be assumed classical')
        choices.append(aliases[value])
    if len(set(choices)) > 1 or (choices and choices[0] != requested):
        raise ValueError('Record semantics conflict with the selected inference semantics')

def AUTO_validate_fields(fields):
    if fields is None:
        return
    allowed = AUTO_FOL_FIELD_ROLES | set(AUTO_CB_ALIASES)
    if not isinstance(fields, dict) or set(fields) - allowed:
        raise ValueError('Unsupported field mapping; use documented formal or causal field roles')
    if any((not isinstance(value, str) or not value.strip() for value in fields.values())):
        raise ValueError('Field mappings must name nonempty column names')
    references = {'gold', 'gold_label', 'label', 'labels', 'answer', 'answers', 'flag', 'prediction', 'predictions', 'predicted_label', 'predicted_labels', 'predicted_flag', 'reasoning', 'explanation', 'explanations', 'expected', 'expected_answer', 'expected_label', 'ground_truth', 'reference_answer'}
    if any((role != 'id' and column.casefold() in references for role, column in fields.items())):
        raise ValueError('Reference-answer and explanation columns cannot supply logical input fields')

def AUTO_formal_fields(fields):
    return {key: value for key, value in (fields or {}).items() if key in AUTO_FOL_FIELD_ROLES}

def AUTO_has_causal_model(record, fields=None):
    mapped = (fields or {}).get('equations')
    return is_structured_causal_record(record) or bool(mapped and mapped in record)

def AUTO_formal_extra_semantics(record, fields=None):
    causal_only = {alias for role, aliases in AUTO_CB_ALIASES.items() if role != 'query' for alias in aliases} | {'variables', 'causal_model'}
    causal_only |= {column for role, column in (fields or {}).items() if role in AUTO_CB_ALIASES and role != 'query'}
    return sorted((causal_only | {'constraints', 'assumptions'}) & record.keys())

def AUTO_legacy_horn_record(record, fields=None):
    try:
        context = input_field(record, 'context', fields)
        query = input_field(record, 'query', fields)
    except (KeyError, TypeError, ValueError):
        return False
    if not isinstance(context, str) or not isinstance(query, str) or (not query.strip()):
        return False
    if re.search('[()∀∃¬∧∨⊕→⇒↔⇔⟷=&|~!]|->|<->|—>|\\?\\w|\\b(?:forall|exists|implies|iff|xor)\\b', context + '\n' + query, re.I):
        return False
    return re.fullmatch('\\s*Facts\\s*:\\s*(.*?)\\s*Rules\\s*:\\s*(.*?)\\s*', context, re.I | re.S) is not None

def AUTO_prepare_legacy_causal(record, fields=None, default_mode='factual'):
    for role in ('given_info', 'question'):
        if not isinstance(input_field(record, role, fields), str):
            raise ValueError('Legacy causal given_info/question must be text')
    mapped = (fields or {}).get('timing')
    keys = (mapped,) if mapped else AUTO_CB_ALIASES['timing']
    declared = []
    for key in keys:
        value = record.get(key)
        if value is None or (isinstance(value, str) and (not value.strip())):
            continue
        if not isinstance(value, str) or value.strip().lower() not in {'factual', 'counterfactual'}:
            raise ValueError('Observation timing must be factual or counterfactual')
        declared.append(value.strip().lower())
    if len(set(declared)) > 1:
        raise ValueError('Conflicting explicit causal observation timings')
    task = record.get('type')
    conditional = isinstance(task, str) and task.strip().lower() == 'conditional'
    if declared:
        mode, source = (declared[0], 'explicit_timing')
    elif conditional:
        mode, source = ('counterfactual', 'legacy_conditional_task_contract')
    else:
        mode, source = (default_mode, 'caller_default')
    if not isinstance(mode, str) or mode.strip().lower() not in {'factual', 'counterfactual'}:
        raise ValueError('Default observation timing must be factual or counterfactual')
    mode = mode.strip().lower()
    return (dict(record, observation_mode=mode), {'mode': mode, 'source': source})

def AUTO_input_identifier(record, fields=None):
    keys = ((fields or {}).get('id'), *INPUT_ALIASES['id'])
    candidates = [record] + [record[key] for key in ('problem', 'instance', 'input') if isinstance(record.get(key), dict)]
    candidates = [{key: value for key, value in item.items() if not (key in keys and isinstance(value, str) and (not value.strip()))} for item in candidates]
    identifiers = [input_field(item, 'id', fields) for item in candidates if any((key in item and item[key] is not None for key in keys if key is not None))]
    if not identifiers:
        raise ValueError('No supplied problem identifier')
    if any((value != identifiers[0] for value in identifiers[1:])):
        raise ValueError('Conflicting outer and nested problem identifiers')
    return identifiers[0]

def AUTO_batch_identifier(record, fields, row_number, generate_missing=True):
    keys = ((fields or {}).get('id'), *INPUT_ALIASES['id'])
    candidates = [record] + [record[key] for key in ('problem', 'instance', 'input') if isinstance(record.get(key), dict)]
    supplied = any((key in candidate and candidate[key] is not None and (not (isinstance(candidate[key], str) and (not candidate[key].strip()))) for candidate in candidates for key in keys if key is not None))
    if supplied or not generate_missing:
        return (AUTO_input_identifier(record, fields), False)
    return (f'auto-row-{row_number}', True)

def AUTO_select_route(record, fields=None):
    payload = AUTO_unwrap_record(record, fields)
    payload, typed_columns = prefer_explicit_formal_columns(payload, fields)
    declared = AUTO_representation(payload)
    structured_causal = AUTO_has_causal_model(payload, fields)
    formal = is_formal_record(payload, AUTO_formal_fields(fields))
    causal_text = isinstance(payload.get('given_info'), str) and isinstance(payload.get('question'), str)
    legacy_horn = AUTO_legacy_horn_record(payload, fields)
    checked = {'formal_record': formal, 'structured_causal_record': structured_causal, 'legacy_causal_text': causal_text, 'ground_facts_rules_text': legacy_horn, 'typed_formal_columns': typed_columns}
    if declared == 'fol':
        if structured_causal or any((key in payload for key in ('interventions', 'intervention', 'do'))):
            raise ValueError('Causal model/intervention fields conflict with ordinary FOL metadata')
        return (payload, 'fol', 'Explicit formal representation', checked)
    if declared == 'causal':
        if structured_causal:
            return (payload, 'structured_causal', 'Explicit causal representation and model', checked)
        if causal_text:
            return (payload, 'causal', 'Explicit causal representation with legacy text fields', checked)
        raise ValueError('Causal input needs explicit structural equations or legacy given_info/question')
    if structured_causal:
        formal_inputs = {alias for role, aliases in AUTO_FOL_ALIASES.items() if role != 'query' for alias in aliases}
        conflicting = sorted(formal_inputs & payload.keys())
        if conflicting:
            raise ValueError('Both causal equations and ordinary logical premises/rules are present; their relationship is unspecified')
        return (payload, 'structured_causal', 'Explicit structural-equation schema', checked)
    if causal_text and formal:
        raise ValueError('Both formal and legacy causal-text schemas are present')
    if legacy_horn:
        formal_inputs = {alias for role, aliases in AUTO_FOL_ALIASES.items() if role != 'query' for alias in aliases} - {'context'}
        extra = AUTO_formal_extra_semantics(payload, fields)
        if formal or causal_text or formal_inputs.intersection(payload) or extra:
            raise ValueError('Ground Facts:/Rules: text is mixed with another logical representation')
        return (payload, 'ruletaker', 'Recognized ground Facts:/Rules: grammar', checked)
    if formal:
        if any((key in payload for key in ('interventions', 'intervention', 'do'))):
            raise ValueError('Interventions require an explicit causal model; they are not ordinary facts')
        return (payload, 'fol', 'Converted premises/facts/rules/clauses with a query', checked)
    if causal_text:
        return (payload, 'causal', 'Legacy causal-text field contract', checked)
    raise ValueError('Unrecognized converted input. Supported: symbolic premises/query, structured atoms and rules, explicit clauses/ASTs, or structural causal equations and interventions')

def predict(record, mode='auto', semantics='classical', timeout=2.0, fields=None, causal_observation_mode='factual'):
    started = perf_counter()
    route, reason, checked = (None, '', {})
    normalized = None
    try:
        AUTO_validate_fields(fields)
        if mode not in {'auto', 'fol', 'ruletaker', 'causal'}:
            raise ValueError('mode must be auto, fol, ruletaker, or causal')
        if mode == 'auto':
            payload, route, reason, checked = AUTO_select_route(record, fields)
        else:
            payload = AUTO_unwrap_record(record, fields)
            payload, typed_columns = prefer_explicit_formal_columns(payload, fields)
            checked['typed_formal_columns'] = typed_columns
            route, reason = (mode, 'Explicit caller mode')
            declared = AUTO_representation(payload)
            expected = 'fol' if mode == 'fol' else 'causal' if mode == 'causal' else None
            if declared is not None and (expected is None or declared != expected):
                raise ValueError('Record representation conflicts with the explicit caller mode')
            if mode == 'causal' and AUTO_has_causal_model(payload, fields):
                route = 'structured_causal'
        AUTO_validate_semantics(payload, semantics)
        if route != 'fol' and any((key in payload for key in ('constraints', 'assumptions'))):
            raise ValueError('Extra constraints/assumptions need a documented interpretation; they were not discarded')
        if route == 'fol':
            if semantics != 'classical':
                raise ValueError('Formal input requires classical semantics')
            extra = AUTO_formal_extra_semantics(payload, fields)
            if extra:
                raise ValueError('Ordinary FOL input cannot discard extra semantic fields: ' + ', '.join(extra) + '. Put ordinary constraints in premises; represent causal observations/interventions explicitly.')
            normalized = normalize_formal_problem(payload, AUTO_formal_fields(fields))
            result = _predict_f1(normalized, 'fol', 'classical', timeout)
        elif route == 'structured_causal':
            if semantics != 'classical':
                raise ValueError('Causal input requires classical semantics')
            normalized = normalize_causal_problem(payload, fields)
            if not normalized.get('observation_mode_explicit', False):
                normalized['observation_mode'] = causal_observation_mode
            result = solve_structured_causal(normalized, timeout)
            result['engine'] = 'counterbench'
            result['input'] = normalized
        else:
            effective = 'ground_horn' if mode == 'auto' and route == 'ruletaker' and (semantics == 'classical') else semantics
            if route == 'causal':
                payload, timing = AUTO_prepare_legacy_causal(payload, fields, causal_observation_mode)
                checked['causal_observation_timing'] = timing
            result = _predict_f1(payload, route, effective, timeout, fields, causal_observation_mode)
        if not isinstance(result.get('flag'), bool):
            raise ValueError('The selected engine did not return a Boolean flag')
        if 'category' not in result:
            result['category'] = result.get('reason_code', 'established' if result.get('resolved') else result.get('status', 'unsupported'))
    except Exception as error:
        result = {'flag': False, 'resolved': False, 'status': 'unsupported', 'trace': [], 'engine': 'router', 'category': 'input_representation', 'reason': f'{type(error).__name__}: {error}'}
        if normalized is not None:
            result['input'] = normalized
    result['routing'] = {'requested_mode': mode, 'selected_route': route, 'selection_reason': reason, 'schemas_checked': checked, 'selection_uses_reference_answers': False}
    result['route'] = route
    result['seconds'] = perf_counter() - started
    return result


In [9]:
def read_input(path):
    path = Path(path)
    if path.suffix.lower() == '.csv':
        with path.open(encoding='utf-8-sig', newline='') as stream:
            records = list(csv.DictReader(stream))
    elif path.suffix.lower() in ('.json', '.jsonl'):
        text = path.read_text(encoding='utf-8-sig')
        if path.suffix.lower() == '.jsonl':
            records = []
            for line_number, line in enumerate(text.splitlines(), 1):
                if line.strip():
                    try:
                        records.append(json.loads(line))
                    except json.JSONDecodeError as error:
                        raise ValueError(f'Invalid JSON on line {line_number}: {error}') from error
        else:
            records = json.loads(text)
            if isinstance(records, dict):
                records = AUTO_records_container(records)
    else:
        raise ValueError('Use a .json, .jsonl, or .csv input file')
    if not isinstance(records, list) or not all((isinstance(row, dict) for row in records)):
        raise ValueError('Input must contain problem dictionaries')
    if not records:
        raise ValueError('The input file contains no records')
    return records

def normalize_flag(value, allow_uncertain=False):
    if isinstance(value, bool):
        return value
    text = str(value).strip().lower()
    if text in {'true', 'yes', 'entailment', '1'}:
        return True
    if text in {'false', 'no', 'not entailment', '0'}:
        return False
    if allow_uncertain and text == 'uncertain':
        return False
    raise ValueError(f'Unrecognized binary reference label: {value!r}')

def run_records(records, mode='auto', semantics='classical', timeout=2.0, progress_every=25, fields=None, causal_observation_mode='factual', generate_missing_ids=True):
    records = list(records)
    if not records:
        raise ValueError('No input records were supplied')
    identifiers, generated_flags = ([], [])
    for row_number, record in enumerate(records, 1):
        if not isinstance(record, dict):
            raise ValueError(f'Row {row_number} is not a dictionary')
        identifier, generated = AUTO_batch_identifier(record, fields, row_number, generate_missing_ids)
        if isinstance(identifier, bool) or not isinstance(identifier, (str, int)) or (not str(identifier).strip()):
            raise ValueError(f'Row {row_number} needs a nonempty string/integer ID')
        identifiers.append(str(identifier))
        generated_flags.append(generated)
    if len(set(identifiers)) != len(identifiers):
        raise ValueError('Input identifiers are not unique; no row was silently dropped')
    if any(generated_flags):
        print(f'Generated {sum(generated_flags)} row IDs; map an existing ID column through TA_FIELDS when available.')
    results = []
    started = perf_counter()
    for index, (record, identifier) in enumerate(zip(records, identifiers), 1):
        result = predict(record, mode, semantics, timeout, fields, causal_observation_mode)
        result.update(id=identifier, source=record.get('source', 'TA input'), id_generated=generated_flags[index - 1])
        results.append(result)
        if progress_every and (index == 1 or index % progress_every == 0 or index == len(records)):
            print(f"  {index}/{len(records)} predictions; elapsed {perf_counter() - started:.1f}s; unresolved {sum((not row['resolved'] for row in results))}", flush=True)
    return results

def write_predictions(results, path):
    results = list(results)
    if not results:
        raise ValueError('No predictions to write')
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['id', 'flag'])
        writer.writeheader()
        writer.writerows(({'id': row['id'], 'flag': row['flag']} for row in results))
    path.with_suffix('.diagnostics.json').write_text(json.dumps(results, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'  Saved {len(results)} predictions: {path}')

def write_detailed_csv(results, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = ['source', 'id', 'flag', 'gold', 'resolved', 'status', 'category', 'engine', 'route', 'observation_mode', 'seconds', 'reason']
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(({key: row.get(key, '') for key in columns} for row in results))

def write_failure_report(results, path):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = ['source', 'id', 'flag', 'gold', 'resolved', 'status', 'category', 'seconds', 'reason', 'route', 'routing', 'formula_location', 'formula_index', 'formula', 'problem']
    count = 0
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        for result in results:
            wrong = 'gold' in result and result['flag'] != result['gold']
            if result['resolved'] and (not wrong):
                continue
            row = {key: result.get(key, '') for key in columns}
            parser_error = result.get('details', {}).get('parser_error', {})
            row['routing'] = json.dumps(result.get('routing', {}), ensure_ascii=False)
            row.update(formula_location=parser_error.get('location', ''), formula_index=parser_error.get('index', ''), formula=parser_error.get('formula', ''), problem=json.dumps(result.get('input', {}), ensure_ascii=False))
            writer.writerow(row)
            count += 1
    print(f'  Saved {count} unresolved or reference-disagreeing rows: {path}')

def score_results(results, gold_flags):
    if len(results) != len(gold_flags) or not results:
        raise ValueError('Scoring requires one reference label for every prediction')
    count = len(results)
    correct = sum((row['flag'] == gold for row, gold in zip(results, gold_flags)))
    established_correct = sum((row['resolved'] and row['flag'] == gold for row, gold in zip(results, gold_flags)))
    return {'questions': count, 'correct': correct, 'accuracy': correct / count, 'resolved': sum((bool(row['resolved']) for row in results)), 'resolved_and_correct_rate': established_correct / count, 'inference_seconds': sum((row['seconds'] for row in results))}

def AUTO_records_container(value):
    if isinstance(value, dict):
        arrays = [value[key] for key in ('records', 'problems', 'examples', 'data') if key in value and isinstance(value[key], list)]
        if arrays:
            if any((items != arrays[0] for items in arrays[1:])):
                raise ValueError('Conflicting JSON problem-list envelopes')
            return arrays[0]
        return [value]
    return value

def run_huggingface(dataset_name, split='test', config_name=None, output_csv='submission.csv', mode='auto', semantics='classical', timeout=2.0, label_field=None, fields=None, allow_uncertain=False, progress_every=25, causal_observation_mode='factual', revision=None, install_packages=False, login=False, generate_missing_ids=True):
    if not isinstance(dataset_name, str) or not dataset_name.strip():
        raise ValueError('Supply the Hugging Face dataset repository name')
    if not isinstance(split, str) or not split.strip():
        raise ValueError('Specify the intended Hugging Face split')
    try:
        datasets = importlib.import_module('datasets')
    except ImportError as error:
        if not install_packages:
            raise ImportError('Hugging Face input needs datasets and huggingface_hub. Set INSTALL_INPUT_HF_PACKAGES=True, or install them in this kernel.') from error
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'datasets', 'huggingface_hub'])
        datasets = importlib.import_module('datasets')
    if login:
        hub = importlib.import_module('huggingface_hub')
        hub.login()
    arguments = {'split': split}
    if config_name is not None:
        arguments['name'] = config_name
    if revision is not None:
        arguments['revision'] = revision
    dataset = datasets.load_dataset(dataset_name, **arguments)
    records = [dict(row) for row in dataset]
    print(f'Hugging Face input: {dataset_name}; split={split}; {len(records)} records', flush=True)
    results, summary = run_input_records(records, output_csv=output_csv, mode=mode, semantics=semantics, timeout=timeout, label_field=label_field, fields=fields, allow_uncertain=allow_uncertain, progress_every=progress_every, causal_observation_mode=causal_observation_mode, generate_missing_ids=generate_missing_ids)
    summary['input_source'] = {'kind': 'huggingface', 'dataset': dataset_name, 'config': config_name, 'split': split, 'requested_revision': revision, 'fingerprint': getattr(dataset, '_fingerprint', None)}
    Path(output_csv).with_suffix('.summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    return (results, summary)

def run_input_records(records, output_csv='submission.csv', mode='auto', semantics='classical', timeout=2.0, label_field=None, fields=None, allow_uncertain=False, progress_every=25, causal_observation_mode='factual', generate_missing_ids=True):
    records = list(records)
    results = run_records(records, mode, semantics, timeout, progress_every, fields, causal_observation_mode, generate_missing_ids)
    summary = {'questions': len(results), 'resolved': sum((bool(r['resolved']) for r in results)), 'inference_seconds': sum((r['seconds'] for r in results)), 'generated_ids': sum((bool(r.get('id_generated')) for r in results)), 'routes': dict(Counter((r.get('routing', {}).get('selected_route', r['engine']) for r in results)))}
    review_results = results
    write_predictions(results, output_csv)
    if label_field is not None:
        if any((label_field not in row for row in records)):
            raise ValueError(f'Each row needs the configured reference field {label_field!r}')
        gold_flags = [normalize_flag(row[label_field], allow_uncertain) for row in records]
        summary.update(score_results(results, gold_flags))
        review_results = [dict(row, gold=gold) for row, gold in zip(results, gold_flags)]
        print(f"Sample accuracy: {100 * summary['accuracy']:.2f}% ({summary['correct']}/{summary['questions']})")
    else:
        print('No reference labels configured; predictions and timing only.')
    write_failure_report(review_results, Path(output_csv).with_suffix('.failures.csv'))
    Path(output_csv).with_suffix('.summary.json').write_text(json.dumps(summary, indent=2), encoding='utf-8')
    print(f"Resolved: {summary['resolved']}/{summary['questions']}; inference time: {summary['inference_seconds']:.6f}s")
    return (results, summary)

def run_file(input_path, output_csv='submission.csv', mode='auto', semantics='classical', timeout=2.0, label_field=None, fields=None, allow_uncertain=False, progress_every=25, causal_observation_mode='factual', generate_missing_ids=True):
    records = read_input(input_path)
    print(f'File input: {len(records)} records; mode={mode}', flush=True)
    return run_input_records(records, output_csv, mode, semantics, timeout, label_field, fields, allow_uncertain, progress_every, causal_observation_mode, generate_missing_ids)


In [10]:
import re
AUTO_SCHEMA_QUERY_KEYS = {'query', 'statement', 'question', 'conclusion', 'hypothesis', 'goal', 'conclusion-FOL', 'conclusion_fol', 'query_expression'}
AUTO_SCHEMA_KB_KEYS = {'premises', 'premises-FOL', 'premises_fol', 'context', 'facts', 'rules', 'clauses', 'knowledge_base', 'kb'}
AUTO_SCHEMA_CAUSAL_KEYS = {'equations', 'structural_equations', 'causal_model'}
AUTO_SCHEMA_ENVELOPES = ('problem', 'instance', 'input')
AUTO_SCHEMA_MODEL_WRAPPERS = ('model', 'causal_model', 'graph')
AUTO_SCHEMA_QUESTION_METADATA = {'question_id', 'problem_id'}
AUTO_SCHEMA_QUESTION_CONTROLS = {'given_info', 'intervention', 'interventions', 'do', 'observation', 'observations', 'evidence', 'factual', 'factual_evidence', 'counterfactual', 'counterfactual_evidence', 'counterfactual_observations'}
AUTO_SCHEMA_FIELD_ROLES = {'id', 'premises', 'query', 'context', 'facts', 'rules', 'clauses', 'kb', 'given_info', 'question', 'equations', 'interventions', 'observations', 'factual', 'counterfactual', 'priors', 'roots', 'timing'}
AUTO_SCHEMA_REFERENCE_COLUMNS = {'gold', 'gold_label', 'label', 'labels', 'answer', 'answers', 'flag', 'prediction', 'predictions', 'predicted_label', 'predicted_labels', 'predicted_flag', 'reasoning', 'explanation', 'explanations', 'expected', 'expected_answer', 'expected_label', 'ground_truth', 'reference_answer'}
AUTO_SCHEMA_MAX_DEPTH = 16
AUTO_SCHEMA_MAX_OBJECTS = 10000
AUTO_SCHEMA_NONINPUT_BRANCHES = AUTO_SCHEMA_REFERENCE_COLUMNS | {'labels', 'answers', 'gold_labels', 'proof', 'proofs', 'rationale', 'solution', 'solutions', 'reference', 'references', 'reference_answer', 'reference_explanation', 'explanations', 'source', 'dataset', 'dataset_name', 'dataset_id', 'partition', 'split', 'provenance'}

def AUTO_SCHEMA_result(classification, schema, reason):
    return {'classification': classification, 'schema': schema, 'reason': reason}

def AUTO_SCHEMA_graph_container(value):
    if not isinstance(value, dict):
        return False
    if 'adjacency' in value and isinstance(value['adjacency'], dict):
        return True
    if 'nodes' in value and isinstance(value['nodes'], (list, tuple, dict)):
        return 'edges' not in value or isinstance(value['edges'], (list, tuple, dict))
    if 'edges' in value and isinstance(value['edges'], (list, tuple, dict)):
        return True
    return False

def AUTO_SCHEMA_nested_input_guard(value, query_keys, causal_keys, equation_keys):
    state = {'objects': 0}

    def walk(item, depth=0, ancestors=frozenset(), context='record'):
        if not isinstance(item, (dict, list, tuple, set)):
            return None
        state['objects'] += 1
        if depth > AUTO_SCHEMA_MAX_DEPTH or state['objects'] > AUTO_SCHEMA_MAX_OBJECTS or len(item) > AUTO_SCHEMA_MAX_OBJECTS or (id(item) in ancestors):
            return 'Nested input inspection exceeded safe structural limits'
        ancestors = ancestors | {id(item)}
        if isinstance(item, dict):
            keys = set(item)
            if context == 'record':
                if keys & query_keys or ('target' in keys and keys & causal_keys):
                    return 'A nested object contains a supported query marker'
                if keys & (set(AUTO_SCHEMA_ENVELOPES) | AUTO_SCHEMA_QUESTION_METADATA):
                    return 'A nested object contains a problem-shaped envelope or question identifier'
            for key in item:
                if context == 'record' and isinstance(key, str) and (key.casefold() in AUTO_SCHEMA_NONINPUT_BRANCHES):
                    continue
                child = item[key]
                if context == 'index':
                    child_context = 'record'
                elif context == 'expression' or key in equation_keys:
                    child_context = 'expression'
                elif key in {'nodes', 'edges', 'adjacency', 'variable_mapping'} and isinstance(child, dict):
                    child_context = 'index'
                else:
                    child_context = 'record'
                reason = walk(child, depth + 1, ancestors, child_context)
                if reason:
                    return reason
        else:
            for child in item:
                reason = walk(child, depth + 1, ancestors, 'expression' if context == 'expression' else 'record')
                if reason:
                    return reason
        return None
    return walk(value)

def describe_reasoning_record(record, fields=None):
    if fields is None:
        fields = {}
    if not isinstance(fields, dict) or set(fields) - AUTO_SCHEMA_FIELD_ROLES:
        return AUTO_SCHEMA_result('unrecognized', 'invalid_column_map', 'Unsupported column-map structure; retain the row for an explicit diagnostic')
    if any((not isinstance(column, str) or not column.strip() for column in fields.values())):
        return AUTO_SCHEMA_result('unrecognized', 'invalid_column_map', 'Column mappings must name nonempty fields')
    if any((role != 'id' and column.casefold() in AUTO_SCHEMA_REFERENCE_COLUMNS for role, column in fields.items())):
        return AUTO_SCHEMA_result('unrecognized', 'invalid_column_map', 'Reference columns cannot define logical-input schema')
    query_keys, kb_keys, causal_keys = (set(AUTO_SCHEMA_QUERY_KEYS), set(AUTO_SCHEMA_KB_KEYS), set(AUTO_SCHEMA_CAUSAL_KEYS))
    controls = set(AUTO_SCHEMA_QUESTION_CONTROLS)
    for role, column in fields.items():
        if role in {'query', 'question'} and column != 'target':
            query_keys.add(column)
        elif role in {'premises', 'context', 'facts', 'rules', 'clauses', 'kb'}:
            kb_keys.add(column)
        elif role == 'equations':
            causal_keys.add(column)
        elif role in {'given_info', 'interventions', 'observations', 'factual', 'counterfactual'}:
            controls.add(column)
    causal_query_keys = query_keys | {'target'}
    given_key, question_key = (fields.get('given_info', 'given_info'), fields.get('question', 'question'))
    context_key = fields.get('context', 'context')
    state = {'objects': 0}

    def inspect(value, depth=0, ancestors=frozenset()):
        state['objects'] += 1
        if depth > AUTO_SCHEMA_MAX_DEPTH or state['objects'] > AUTO_SCHEMA_MAX_OBJECTS:
            return AUTO_SCHEMA_result('unrecognized', 'inspection_limit', 'Structure exceeds bounded schema inspection; retain for a diagnostic')
        if not isinstance(value, dict):
            return AUTO_SCHEMA_result('unrecognized', 'non_dictionary', 'A reasoning row needs an object; retain this value for a diagnostic')
        if len(value) > AUTO_SCHEMA_MAX_OBJECTS:
            return AUTO_SCHEMA_result('unrecognized', 'inspection_limit', 'Object has too many fields for bounded inspection; retain')
        if id(value) in ancestors:
            return AUTO_SCHEMA_result('unrecognized', 'cyclic_object', 'Cyclic input cannot be classified safely')
        ancestors = ancestors | {id(value)}
        keys = set(value)

        def support(schema, reason):
            equations = {'equations', 'structural_equations'} | ({fields['equations']} if 'equations' in fields else set())
            marker = AUTO_SCHEMA_nested_input_guard(value, query_keys, causal_keys, equations)
            if marker:
                return AUTO_SCHEMA_result('unrecognized', 'nested_input_or_limit', marker + '; retain instead of excluding as support')
            return AUTO_SCHEMA_result('support', schema, reason)
        query_present = bool(keys & query_keys)
        kb_present = bool(keys & kb_keys)
        causal_present = bool(keys & causal_keys)
        causal_query_present = bool(keys & causal_query_keys)
        legacy_causal = given_key in value and question_key in value
        ordinary_problem = query_present and kb_present
        causal_problem = causal_present and causal_query_present or legacy_causal
        if ordinary_problem or causal_problem:
            if ordinary_problem and causal_problem:
                return AUTO_SCHEMA_result('problem', 'mixed_problem', 'Logical and causal input markers coexist; retain for the router to check ambiguity')
            if causal_problem:
                schema = 'legacy_causal_problem' if legacy_causal and (not causal_present) else 'causal_problem'
                return AUTO_SCHEMA_result('problem', schema, 'Query and causal knowledge fields are present; malformed values are retained')
            context = value.get(context_key)
            if isinstance(context, str) and len(context) <= 200000 and re.fullmatch('\\s*Facts\\s*:.*?Rules\\s*:.*', context, re.I | re.S):
                return AUTO_SCHEMA_result('problem', 'legacy_facts_rules_problem', 'Query and explicit Facts:/Rules: text fields are present')
            return AUTO_SCHEMA_result('problem', 'formal_problem', 'Query and logical knowledge fields are present; malformed values are retained')
        envelopes = [key for key in AUTO_SCHEMA_ENVELOPES if key in value]
        if envelopes:
            children = [inspect(value[key], depth + 1, ancestors) for key in envelopes]
            if any((child['classification'] == 'problem' for child in children)):
                return AUTO_SCHEMA_result('problem', 'problem_envelope', 'A supported envelope contains a candidate reasoning problem')
            if query_present or kb_present or causal_query_present or (len(envelopes) != 1):
                return AUTO_SCHEMA_result('unrecognized', 'ambiguous_envelope', 'Outer or conflicting envelope fields may be an incomplete problem; retain')
            if children[0]['classification'] == 'support':
                return AUTO_SCHEMA_result('unrecognized', 'incomplete_problem_envelope', 'A problem-shaped envelope contains a model but no query; retain instead of assuming support-only')
            return AUTO_SCHEMA_result('unrecognized', 'incomplete_problem_envelope', 'Problem-shaped envelope lacks a complete recognized problem or support schema')
        if query_present or causal_query_present:
            return AUTO_SCHEMA_result('unrecognized', 'query_without_recognized_kb', 'A query-shaped field exists without a recognized knowledge base; retain')
        if kb_present:
            return AUTO_SCHEMA_result('unrecognized', 'knowledge_without_query', 'Logical knowledge fields lack a query; this is not assumed to be graph-only support')
        if keys & (AUTO_SCHEMA_QUESTION_METADATA | controls):
            return AUTO_SCHEMA_result('unrecognized', 'incomplete_question', 'Question-shaped metadata or evidence exists without a query; retain')
        wrappers = [key for key in AUTO_SCHEMA_MODEL_WRAPPERS if key in value]
        if wrappers:
            children = []
            for key in wrappers:
                child_value = value[key]
                if key == 'graph' and isinstance(child_value, dict) and child_value and (len(child_value) <= AUTO_SCHEMA_MAX_OBJECTS) and (not set(child_value) & (query_keys | {'target'} | set(AUTO_SCHEMA_ENVELOPES) | AUTO_SCHEMA_QUESTION_METADATA | controls)) and all((isinstance(neighbors, (list, tuple, set)) and len(neighbors) <= AUTO_SCHEMA_MAX_OBJECTS and all((isinstance(node, (str, int, float)) for node in neighbors)) for neighbors in child_value.values())):
                    children.append(AUTO_SCHEMA_result('support', 'adjacency_model', 'Explicit graph wrapper contains an adjacency mapping'))
                else:
                    children.append(inspect(child_value, depth + 1, ancestors))
            if all((child['classification'] == 'support' for child in children)):
                return support('model_wrapper', 'All explicit model/graph wrappers contain query-free support structures')
            return AUTO_SCHEMA_result('unrecognized', 'ambiguous_model_wrapper', 'A model wrapper is malformed or may contain a query; retain')
        if {'variable_mapping', 'background', 'model_id'} <= keys:
            if isinstance(value['variable_mapping'], dict) and isinstance(value['background'], str):
                return support('model_blueprint', 'Query-free model blueprint has variable mapping, background text and a model identifier field')
        if AUTO_SCHEMA_graph_container(value):
            return support('graph_model', 'Explicit query-free nodes/edges/adjacency structure')
        equation_keys = {'equations', 'structural_equations'} | ({fields['equations']} if 'equations' in fields else set())
        present_equations = [key for key in equation_keys if key in value]
        if present_equations:
            if all((isinstance(value[key], (dict, list, tuple)) for key in present_equations)):
                return support('structural_model', 'Explicit query-free structural-equation containers')
            return AUTO_SCHEMA_result('unrecognized', 'malformed_model', 'Structural-equation fields have an unrecognized shape; retain')
        if value and all((isinstance(child, dict) for child in value.values())):
            children = []
            for child in value.values():
                if state['objects'] >= AUTO_SCHEMA_MAX_OBJECTS:
                    return AUTO_SCHEMA_result('unrecognized', 'inspection_limit', 'Support-collection inspection limit reached; retain')
                children.append(inspect(child, depth + 1, ancestors))
            if all((child['classification'] == 'support' for child in children)):
                return support('graph_collection', 'Every value in this nonempty mapping is positively identified query-free model support')
            if any((child['classification'] == 'problem' for child in children)):
                return AUTO_SCHEMA_result('unrecognized', 'problem_collection', 'Mapping contains reasoning problems; unpack or retain for a diagnostic')
        return AUTO_SCHEMA_result('unrecognized', 'unknown_structure', 'No complete problem or positively identified support-only schema; retain')
    try:
        return inspect(record)
    except (TypeError, ValueError, RecursionError):
        return AUTO_SCHEMA_result('unrecognized', 'inspection_error', 'Schema inspection could not safely classify the value; retain for a diagnostic')
import bz2
import csv
import gzip
import importlib
import inspect
import json
import lzma
import os
import re
import shutil
import subprocess
import sys
import tempfile
from collections import Counter
from collections.abc import Mapping
from pathlib import Path, PurePosixPath
from urllib.parse import unquote, urlsplit, urlunsplit
from urllib.request import urlopen

class LOAD_SourceError(ValueError):

    def __init__(self, message, audit):
        super().__init__(message)
        self.audit = audit
LOAD_QUERY_KEYS = {'query', 'question', 'statement', 'conclusion', 'hypothesis', 'goal', 'conclusion-FOL', 'conclusion_fol', 'query_expression'}
LOAD_ID_KEYS = {'id', 'question_id', 'problem_id', 'example_id', 'example-id', 'qid', 'query_id', 'record_id', 'uid'}
LOAD_REFERENCE_KEYS = {'gold', 'gold_label', 'label', 'labels', 'answer', 'answers', 'flag', 'prediction', 'predictions', 'predicted_label', 'predicted_labels', 'predicted_flag', 'reasoning', 'explanation', 'explanations', 'expected', 'expected_answer', 'expected_label', 'ground_truth', 'reference_answer'}
LOAD_ARRAY_KEYS = ('records', 'problems', 'examples', 'data', 'rows', 'instances')
LOAD_SHARED_INPUT_KEYS = {'premises', 'premises-FOL', 'premises_fol', 'context', 'facts', 'rules', 'clauses', 'knowledge_base', 'kb', 'given_info', 'equations', 'structural_equations', 'causal_model', 'constraints', 'assumptions', 'interventions', 'intervention', 'do', 'observations', 'evidence'}
LOAD_TEXT_EXTENSIONS = {'.json', '.jsonl', '.ndjson', '.csv', '.tsv'}
LOAD_OTHER_DATA_EXTENSIONS = {'.arrow', '.feather', '.avro', '.json5', '.xlsx', '.xls', '.pkl', '.pickle', '.zip', '.tar', '.h5', '.hdf5', '.db', '.sqlite'}

def LOAD_display(value):
    text = str(value)
    parsed = urlsplit(text)
    if parsed.scheme in {'http', 'https', 'hf'}:
        hostname = parsed.hostname or ''
        if parsed.port is not None:
            hostname += ':' + str(parsed.port)
        return urlunsplit((parsed.scheme, hostname, parsed.path, '', ''))
    return text

def LOAD_safe(value):
    if isinstance(value, dict):
        return {str(key): LOAD_safe(item) for key, item in value.items()}
    if isinstance(value, (list, tuple, set)):
        return [LOAD_safe(item) for item in value]
    if value is None or isinstance(value, (str, bool, int)):
        return value
    return str(value)

def LOAD_has_query(value, fields=None):
    keys = set(LOAD_QUERY_KEYS)
    if isinstance(fields, dict) and isinstance(fields.get('query'), str):
        keys.add(fields['query'])
    pending, seen = ([(value, 0)], 0)
    while pending:
        current, depth = pending.pop()
        seen += 1
        if seen > 10000 or depth > 16:
            return True
        if isinstance(current, Mapping):
            if any((key in current for key in keys)):
                return True
            pending.extend(((item, depth + 1) for key, item in current.items() if str(key).casefold() not in LOAD_REFERENCE_KEYS and isinstance(item, (Mapping, list, tuple))))
        elif isinstance(current, (list, tuple)):
            pending.extend(((item, depth + 1) for item in current if isinstance(item, (Mapping, list, tuple))))
    return False

def LOAD_describe(record, fields):
    function = globals().get('describe_reasoning_record')
    if not callable(function):
        raise RuntimeError('Load describe_reasoning_record before inspecting a reasoning source')
    description = function(record, fields)
    if not isinstance(description, dict) or description.get('classification') not in {'problem', 'support', 'unrecognized'}:
        raise ValueError('Record classifier returned an invalid classification')
    if description['classification'] == 'support' and LOAD_has_query(record, fields):
        return {'classification': 'unrecognized', 'schema': 'query_present_support_guard', 'reason': 'A query-shaped field is present; the row was retained conservatively.'}
    return description

def LOAD_json_rows(value, fields):
    if isinstance(value, list):
        return (value, {'container': 'array'})
    if not isinstance(value, dict):
        return ([value], {'container': 'single_unrecognized_value'})
    direct_query_keys = set(LOAD_QUERY_KEYS)
    if isinstance(fields, dict) and isinstance(fields.get('query'), str):
        direct_query_keys.add(fields['query'])
    description = LOAD_describe(value, fields)
    if any((key in value for key in direct_query_keys)) or description['classification'] == 'problem':
        return ([value], {'container': 'single_problem_record'})
    if any((isinstance(value.get(key), dict) and LOAD_has_query(value[key], fields) for key in ('problem', 'instance', 'input'))):
        return ([value], {'container': 'single_problem_envelope_including_incomplete_input'})
    arrays = [(key, value[key]) for key in LOAD_ARRAY_KEYS if isinstance(value.get(key), list)]
    if arrays:
        if any((items != arrays[0][1] for _, items in arrays[1:])):
            raise ValueError('Conflicting JSON record-array envelopes; choose an unambiguous records array')
        extra = [key for key in value if key not in {name for name, _ in arrays}]
        shared = [key for key in extra if key in LOAD_SHARED_INPUT_KEYS]
        if shared:
            raise ValueError('Shared semantic input fields occur outside the record array and cannot be discarded or guessed onto rows: ' + ', '.join(shared))
        if any((LOAD_has_query(value[key], fields) for key in extra if str(key).casefold() not in LOAD_REFERENCE_KEYS)):
            raise ValueError('A second nested question collection occurs outside the selected JSON envelope')
        return (arrays[0][1], {'container': 'explicit_array_envelope', 'envelope_keys': [key for key, _ in arrays], 'other_top_level_fields': extra})
    id_keys = set(LOAD_ID_KEYS)
    if isinstance(fields, dict) and isinstance(fields.get('id'), str):
        id_keys.add(fields['id'])
    if value and (not any((str(key).casefold() in LOAD_REFERENCE_KEYS for key in value))) and all((isinstance(item, dict) for item in value.values())):
        descriptions = [LOAD_describe(item, fields) for item in value.values()]
        has_problem = any((item['classification'] == 'problem' for item in descriptions))
        if has_problem and set(value) & LOAD_SHARED_INPUT_KEYS:
            raise ValueError('An ID-keyed collection also contains apparent shared semantic fields; export self-contained problem records')
        valid_collection = all((description['classification'] in {'problem', 'support'} or LOAD_has_query(item, fields) or bool(set(item) & id_keys) for item, description in zip(value.values(), descriptions)))
        if has_problem and valid_collection:
            rows, supplied = ([], 0)
            for key, item in value.items():
                row = dict(item)
                if not set(row) & id_keys:
                    row['id'] = str(key)
                    supplied += 1
                rows.append(row)
            return (rows, {'container': 'validated_id_keyed_record_collection', 'ids_from_outer_keys': supplied, 'existing_ids_preserved': len(rows) - supplied, 'outer_keys_preview': list(value)[:10]})
        if has_problem:
            raise ValueError('Ambiguous heterogeneous JSON object contains nested questions; export an explicit records array')
    if LOAD_has_query(value, fields) and description['classification'] != 'problem':
        raise ValueError('Unrecognized nested question container; export an explicit records array so no question is lost')
    return ([value], {'container': 'single_record_or_support_collection'})

def LOAD_file_type(locator):
    path = urlsplit(str(locator)).path if '://' in str(locator) else str(locator)
    suffixes = [suffix.lower() for suffix in PurePosixPath(path.replace('\\', '/')).suffixes]
    compression = suffixes[-1] if suffixes and suffixes[-1] in {'.gz', '.bz2', '.xz'} else None
    extension = suffixes[-2] if compression and len(suffixes) > 1 else suffixes[-1] if suffixes else ''
    return (extension, compression)

def LOAD_read_file(path, locator, datasets, fields):
    extension, compression = LOAD_file_type(locator)
    if extension == '.parquet':
        if compression:
            raise ValueError('Externally compressed parquet files are not supported by this inspector')
        if datasets is None:
            datasets, _ = LOAD_hf_dependencies(False, False)
        data = datasets.load_dataset('parquet', data_files={'train': str(path)}, split='train')
        return ([dict(row) for row in data], {'container': 'single_parquet_file', 'columns': list(data.column_names)})
    opener = {'.gz': gzip.open, '.bz2': bz2.open, '.xz': lzma.open}.get(compression, open)
    with opener(path, 'rt', encoding='utf-8-sig', newline='') as stream:
        if extension == '.json':
            return LOAD_json_rows(json.load(stream), fields)
        if extension in {'.jsonl', '.ndjson'}:
            rows = []
            for line_number, line in enumerate(stream, 1):
                if not line.strip():
                    continue
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as error:
                    raise ValueError(f'Malformed JSONL at line {line_number}; {len(rows)} earlier rows were not silently accepted as a complete file: {error}') from error
            return (rows, {'container': 'json_lines'})
        if extension in {'.csv', '.tsv'}:
            reader = csv.DictReader(stream, delimiter='\t' if extension == '.tsv' else ',')
            if reader.fieldnames and (len(set(reader.fieldnames)) != len(reader.fieldnames) or any((not name for name in reader.fieldnames))):
                raise ValueError('CSV column names are duplicated or empty; no column was overwritten')
            rows = list(reader)
            return (rows, {'container': 'delimited_rows', 'columns': reader.fieldnames or [], 'rows_with_extra_cells': sum((None in row for row in rows)), 'rows_with_missing_cells': sum((any((value is None for value in row.values())) for row in rows))})
    raise ValueError('Unsupported data format: ' + extension)

def LOAD_hf_dependencies(install, login):
    if not isinstance(install, bool) or not isinstance(login, bool):
        raise ValueError('HF installation/login controls must be Boolean')
    try:
        datasets = importlib.import_module('datasets')
        hub = importlib.import_module('huggingface_hub')
    except ImportError as error:
        if not install:
            raise ImportError('HF inspection needs datasets and huggingface_hub; set INSTALL_HF_PACKAGES=True or install them in this kernel') from error
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'datasets', 'huggingface_hub'])
        datasets = importlib.import_module('datasets')
        hub = importlib.import_module('huggingface_hub')
    if login:
        hub.login()
    return (datasets, hub)

def LOAD_splits(selection):
    if selection is None:
        return None
    function = globals().get('parse_split_selection')
    if callable(function):
        parts = function(selection)
    else:
        values = [selection] if isinstance(selection, str) else selection
        if not isinstance(values, (list, tuple)):
            raise ValueError('HF_SPLIT must be None, a whole split name, or a list/plus combination')
        parts = [part.strip() for value in values if isinstance(value, str) for part in value.split('+')]
        if len(values) != sum((isinstance(value, str) for value in values)):
            raise ValueError('HF_SPLIT entries must be strings')
    if not parts or any((not isinstance(part, str) or not part.strip() or any((mark in part for mark in '[]:')) for part in parts)):
        raise ValueError("Repository-file inspection supports whole split names, not slices. Use HF_SPLIT=None, a name, or names joined with '+'")
    if len(set(parts)) != len(parts):
        raise ValueError('HF_SPLIT repeats a split; no duplicate selection was silently removed')
    return parts

def LOAD_partition_files(manifest):
    entries = []
    for split, locators in manifest.items():
        families = []
        for locator in locators:
            base = PurePosixPath(urlsplit(str(locator)).path).name
            extension, compression = LOAD_file_type(base)
            stem = base
            if compression:
                stem = stem[:-len(compression)]
            if extension:
                stem = stem[:-len(extension)]
            match = re.fullmatch('(.+?)[-_]\\d{5}-of-\\d{5}(?:-[0-9a-fA-F]+)?', stem)
            families.append((match[1] if match else stem or base, bool(match)))
        distinct = {family for family, _ in families}
        for locator, (family, shard) in zip(locators, families):
            partition = str(split) if len(distinct) <= 1 else str(split) + ' / ' + family
            entries.append({'split': str(split), 'partition': partition, 'locator': str(locator), 'shard_family': family if shard else None})
    return entries

def LOAD_repo_file(locator, repository, requested_repository, revision, hub, temporary, datasets, revision_cache=None):
    text, parsed = (str(locator), urlsplit(str(locator)))
    filename, reference = (None, revision)
    candidate = Path(text)
    if candidate.is_absolute() and parsed.scheme not in {'hf', 'https', 'http'}:
        hub_constants = importlib.import_module('huggingface_hub.constants')
        cache = getattr(hub_constants, 'HF_HUB_CACHE', None)
        allowed = [Path(cache) / ('datasets--' + repository.replace('/', '--'))] if cache else []
        resolved = candidate.resolve()
        if not candidate.is_file() or not any((root and resolved.is_relative_to(Path(root).resolve()) for root in allowed)):
            raise ValueError("A remote manifest cannot read outside the selected repository's cache; provide that local file explicitly as DATASET")
        return (resolved, {'origin': str(candidate.absolute()), 'cached_manifest_path': True, 'cache_scope': 'selected repository only'})
    if parsed.scheme == 'hf':
        match = re.fullmatch('hf://datasets/([^/]+/[^/@]+)(?:@([^/]+))?/(.+)', text)
        if not match or unquote(match[1]) not in {repository, requested_repository}:
            raise ValueError('HF manifest file points outside the selected repository')
        reference, filename = (unquote(match[2]) if match[2] else revision, unquote(match[3]))
    elif parsed.scheme in {'https', 'http'}:
        prefix = '/datasets/'
        path = parsed.path
        if parsed.hostname in {'huggingface.co', 'www.huggingface.co'} and path.startswith(prefix):
            rest = path[len(prefix):]
            match = re.fullmatch('([^/]+/[^/]+)/resolve/([^/]+)/(.+)', rest)
            if match and unquote(match[1]) in {repository, requested_repository}:
                reference, filename = (unquote(match[2]), unquote(match[3]))
            else:
                raise ValueError('Unrecognized HF manifest file URL')
        else:
            target = Path(temporary) / (str(len(list(Path(temporary).iterdir()))) + '_' + (PurePosixPath(parsed.path).name or 'data'))
            with urlopen(text, timeout=120) as response, target.open('wb') as stream:
                shutil.copyfileobj(response, stream)
            return (target, {'origin': LOAD_display(text), 'external_manifest_url': True, 'revision_pins_external_content': False})
    elif parsed.scheme:
        raise ValueError('Unsupported manifest locator scheme: ' + parsed.scheme)
    else:
        filename = text.replace('\\', '/')
    parts = PurePosixPath(filename).parts
    if not filename or filename.startswith('/') or '..' in parts:
        raise ValueError('Unsafe repository-relative data path')
    file_revision = revision
    if reference != revision:
        cache = {} if revision_cache is None else revision_cache
        if reference not in cache:
            cache[reference] = getattr(hub.HfApi().dataset_info(repo_id=repository, revision=reference), 'sha', None)
        file_revision = cache[reference]
        if not isinstance(file_revision, str) or not file_revision:
            raise ValueError("Could not pin the file manifest's explicit data revision")
    path = hub.hf_hub_download(repo_id=repository, filename=filename, repo_type='dataset', revision=file_revision)
    return (Path(path), {'origin': f'hf://datasets/{repository}@{file_revision}/{filename}', 'repository_path': filename, 'manifest_reference': reference, 'file_revision': file_revision})

def LOAD_classify_rows(rows, file_audit, fields, exclude_support):
    retained, schemas = ([], Counter())
    file_audit.update(rows_total=len(rows), problem_rows=0, support_rows=0, unrecognized_rows=0, rows_included=0, rows_excluded=0, classification_errors=0, samples=[])
    for index, record in enumerate(rows):
        try:
            description = LOAD_describe(record, fields)
        except Exception as error:
            file_audit['classification_errors'] += 1
            description = {'classification': 'unrecognized', 'schema': 'inspection_error', 'reason': type(error).__name__ + ': ' + str(error)[:300]}
        classification = description['classification']
        file_audit[classification + '_rows'] += 1
        schemas[str(description.get('schema', 'unknown'))] += 1
        if classification == 'support' and exclude_support:
            file_audit['rows_excluded'] += 1
            continue
        if classification == 'unrecognized' and len(file_audit['samples']) < 5:
            file_audit['samples'].append({'row': index, 'classification': classification, 'schema': description.get('schema'), 'reason': description.get('reason'), 'keys': list(record)[:30] if isinstance(record, dict) else None, 'record_type': type(record).__name__})
        retained.append(dict(record) if isinstance(record, Mapping) else {'unrecognized_record': record})
    file_audit['rows_included'] = len(retained)
    file_audit['schemas'] = dict(schemas)
    file_audit['status'] = 'empty' if not rows else 'support_only' if not retained else 'included'
    return retained

def load_reasoning_source(kind, location, hf_split=None, hf_config=None, hf_revision=None, install_hf_packages=False, hf_login=False, column_map=None):
    audit = {'loader': 'generic_file_inspection', 'kind': kind, 'location': LOAD_display(location), 'requested_config': hf_config, 'requested_revision': hf_revision, 'requested_split': hf_split, 'files': [], 'warnings': [], 'row_origins': [], 'complete': False, 'support_filter_policy': 'HF: positive query-free support only; local: all rows retained', 'inference_called': False, 'repository_names_select_engines': False}
    records, groups, order = ([], [], [])
    try:
        if not callable(globals().get('describe_reasoning_record')):
            raise RuntimeError('The content-schema inspection helper is missing')
        if column_map is not None and (not isinstance(column_map, dict)):
            raise ValueError('column_map must be a dictionary or None')
        if isinstance(column_map, dict) and any((role != 'id' and isinstance(column, str) and (column.casefold() in LOAD_REFERENCE_KEYS) for role, column in column_map.items())):
            raise ValueError('Reference-label or explanation columns cannot define the problem schema')
        if kind == 'file':
            path = Path(location).expanduser().resolve()
            if not path.is_file():
                raise FileNotFoundError('Dataset file was not found: ' + str(path))
            extension, _ = LOAD_file_type(path)
            if extension not in LOAD_TEXT_EXTENSIONS | {'.parquet'}:
                raise ValueError('Unsupported local data format: ' + extension)
            datasets = None
            if extension == '.parquet':
                datasets, _ = LOAD_hf_dependencies(install_hf_packages, False)
            entry = {'file': str(path), 'origin': str(path), 'scope': 'explicit local file'}
            audit['files'].append(entry)
            rows, details = LOAD_read_file(path, path, datasets, column_map)
            entry.update(details)
            retained = LOAD_classify_rows(rows, entry, column_map, False)
            for row in retained:
                dataset = row.get('dataset')
                dataset = dataset.strip() if isinstance(dataset, str) and dataset.strip() else path.stem
                split = row.get('split')
                split = str(split).strip() if isinstance(split, (str, int)) and (not isinstance(split, bool)) else 'all records'
                group = (dataset, split or 'all records')
                records.append(row)
                groups.append(group)
                audit['row_origins'].append(str(path))
                if group not in order:
                    order.append(group)
            audit['scope'] = 'explicit local file; no support-row exclusion'
        elif kind == 'huggingface':
            if not isinstance(location, str) or not location.strip():
                raise ValueError('Supply a Hugging Face repository')
            selections = LOAD_splits(hf_split)
            datasets, hub = LOAD_hf_dependencies(install_hf_packages, hf_login)
            api = hub.HfApi()
            info = api.dataset_info(repo_id=location, revision=hf_revision)
            revision = getattr(info, 'sha', None)
            repository = getattr(info, 'id', None) or location
            if not isinstance(revision, str) or not revision:
                raise ValueError('Could not pin an actual repository revision before loading files')
            audit.update(repository=repository, actual_revision=revision)
            arguments = {'revision': revision}
            if hf_config is not None:
                arguments['name'] = hf_config
            if 'trust_remote_code' in inspect.signature(datasets.load_dataset_builder).parameters:
                arguments['trust_remote_code'] = False
            manifest, builder_error = ({}, None)
            try:
                builder = datasets.load_dataset_builder(location, **arguments)
                data_files = getattr(getattr(builder, 'config', None), 'data_files', None)
                if isinstance(data_files, Mapping):
                    for split, entries in data_files.items():
                        manifest[str(split)] = [str(entries)] if isinstance(entries, (str, os.PathLike)) else [str(entry) for entry in entries]
                audit['resolved_config'] = getattr(getattr(builder, 'config', None), 'name', hf_config)
            except Exception as error:
                manifest = {}
                builder_error = type(error).__name__ + ': ' + str(error)[:1000]
            if not manifest:
                if hf_config is not None:
                    raise ValueError('Requested HF_CONFIG has no inspectable builder manifest; it was not silently replaced with all repository files. ' + str(builder_error or ''))
                if selections is not None:
                    raise ValueError('HF_SPLIT cannot be verified without a builder file manifest; use HF_SPLIT=None or export the desired split locally')
                listed = api.list_repo_files(repo_id=repository, repo_type='dataset', revision=revision)
                manifest = {'repository files': list(listed)}
                audit['manifest_source'] = 'pinned repository file listing'
                audit['warnings'].append('Builder manifest unavailable; every listed repository file is audited separately.')
                if builder_error:
                    audit['builder_inspection_error'] = builder_error
            else:
                audit['manifest_source'] = 'builder.config.data_files (no Arrow build)'
                audit['available_splits'] = list(manifest)
                if selections is not None:
                    absent = [split for split in selections if split not in manifest]
                    if absent:
                        raise ValueError('Requested split(s) absent from manifest: ' + ', '.join(absent) + '; available: ' + ', '.join(manifest))
                    audit['unselected_splits'] = [split for split in manifest if split not in selections]
                    manifest = {split: manifest[split] for split in selections}
            audit['manifest'] = {split: [LOAD_display(path) for path in paths] for split, paths in manifest.items()}
            dataset_name = location if hf_config is None else f'{location} ({hf_config})'
            failures = []
            revision_cache = {revision: revision}
            revision_cache['main' if hf_revision is None else str(hf_revision)] = revision
            with tempfile.TemporaryDirectory(prefix='reasoning_source_') as temporary:
                for specification in LOAD_partition_files(manifest):
                    locator = specification['locator']
                    entry = {key: value for key, value in specification.items() if key != 'locator'}
                    entry['file'] = LOAD_display(locator)
                    audit['files'].append(entry)
                    extension, _ = LOAD_file_type(locator)
                    if extension not in LOAD_TEXT_EXTENSIONS | {'.parquet'}:
                        entry.update(status='not_inspected_extension', rows_included=0, rows_excluded=0, reason='This extension is not an inspectable JSON/JSONL/CSV/TSV/parquet data file.')
                        if audit['manifest_source'].startswith('builder') or extension in LOAD_OTHER_DATA_EXTENSIONS:
                            failures.append(entry['file'] + ': selected data format is unsupported: ' + (extension or 'no extension'))
                        else:
                            audit['warnings'].append('Not inspected as reasoning rows: ' + entry['file'])
                        continue
                    try:
                        path, provenance = LOAD_repo_file(locator, repository, location, revision, hub, temporary, datasets, revision_cache)
                        entry.update(provenance)
                        rows, details = LOAD_read_file(path, locator, datasets, column_map)
                        entry.update(details)
                        retained = LOAD_classify_rows(rows, entry, column_map, True)
                        group = (dataset_name, specification['partition'])
                        records.extend(retained)
                        groups.extend([group] * len(retained))
                        audit['row_origins'].extend([entry['origin']] * len(retained))
                        if retained and group not in order:
                            order.append(group)
                    except Exception as error:
                        entry.update(status='error', error=type(error).__name__ + ': ' + str(error).replace(locator, LOAD_display(locator))[:1000])
                        failures.append(entry['file'] + ': ' + entry['error'])
            if failures:
                raise ValueError('Some selected candidate data files could not be inspected. No partial source was silently accepted: ' + ' | '.join(failures[:10]))
            audit['scope'] = 'selected configured manifest files' if audit['manifest_source'].startswith('builder') else 'all pinned repository files'
        else:
            raise ValueError('kind must be file or huggingface')
        audit['counts'] = {'included': len(records), 'problem': sum((file.get('problem_rows', 0) for file in audit['files'])), 'support': sum((file.get('support_rows', 0) for file in audit['files'])), 'unrecognized': sum((file.get('unrecognized_rows', 0) for file in audit['files'])), 'excluded_support': sum((file.get('rows_excluded', 0) for file in audit['files']))}
        if not records:
            raise ValueError('No problem or unrecognized rows remain after inspection; see the support/empty-file audit')
        audit['complete'] = True
        audit['group_order'] = [list(group) for group in order]
        return (records, groups, order, LOAD_safe(audit))
    except LOAD_SourceError:
        raise
    except Exception as error:
        audit['complete'] = False
        audit['error'] = type(error).__name__ + ': ' + str(error)
        audit['retained_before_failure'] = len(records)
        raise LOAD_SourceError(str(error), LOAD_safe(audit)) from error

def prepare_group_identifiers(records, groups, column_map=None, origins=None):
    if len(records) != len(groups):
        raise ValueError('Input rows and partition provenance are misaligned')
    if origins is not None and len(origins) != len(records):
        raise ValueError('Input rows and file origins are misaligned')
    origins = [None] * len(records) if origins is None else origins
    partitions = [(group, origin) for group, origin in zip(groups, origins)]
    if column_map is not None and (not isinstance(column_map, dict)):
        raise ValueError('COLUMN_MAP must be a dictionary')
    if any((not isinstance(column, str) or not column.strip() for column in (column_map or {}).values())):
        raise ValueError('COLUMN_MAP values must name nonempty columns')
    identities, generated, within, owners = ([], [], set(), {})
    for number, (record, partition) in enumerate(zip(records, partitions), 1):
        identifier, was_generated = AUTO_batch_identifier(record, column_map, number, True)
        if isinstance(identifier, bool) or not isinstance(identifier, (str, int)) or (not str(identifier).strip()):
            raise ValueError(f'Row {number} needs a nonempty string/integer ID')
        identities.append(identifier)
        generated.append(was_generated)
    supplied = {str(identifier) for identifier, made in zip(identities, generated) if not made}
    assigned, generated_adjustments = (set(), 0)
    for index, (identifier, made) in enumerate(zip(identities, generated)):
        if made:
            base, suffix = (str(identifier), 0)
            while str(identifier) in supplied or str(identifier) in assigned:
                suffix += 1
                identifier = f'{base}-generated-{suffix}'
            generated_adjustments += str(identifier) != base
            identities[index] = identifier
        assigned.add(str(identifier))
    for identifier, partition in zip(identities, partitions):
        key = str(identifier)
        if (partition, key) in within:
            raise ValueError(f'Duplicate ID {key!r} inside input partition {partition!r}')
        within.add((partition, key))
        owners.setdefault(key, set()).add(partition)
    collisions = {key for key, source_groups in owners.items() if len(source_groups) > 1}
    provenance = [{'original_id': None if made else identifier, 'id_generated': made, 'id_namespaced': str(identifier) in collisions, 'input_origin': origin} for identifier, made, origin in zip(identities, generated, origins)]
    notes = {'namespaced_rows': sum((item['id_namespaced'] for item in provenance)), 'generated_rows': sum(generated), 'transport_id_column': None, 'generated_id_adjustments': generated_adjustments, 'policy': 'Unique supplied IDs preserved; only cross-partition collisions are namespaced.'}
    if not collisions and (not generated_adjustments):
        return (records, column_map, provenance, notes)
    candidates = [record for record in records]
    candidates += [record[key] for record in records for key in ('problem', 'instance', 'input') if isinstance(record.get(key), dict)]
    column, suffix = ('__submission_row_id__', 0)
    mapped_columns = set((column_map or {}).values())
    while column in mapped_columns or any((column in item for item in candidates)):
        suffix += 1
        column = f'__submission_row_id_{suffix}__'
    group_numbers = {partition: number for number, partition in enumerate(dict.fromkeys(partitions), 1)}
    reserved, used, normalized = (set(map(str, identities)), set(), [])
    for record, partition, identifier, detail in zip(records, partitions, identities, provenance):
        output_id = identifier
        if detail['id_namespaced']:
            base = f'partition-{group_numbers[partition]}::{identifier}'
            output_id, index = (base, 0)
            while str(output_id) in reserved or str(output_id) in used:
                index += 1
                output_id = f'partition-{group_numbers[partition]}-{index}::{identifier}'
        used.add(str(output_id))
        copied = dict(record)
        copied[column] = output_id
        for key in ('problem', 'instance', 'input'):
            if isinstance(record.get(key), dict):
                copied[key] = dict(record[key], **{column: output_id})
        normalized.append(copied)
    fields = dict(column_map or {})
    fields['id'] = column
    notes['transport_id_column'] = column
    return (normalized, fields, provenance, notes)
import os
import re
import csv
import json
import importlib
import subprocess
import sys
from collections import Counter
from collections.abc import Mapping
from pathlib import Path
from urllib.parse import unquote, urlsplit

def identify_dataset_source(dataset):
    if isinstance(dataset, os.PathLike):
        path = Path(dataset).expanduser()
        if not path.is_file():
            raise FileNotFoundError(f'Dataset file was not found: {path}')
        return ('file', str(path))
    if not isinstance(dataset, str) or not dataset.strip():
        raise ValueError('Enter your file path or Hugging Face repository in DATASET')
    text = dataset.strip()
    if text.startswith('hf://'):
        repository = text[5:].strip('/')
        if repository.startswith('datasets/') and len(repository.split('/')) == 3:
            repository = repository[len('datasets/'):]
    elif text.startswith(('https://', 'http://')):
        address = urlsplit(text)
        if address.scheme != 'https' or address.hostname not in {'huggingface.co', 'www.huggingface.co'}:
            raise ValueError('Use a local dataset file or an HTTPS Hugging Face dataset repository URL')
        parts = unquote(address.path).strip('/').split('/')
        if not parts or parts[0] != 'datasets' or len(parts) not in (2, 3):
            raise ValueError('Use the dataset repository URL; select its split using HF_SPLIT')
        repository = '/'.join(parts[1:])
    else:
        path = Path(text).expanduser()
        if path.exists():
            if not path.is_file():
                raise ValueError('DATASET must identify a file, not a folder')
            return ('file', str(path))
        looks_local = path.suffix.lower() in {'.csv', '.tsv', '.json', '.jsonl', '.ndjson', '.parquet', '.gz', '.bz2', '.xz'} or path.is_absolute() or bool(re.match('^[A-Za-z]:', text)) or text.startswith(('./', '../', '.\\', '..\\', '~/', '~\\', '\\', '/'))
        if looks_local:
            raise FileNotFoundError(f'Dataset file was not found: {path}')
        repository = text
    if not re.fullmatch('[A-Za-z0-9_.-]+(?:/[A-Za-z0-9_.-]+)?', repository):
        raise ValueError('Supply a file path or a Hugging Face repository such as organization/dataset')
    return ('huggingface', repository)

def parse_split_selection(selection):
    if selection is None:
        return None
    selections = [selection] if isinstance(selection, str) else selection
    if not isinstance(selections, (list, tuple)) or not selections:
        raise ValueError('HF_SPLIT must be None, a split expression, or a list of split expressions')
    parts = []
    for item in selections:
        if not isinstance(item, str) or not item.strip():
            raise ValueError('Each requested split must be a nonempty string')
        start, depth = (0, 0)
        for index, character in enumerate(item):
            if character == '[':
                depth += 1
            elif character == ']':
                depth -= 1
                if depth < 0:
                    raise ValueError('Unbalanced brackets in HF_SPLIT')
            elif character == '+' and depth == 0:
                part = item[start:index].strip()
                if not part:
                    raise ValueError('Empty component in HF_SPLIT')
                parts.append(part)
                start = index + 1
        part = item[start:].strip()
        if depth or not part:
            raise ValueError('Incomplete split expression in HF_SPLIT')
        parts.append(part)
    return parts

def load_dataset_records(kind, location, hf_split=None, hf_config=None, hf_revision=None, install_hf_packages=False, hf_login=False, column_map=None):
    return load_reasoning_source(kind, location, hf_split, hf_config, hf_revision, install_hf_packages, hf_login, column_map)

def print_source_audit(audit):
    print('Source inspection:')
    files = audit.get('files', []) if isinstance(audit, dict) else []
    for entry in files[:20]:
        print(' -', json.dumps(entry, ensure_ascii=False))
    if isinstance(audit, dict) and 'counts' in audit:
        print('Source totals:', json.dumps(audit['counts'], ensure_ascii=False))
    if isinstance(audit, dict):
        for note in audit.get('warnings', [])[:5]:
            print('Source note:', note)
    if len(files) > 20:
        print(f' - {len(files) - 20} more file entries are retained in the full source audit.')

def select_reference_column(records, requested='auto'):
    notes = {'policy': 'Binary entailment: Uncertain references map to False', 'status': 'disabled'}
    if requested is None:
        return (None, None, notes)
    if not isinstance(requested, str) or not requested.strip():
        raise ValueError("LABEL_COLUMN must be a column name, 'auto', or None")
    if requested.strip().lower() != 'auto':
        if any((requested not in row for row in records)):
            raise ValueError(f'Prediction CSVs were written, but a row lacks reference column {requested!r}')
        values = [normalize_flag(row[requested], allow_uncertain=True) for row in records]
        notes.update(status='explicit', column=requested, uncertain_count=sum((str(row[requested]).strip().lower() == 'uncertain' for row in records)))
        return (requested, values, notes)
    valid = []
    for name in ('gold', 'gold_label', 'label', 'labels', 'flag', 'answer', 'answers', 'expected', 'expected_answer', 'expected_label', 'ground_truth', 'reference_answer'):
        if not all((name in row for row in records)):
            continue
        try:
            values = [normalize_flag(row[name], allow_uncertain=True) for row in records]
        except (TypeError, ValueError):
            continue
        valid.append((name, values))
    if not valid:
        notes.update(status='no_complete_binary_reference_column')
        return (None, None, notes)
    if any((values != valid[0][1] for _, values in valid[1:])):
        notes.update(status='conflicting_reference_columns', candidates=[name for name, _ in valid])
        return (None, None, notes)
    name, values = valid[0]
    notes.update(status='detected', column=name, equivalent_columns=[key for key, _ in valid], uncertain_count=sum((str(row[name]).strip().lower() == 'uncertain' for row in records)))
    return (name, values, notes)

def save_submission(predictions, output_csv='submission.csv'):
    path = Path(output_csv)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('w', encoding='utf-8', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=['id', 'flag'])
        writer.writeheader()
        writer.writerows(({'id': result['id'], 'flag': result['flag']} for result in predictions))
    print('Prediction CSV:', path.resolve())

def collect_split_metrics(predictions, references, groups, group_order):
    if len(predictions) != len(groups) or (references is not None and len(references) != len(predictions)):
        raise ValueError('Prediction, provenance and reference rows are misaligned')
    buckets = {group: [] for group in group_order}
    for index, group in enumerate(groups):
        buckets.setdefault(group, []).append(index)
    metrics = []
    for (dataset_name, split), indices in buckets.items():
        count = len(indices)
        correct = None if references is None else sum((predictions[index]['flag'] == references[index] for index in indices))
        metrics.append({'dataset': dataset_name, 'split': split, 'questions': count, 'correct': correct, 'accuracy': correct / count if correct is not None and count else None, 'resolved': sum((bool(predictions[index]['resolved']) for index in indices)), 'inference_seconds': sum((predictions[index]['seconds'] for index in indices))})
    if sum((item['questions'] for item in metrics)) != len(predictions):
        raise ValueError('Split groups do not cover the complete prediction batch')
    return (metrics, buckets)

def save_split_submissions(predictions, buckets, output_csv='submission.csv', input_path=None):
    combined = Path(output_csv)
    populated = [(group, indices) for group, indices in buckets.items() if indices]
    planned = [(combined, None, None)]
    if len(populated) > 1:
        folder = combined.parent / (combined.stem + '_splits')
        for number, ((dataset_name, split), indices) in enumerate(populated, 1):
            safe_name = re.sub('[^A-Za-z0-9_.-]+', '_', dataset_name).strip('._')[:70] or 'dataset'
            safe_split = re.sub('[^A-Za-z0-9_.-]+', '_', split).strip('._')[:70] or 'split'
            filename = f'{number:02d}_{safe_name}__{safe_split}.csv'
            planned.append((folder / filename, (dataset_name, split), indices))
    resolved_paths = [os.path.normcase(str(path.resolve())) for path, _, _ in planned]
    if len(set(resolved_paths)) != len(resolved_paths):
        raise ValueError('Prediction output paths conflict')
    if input_path is not None and os.path.normcase(str(Path(input_path).resolve())) in resolved_paths:
        raise ValueError('A prediction output would overwrite the input dataset')
    written = []
    for path, group, indices in planned:
        rows = predictions if indices is None else [predictions[index] for index in indices]
        save_submission(rows, path)
        written.append({'path': str(path.resolve()), 'dataset': group[0] if group else None, 'split': group[1] if group else None, 'questions': len(rows), 'combined': group is None})
    return written

def run_dataset(dataset, output_csv='submission.csv', label_column='auto', column_map=None, hf_split=None, hf_config=None, hf_revision=None, install_hf_packages=False, hf_login=False, timeout=2.0, progress_every=25):
    kind, location = identify_dataset_source(dataset)
    if kind == 'file' and Path(location).resolve() == Path(output_csv).resolve():
        raise ValueError('Input dataset and output CSV must use different paths')
    try:
        records, groups, group_order, source_audit = load_dataset_records(kind, location, hf_split, hf_config, hf_revision, install_hf_packages, hf_login, column_map)
    except Exception as error:
        if isinstance(getattr(error, 'audit', None), dict):
            print_source_audit(error.audit)
        raise
    print_source_audit(source_audit)
    prepared, fields, id_provenance, id_notes = prepare_group_identifiers(records, groups, column_map, source_audit.get('row_origins'))
    if id_notes['namespaced_rows']:
        print(f"Namespaced {id_notes['namespaced_rows']} IDs reused across independent partitions; original IDs are preserved in results.")
    if id_notes['transport_id_column'] is not None and id_notes['generated_rows']:
        print(f"Generated {id_notes['generated_rows']} nonconflicting IDs for rows without a supplied ID.")
    predictions = run_records(prepared, mode='auto', semantics='classical', timeout=timeout, progress_every=progress_every, fields=fields)
    for result, provenance in zip(predictions, id_provenance):
        result.update(provenance)
    summary = {'questions': len(predictions), 'source_audit': source_audit, 'id_normalization': id_notes, 'resolved': sum((bool(result['resolved']) for result in predictions)), 'inference_seconds': sum((result['seconds'] for result in predictions)), 'engines': dict(Counter((result['engine'] for result in predictions))), 'unresolved_categories': dict(Counter((result.get('category', 'unsupported') for result in predictions if not result['resolved'])))}
    _, buckets = collect_split_metrics(predictions, None, groups, group_order)
    summary['prediction_files'] = save_split_submissions(predictions, buckets, output_csv, location if kind == 'file' else None)
    chosen_reference, references, reference_notes = select_reference_column(records, label_column)
    summary['reference_column'], summary['reference_handling'] = (chosen_reference, reference_notes)
    if references is not None:
        summary.update(score_results(predictions, references))
        print('Reference-answer column:', chosen_reference)
        if reference_notes.get('uncertain_count'):
            print(f"Mapped {reference_notes['uncertain_count']} Uncertain references to False for binary entailment.")
        print(f"Overall accuracy: {100 * summary['accuracy']:.2f}% ({summary['correct']}/{summary['questions']})")
    else:
        print('Accuracy unavailable:', reference_notes['status'], '(predictions are still saved).')
    per_split, _ = collect_split_metrics(predictions, references, groups, group_order)
    summary['per_split'] = per_split
    for result, (dataset_name, split) in zip(predictions, groups):
        result['input_dataset'], result['input_split'] = (dataset_name, split)
    print('\nDataset | split | questions | correct | accuracy | resolved | inference seconds')
    for item in per_split:
        accuracy = 'N/A' if item['accuracy'] is None else f"{100 * item['accuracy']:.2f}%"
        correct = 'N/A' if item['correct'] is None else str(item['correct'])
        print(f"{item['dataset']} | {item['split']} | {item['questions']} | {correct} | {accuracy} | {item['resolved']}/{item['questions']} | {item['inference_seconds']:.6f}")
    print(f"Resolved: {summary['resolved']}/{summary['questions']}")
    print(f"Inference time: {summary['inference_seconds']:.6f} seconds")
    print('Reasoning engines:', summary['engines'])
    print('Unresolved categories:', summary['unresolved_categories'])
    return (predictions, summary)


In [11]:
def show_trace(result):
    print('ID:', result.get('id'))
    print('Prediction:', result['flag'], '| status:', result['status'], '| resolved:', result['resolved'])
    print('Reason:', result['reason'])
    for step in result.get('trace', []):
        print(' -', step)


In [12]:
import os
import re
from collections import Counter
from collections.abc import Mapping
from pathlib import Path

def sequence_slot_is_empty(entry):
    source = entry.get('dataset') if isinstance(entry, Mapping) and 'dataset' in entry else entry
    return source is None or (isinstance(source, str) and (not source.strip()))

def prepare_dataset_sequence(datasets, output_csv, defaults):
    if not isinstance(datasets, (list, tuple)):
        raise ValueError('DATASETS must be a list of dataset slots')
    if not isinstance(output_csv, (str, os.PathLike)) or not str(output_csv).strip():
        raise ValueError('OUTPUT_CSV must be a nonempty file path')
    combined = Path(output_csv).expanduser()
    if combined.exists() and (not combined.is_file()):
        raise ValueError('OUTPUT_CSV must name a file')
    supported = set(defaults) | {'dataset', 'name'}
    plan, stopped_at = ([], None)
    for number, entry in enumerate(datasets, 1):
        if sequence_slot_is_empty(entry):
            stopped_at = number
            if any((not sequence_slot_is_empty(later) for later in datasets[number:])):
                print(f'Slot {number} is empty; later filled slots will not run. Fill slots consecutively.')
            break
        details = dict(entry) if isinstance(entry, Mapping) else {'dataset': entry}
        unknown = set(details) - supported
        if unknown:
            raise ValueError(f'Unsupported options in dataset slot {number}: {sorted(unknown)}')
        if 'dataset' not in details:
            raise ValueError(f'Dataset slot {number} needs a dataset entry')
        kind, location = identify_dataset_source(details['dataset'])
        name = details.get('name', location if kind == 'huggingface' else Path(location).name)
        if not isinstance(name, str) or not name.strip():
            raise ValueError(f'Dataset slot {number} needs a nonempty display name')
        options = dict(defaults)
        options.update({key: details[key] for key in defaults if key in details})
        plan.append({'slot': number, 'dataset': details['dataset'], 'name': name.strip(), 'kind': kind, 'location': location, 'options': options})
    reserved_folders = []
    for item in plan:
        if len(plan) == 1:
            item['output_csv'] = combined
            reserved_folders.append(combined.parent / (combined.stem + '_splits'))
        else:
            slug = re.sub('[^A-Za-z0-9_.-]+', '_', item['name']).strip('._')[:70].strip('._') or 'dataset'
            folder = combined.parent / (combined.stem + '_datasets') / f"{item['slot']:02d}_{slug}"
            item['output_csv'] = folder / 'submission.csv'
            reserved_folders.append(folder)
    output_files = [combined] + ([item['output_csv'] for item in plan] if len(plan) > 1 else [])
    output_paths = [os.path.normcase(str(path.resolve())) for path in output_files]
    if len(set(output_paths)) != len(output_paths):
        raise ValueError('Dataset output paths overlap; change OUTPUT_CSV or the output folders')
    normalized_outputs = set(output_paths)
    normalized_folders = [os.path.normcase(str(path.resolve())) for path in reserved_folders]
    for number, entry in enumerate(datasets, 1):
        if sequence_slot_is_empty(entry):
            continue
        configured = entry.get('dataset') if isinstance(entry, Mapping) else entry
        try:
            kind, location = identify_dataset_source(configured)
        except (TypeError, ValueError, OSError):
            continue
        if kind != 'file':
            continue
        source = Path(location).resolve()
        normalized_source = os.path.normcase(str(source))
        source_parents = {os.path.normcase(str(parent)) for parent in source.parents}
        if normalized_source in normalized_outputs or any((folder == normalized_source or folder in source_parents for folder in normalized_folders)):
            raise ValueError(f'A prediction output could overwrite local input in slot {number}; change OUTPUT_CSV')
    return (plan, combined, stopped_at)

def combine_dataset_predictions(dataset_runs):
    owners, reserved = ({}, set())
    for run in dataset_runs:
        within = set()
        for row in run['results']:
            identifier = row.get('id')
            if isinstance(identifier, bool) or not isinstance(identifier, (str, int)) or (not str(identifier).strip()):
                raise ValueError(f"Invalid prediction ID in dataset slot {run['slot']}")
            key = str(identifier)
            if key in within:
                raise ValueError(f"Duplicate prediction ID {key!r} inside dataset slot {run['slot']}")
            within.add(key)
            owners.setdefault(key, set()).add(run['slot'])
            reserved.add(key)
    reused = {key for key, slots in owners.items() if len(slots) > 1}
    combined, used, namespaced = ([], set(), 0)
    for run in dataset_runs:
        for row in run['results']:
            source_id = row['id']
            key = str(source_id)
            output_id = key
            if key in reused:
                output_id = f"dataset-{run['slot']}::{key}"
                suffix = 0
                while output_id in reserved or output_id in used:
                    suffix += 1
                    output_id = f"dataset-{run['slot']}-{suffix}::{key}"
                namespaced += 1
            if output_id in used:
                raise ValueError('Combined prediction IDs must be unique')
            used.add(output_id)
            copied = dict(row)
            copied.update(id=output_id, dataset_slot=run['slot'], dataset_name=run['name'], dataset_submission_id=source_id, combined_id_namespaced=key in reused)
            combined.append(copied)
    return (combined, namespaced)

def summarize_dataset_sequence(dataset_runs, predictions, stopped_at):
    rows, split_rows = ([], [])
    scored_questions, correct = (0, 0)
    engines, unresolved_categories = (Counter(), Counter())
    for run in dataset_runs:
        item = run['summary']
        count = len(run['results'])
        if item['questions'] != count:
            raise ValueError('Dataset summary and prediction counts disagree')
        scored = item.get('accuracy') is not None and item.get('correct') is not None
        if scored:
            if not isinstance(item['correct'], int) or not 0 <= item['correct'] <= count:
                raise ValueError('Invalid correct-answer count in dataset summary')
            correct += item['correct']
            scored_questions += count
        rows.append({'slot': run['slot'], 'name': run['name'], 'source': str(run['dataset']), 'questions': count, 'scored_questions': count if scored else 0, 'correct': item.get('correct') if scored else None, 'accuracy': item.get('accuracy') if scored else None, 'resolved': item['resolved'], 'inference_seconds': item['inference_seconds'], 'reference_column': item.get('reference_column'), 'reference_handling': item.get('reference_handling'), 'prediction_files': item['prediction_files']})
        for split in item['per_split']:
            split_rows.append(dict(split, dataset_slot=run['slot'], dataset_name=run['name']))
        engines.update(item['engines'])
        unresolved_categories.update(item['unresolved_categories'])
    total = sum((row['questions'] for row in rows))
    if total != len(predictions):
        raise ValueError('Combined prediction and dataset counts disagree')
    resolved = sum((row['resolved'] for row in rows))
    return {'status': 'complete' if dataset_runs else 'no_inputs', 'datasets_completed': len(rows), 'stopped_at_empty_slot': stopped_at, 'questions': total, 'scored_questions': scored_questions, 'unscored_questions': total - scored_questions, 'scored_datasets': sum((row['scored_questions'] > 0 for row in rows)), 'correct': correct if scored_questions else None, 'accuracy': correct / scored_questions if scored_questions else None, 'resolved': resolved, 'unresolved': total - resolved, 'inference_seconds': sum((row['inference_seconds'] for row in rows)), 'engines': dict(engines), 'unresolved_categories': dict(unresolved_categories), 'per_dataset': rows, 'per_split': split_rows}

def print_dataset_sequence_summary(summary):
    print('\nFINAL COMBINED RESULTS')
    print('Slot | dataset | questions | correct | accuracy | resolved | inference seconds')
    for item in summary['per_dataset']:
        accuracy = 'N/A' if item['accuracy'] is None else f"{100 * item['accuracy']:.2f}%"
        correct = 'N/A' if item['correct'] is None else str(item['correct'])
        print(f"{item['slot']} | {item['name']} | {item['questions']} | {correct} | {accuracy} | {item['resolved']}/{item['questions']} | {item['inference_seconds']:.6f}")
    print('Datasets completed:', summary['datasets_completed'])
    print('Total predictions:', summary['questions'])
    if summary['accuracy'] is None:
        print('Combined accuracy: unavailable; no dataset has a complete valid reference column.')
    else:
        print(f"Combined accuracy: {100 * summary['accuracy']:.2f}% ({summary['correct']}/{summary['scored_questions']} scored questions)")
    print(f"Scored questions: {summary['scored_questions']}/{summary['questions']}")
    if summary['unscored_questions']:
        print(f"Unscored predictions: {summary['unscored_questions']}; these remain in the CSV.")
    print(f"Resolved: {summary['resolved']}/{summary['questions']}; unresolved: {summary['unresolved']}")
    print(f"Total inference time: {summary['inference_seconds']:.6f} seconds")
    print('Reasoning engines:', summary['engines'])
    print('Unresolved categories:', summary['unresolved_categories'])
    if summary['stopped_at_empty_slot'] is not None:
        print('Stopped at empty dataset slot:', summary['stopped_at_empty_slot'])

def run_datasets(datasets, output_csv='submission.csv', label_column='auto', column_map=None, hf_split=None, hf_config=None, hf_revision=None, install_hf_packages=False, hf_login=False, timeout=2.0, progress_every=25, run_history=None):
    defaults = dict(label_column=label_column, column_map=column_map, hf_split=hf_split, hf_config=hf_config, hf_revision=hf_revision, install_hf_packages=install_hf_packages, hf_login=hf_login, timeout=timeout, progress_every=progress_every)
    plan, combined_path, stopped_at = prepare_dataset_sequence(datasets, output_csv, defaults)
    history = [] if run_history is None else run_history
    if not isinstance(history, list) or history:
        raise ValueError('run_history must be an empty list for this run')
    if not plan:
        print('Enter a dataset in the first DATASETS slot, then fill the following slots as needed.')
        print('No datasets were processed and no output files were written.')
        return ([], summarize_dataset_sequence([], [], stopped_at))
    for item in plan:
        print(f"\nDATASET SLOT {item['slot']}: {item['name']} ({len(history) + 1}/{len(plan)})")
        try:
            rows, dataset_summary = run_dataset(item['dataset'], output_csv=item['output_csv'], **item['options'])
        except Exception:
            print(f"Stopped because dataset slot {item['slot']} failed. Completed datasets: {len(history)}.")
            print('Final aggregation did not finish. Completed datasets remain in dataset_runs; any CSVs already written remain available.')
            raise
        history.append({'slot': item['slot'], 'name': item['name'], 'dataset': item['dataset'], 'results': rows, 'summary': dataset_summary})
    predictions, namespaced = combine_dataset_predictions(history)
    summary = summarize_dataset_sequence(history, predictions, stopped_at)
    if len(history) > 1:
        save_submission(predictions, combined_path)
    if namespaced:
        print(f'Namespaced {namespaced} IDs reused across datasets in the combined CSV; dataset CSVs keep their IDs.')
    summary['combined_id_namespaced_rows'] = namespaced
    summary['prediction_files'] = [{'path': str(combined_path.resolve()), 'combined': True, 'questions': len(predictions)}]
    if len(history) > 1:
        for run in history:
            summary['prediction_files'].extend((dict(entry, dataset_slot=run['slot'], dataset_name=run['name']) for entry in run['summary']['prediction_files']))
    else:
        summary['prediction_files'] = history[0]['summary']['prediction_files']
    print_dataset_sequence_summary(summary)
    return (predictions, summary)


In [13]:
dataset_runs = []
results, summary = run_datasets(DATASETS, output_csv=OUTPUT_CSV, label_column=LABEL_COLUMN, column_map=COLUMN_MAP, hf_split=HF_SPLIT, hf_config=HF_CONFIG, hf_revision=HF_REVISION, install_hf_packages=INSTALL_HF_PACKAGES, hf_login=HF_LOGIN, timeout=TIMEOUT_SECONDS, progress_every=PROGRESS_EVERY, run_history=dataset_runs)



DATASET SLOT 1: jise/ruletaker (1/3)
Source inspection:
 - {"split": "train", "partition": "train", "shard_family": "train", "file": "hf://datasets/jise/ruletaker@b6e36448e186cde51052e978da0465d84da06f31/data/train-00000-of-00001.parquet", "origin": "hf://datasets/jise/ruletaker@b6e36448e186cde51052e978da0465d84da06f31/data/train-00000-of-00001.parquet", "repository_path": "data/train-00000-of-00001.parquet", "manifest_reference": "b6e36448e186cde51052e978da0465d84da06f31", "file_revision": "b6e36448e186cde51052e978da0465d84da06f31", "container": "single_parquet_file", "columns": ["context", "statement", "reasoning", "depth", "flag"], "rows_total": 1000, "problem_rows": 1000, "support_rows": 0, "unrecognized_rows": 0, "rows_included": 1000, "rows_excluded": 0, "classification_errors": 0, "samples": [], "schemas": {"legacy_facts_rules_problem": 1000}, "status": "included"}
 - {"split": "test", "partition": "test", "shard_family": "test", "file": "hf://datasets/jise/ruletaker@b6e36448e1